In [63]:
#1 Install required libraries
!pip install imdbpy beautifulsoup4 requests pandas tqdm vaderSentiment praw

print("Libraries installed!")

Libraries installed!


In [64]:
#2 Download the TV show reviews dataset
!kaggle datasets download -d vinayaks0n1/imdb-tv-show-reviews

print("TV show reviews downloaded!")

Dataset URL: https://www.kaggle.com/datasets/vinayaks0n1/imdb-tv-show-reviews
License(s): apache-2.0
imdb-tv-show-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)
TV show reviews downloaded!


In [65]:
#3
import zipfile
import pandas as pd

# Unzip
with zipfile.ZipFile('imdb-tv-show-reviews.zip', 'r') as zip_ref:
    zip_ref.extractall('tv_reviews')

print("Unzipped!")

# Find the CSV file name
import os
files = os.listdir('tv_reviews')
print(f"Files in folder: {files}")

# Load it (adjust filename if needed)
# We'll see what the actual filename is first

Unzipped!
Files in folder: ['imdb_tvshows.csv']


In [66]:
#4
df = pd.read_csv('tv_reviews/imdb_tvshows.csv')

# Rename columns to standard names
if 'Review' in df.columns:
    df = df.rename(columns={'Review': 'review_text', 'Show ID': 'show_id', 'Rating (out of 10)': 'rating'})

# Map show IDs to names
show_names = {
    'tt9253284': 'Andor',
    'tt3581920': 'The Last of Us',
    'tt5834204': "The Handmaid's Tale",
    'tt4236770': 'Yellowstone',
    'tt9288030': 'Reacher',
    'tt2442560': 'Peaky Blinders',
    'tt13443470': 'The White Lotus',
    'tt13406094': 'Wednesday',
    'tt8111088': 'The Mandalorian',
    'tt5180504': 'The Witcher'
}

mood_labels = {
    'Andor': 'Balanced',
    'The Last of Us': 'Heavy',
    "The Handmaid's Tale": 'Heavy',
    'Yellowstone': 'Intense',
    'Reacher': 'Intense',
    'Peaky Blinders': 'Intense',
    'The White Lotus': 'Balanced',
    'Wednesday': 'Intense',
    'The Mandalorian': 'Intense',
    'The Witcher': 'Intense'
}

df['show_title'] = df['show_id'].map(show_names)
df['mood_category'] = df['show_title'].map(mood_labels)
df = df.dropna(subset=['show_title'])
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import numpy as np

# Initialize VADER
analyzer = SentimentIntensityAnalyzer()

print("Extracting sentiment from 8,319 reviews...")

# Extract VADER sentiment
sentiment_scores = []

for review in df['review_text']:
    if isinstance(review, str):
        scores = analyzer.polarity_scores(review)
        sentiment_scores.append(scores)
    else:
        sentiment_scores.append({'neg': 0, 'neu': 0, 'pos': 0, 'compound': 0})

# Add to dataframe
df['sentiment_neg'] = [s['neg'] for s in sentiment_scores]
df['sentiment_neu'] = [s['neu'] for s in sentiment_scores]
df['sentiment_pos'] = [s['pos'] for s in sentiment_scores]
df['sentiment_compound'] = [s['compound'] for s in sentiment_scores]

print("\nSentiment extracted!")
print("\nSample:")
print(df[['show_title', 'sentiment_compound']].head(10))

# Save progress
df.to_csv('reviews_with_sentiment.csv', index=False)
print("\nSaved!")
df.to_csv('tv_reviews/imdb_tvshows.csv', index=False)
print("✅ Saved!")

Extracting sentiment from 8,319 reviews...

Sentiment extracted!

Sample:
  show_title  sentiment_compound
0      Andor              0.9848
1      Andor              0.3937
2      Andor              0.9872
3      Andor              0.8071
4      Andor              0.6285
5      Andor              0.9732
6      Andor              0.6428
7      Andor              0.9710
8      Andor              0.9456
9      Andor              0.9866

Saved!
✅ Saved!


In [67]:
#5
!pip install text2emotion

In [68]:
#6
import text2emotion as te
from tqdm import tqdm

print("Extracting emotions from ALL 8,319 reviews...")

emotions_list = []

for idx, review in tqdm(enumerate(df['review_text']), total=len(df)):
    if isinstance(review, str) and len(review) > 50:
        try:
            emotions = te.get_emotion(review)
            emotions_list.append(emotions)
        except:
            emotions_list.append({'Happy': 0, 'Angry': 0, 'Surprise': 0, 'Sad': 0, 'Fear': 0})
    else:
        emotions_list.append({'Happy': 0, 'Angry': 0, 'Surprise': 0, 'Sad': 0, 'Fear': 0})

# Add emotions to dataframe
emotions_df = pd.DataFrame(emotions_list)
df['emotion_happy'] = emotions_df['Happy']
df['emotion_angry'] = emotions_df['Angry']
df['emotion_surprise'] = emotions_df['Surprise']
df['emotion_sad'] = emotions_df['Sad']
df['emotion_fear'] = emotions_df['Fear']

print("\n✅ All emotions extracted!")

# Save
df.to_csv('reviews_with_emotions.csv', index=False)
print("\n💾 Saved to reviews_with_emotions.csv")

Extracting emotions from ALL 8,319 reviews...


100%|██████████| 8319/8319 [00:02<00:00, 3884.54it/s]



✅ All emotions extracted!

💾 Saved to reviews_with_emotions.csv


In [69]:
#7 Create text-based features


# Review length
df['review_length'] = df['review_text'].str.len()
df['review_word_count'] = df['review_text'].str.split().str.len()

mental_health_keywords = [
    # Heavy/distressing indicators
    'stress', 'anxiety', 'anxious', 'depressing', 'depressed', 'dark',
    'heavy', 'disturbing', 'triggering', 'trigger', 'draining',
    'exhausting', 'overwhelming', 'uncomfortable', 'traumatic',
    'devastating', 'haunting', 'bleak', 'nihilistic', 'hopeless',
    'brutal', 'gore', 'gory', 'violent', 'graphic', 'nightmare',
    'dread', 'horrifying', 'scarred', 'horrific', 'grim', 'unsettling',
    'distressing', 'disturbed', 'harrowing', 'tragic',
    'heartbreaking', 'soul-crushing', 'emotionally draining',
    # Intense indicators
    'intense', 'gripping', 'tense', 'suspenseful', 'edge of my seat',
    'nail-biting', 'thrilling', 'shocking', 'jaw-dropping',
    # Positive/comfort indicators
    'comfort', 'comforting', 'wholesome', 'uplifting', 'feel-good',
    'relaxing', 'heartwarming', 'safe', 'cheerful', 'joyful',
    'lighthearted', 'cozy', 'fun', 'delightful', 'feel good',
]

def count_keywords(text, keywords):
    if isinstance(text, str):
        text_lower = text.lower()
        return sum(1 for keyword in keywords if keyword in text_lower)
    return 0

df['mental_health_keywords'] = df['review_text'].apply(
    lambda x: count_keywords(x, mental_health_keywords)
)

#  Exclamation and question marks
df['exclamation_count'] = df['review_text'].str.count('!')
df['question_count'] = df['review_text'].str.count('\?')

print("✅ Features created!")
print(f"\nNew features:")
print(df[['review_length', 'review_word_count', 'mental_health_keywords']].describe())

# Save
df.to_csv('reviews_with_all_features.csv', index=False)
print("\n💾 Saved!")

<>:39: SyntaxWarning: invalid escape sequence '\?'
<>:39: SyntaxWarning: invalid escape sequence '\?'
/tmp/ipykernel_24078/1073902376.py:39: SyntaxWarning: invalid escape sequence '\?'
  df['question_count'] = df['review_text'].str.count('\?')


✅ Features created!

New features:
       review_length  review_word_count  mental_health_keywords
count    8319.000000        8319.000000             8319.000000
mean      708.111672         126.145691                0.439837
std       670.820209         116.856724                0.843175
min        43.000000           2.000000                0.000000
25%       272.000000          49.000000                0.000000
50%       621.000000         111.000000                0.000000
75%       834.000000         149.000000                1.000000
max      9825.000000        1780.000000                9.000000

💾 Saved!


In [70]:
# Anchor training data to fix inbalance
anchor_reviews = {
    "Bluey": ("Positive", [
        "Bluey is the most wholesome, heartwarming show my kids have ever watched. We all love it as a family.",
        "A genuinely funny and sweet animated show. The kids adore it and parents enjoy it too.",
        "So uplifting and comforting. Every episode leaves you feeling warm inside.",
        "The most feel-good show on TV. Perfect for winding down with the family.",
        "Bluey is bright, cheerful, educational and endlessly entertaining for young children.",
        "A relaxing and joyful watch. Never stressful, never dark, just pure wholesome fun.",
        "My toddler learned so much from this show. It's safe, sweet and perfectly paced.",
        "Bluey captures childhood magic beautifully. Heartwarming without being saccharine.",
        "One of the best kids shows ever made. Fun, colourful and genuinely funny.",
        "Perfect comfort viewing. The whole family can watch together and just feel good.",
        "Every episode is uplifting. It teaches kindness, imagination and empathy gently.",
        "A happy, safe, cheerful show. Great for young children and surprisingly touching for adults.",
        "So joyful and colourful. My kids are always laughing and smiling while watching it.",
        "A masterpiece of children's television. Sweet, funny, and always leaves you smiling.",
        "The most wholesome show on any streaming service. Pure happiness in animated form.",
    ]),
    "The Great British Bake Off": ("Positive", [
        "The most relaxing TV show in existence. No drama, no stress, just lovely people baking.",
        "Genuinely feel-good television. Contestants are kind to each other and it's wonderfully calming.",
        "A perfect comfort watch. Warm, funny, and always puts you in a good mood.",
        "The antidote to stressful TV. Wholesome, uplifting and endlessly enjoyable.",
        "Such a feel-good show. You finish every episode feeling happy and inspired.",
        "Zero toxicity, just lovely people doing their best. Incredibly heartwarming.",
        "The perfect show to watch when you need cheering up. Warm, funny and delightful.",
        "Cosy, comforting television at its finest. Never overwhelming or emotionally draining.",
        "A truly joyful watch. The bakers are charming and the hosts are hilarious.",
        "So uplifting. Nothing bad really happens and everyone is kind to each other.",
        "Pure comfort TV. Watch it when you want to feel warm and happy.",
        "The most wholesome competition show ever made. Contestants cheer each other on!",
        "Completely stress-free viewing. Relaxing, cheerful and genuinely funny throughout.",
        "A safe, feel-good show for the whole family. Endlessly watchable and heartwarming.",
        "Nothing more uplifting than watching kind people bake beautiful things.",
    ]),
    "Schitt's Creek": ("Balanced", [
        "A wonderfully warm comedy that gets better with every season. Funny and surprisingly moving.",
        "Starts slow but becomes one of the most heartwarming and funny shows you'll ever watch.",
        "A really balanced mix of comedy and genuine emotion. Never too dark or too light.",
        "Funny, touching and consistently entertaining. A great show for any mood.",
        "The characters grow so much over the course of the series. Really satisfying storytelling.",
        "A perfect balance of humour and heart. Makes you laugh and occasionally tear up.",
        "Genuinely funny with a lot of warmth. Not too heavy, not too lightweight.",
        "Manages to be both hilarious and deeply touching without ever feeling manipulative.",
        "A lovely show with complex characters that grow on you. Equal parts funny and sweet.",
        "Entertaining throughout. Some emotional moments but nothing too heavy or distressing.",
        "A feel-good comedy with real depth. Funny, warm and occasionally quite moving.",
        "The perfect mix — funny enough to laugh, emotional enough to care.",
        "A very watchable, balanced show. Nothing too dark but with genuine emotional weight.",
        "Funny and heartfelt in equal measure. Easy to watch but with real substance.",
        "A great all-rounder. Entertaining, warm, occasionally emotional but never overwhelming.",
    ]),
    "Parks and Recreation": ("Positive", [
        "One of the most purely feel-good shows ever made. Optimistic, funny and uplifting throughout.",
        "The most comforting show on TV. Every character is kind and the humour is joyful.",
        "An uplifting and heartwarming comedy that always puts you in a good mood.",
        "A perfect feel-good watch. Funny, sweet, and genuinely inspiring.",
        "Leslie Knope is the most wholesome character in television. This show is pure joy.",
        "Rewatching this when I'm anxious or stressed always helps. It's so warm and comforting.",
        "A cheerful, optimistic show that makes you feel good about humanity.",
        "Light, fun and endlessly rewatchable. The perfect comfort show.",
        "Such a wholesome, uplifting series. The cast have brilliant chemistry.",
        "Watching Parks and Rec feels like a warm hug. Genuinely joyful television.",
        "The perfect show when you need something relaxing and funny. Never stressful.",
        "A brilliant comedy with a uniquely positive outlook. Heartwarming and hilarious.",
        "One of the best comfort shows out there. Feel-good without being cheesy.",
        "So uplifting and funny. The characters grow to be so lovable.",
        "A delightfully wholesome show that never fails to make you smile.",
    ]),
    "Planet Earth": ("Balanced", [
        "A breathtaking, awe-inspiring documentary series. Beautiful, fascinating and educational.",
        "Stunning visuals and incredible narration. Inspires wonder about the natural world.",
        "A genuinely moving and educational series. Some moments are intense but mostly uplifting.",
        "Beautifully made nature documentary. Occasionally sad but ultimately life-affirming.",
        "Fascinating and visually stunning. A balanced mix of wonder, beauty and reality.",
        "One of the greatest documentaries ever made. Engaging without being overwhelming.",
        "Awe-inspiring footage with thoughtful narration. Educational and deeply moving.",
        "A brilliant, balanced documentary. Some emotional moments but overall uplifting.",
        "Perfect for curious minds. Engaging, educational, and beautifully shot.",
        "A wonderful series that captures nature in all its complexity without being distressing.",
        "Calming yet fascinating viewing. The natural world is both beautiful and surprising.",
        "Informative, stunning and occasionally emotional. A very balanced and satisfying watch.",
        "The perfect blend of education and entertainment. Never too heavy or too light.",
        "Some difficult moments but overall a deeply positive and inspiring series.",
        "Peaceful, educational and visually spectacular. A genuinely enriching viewing experience.",
    ]),
}

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import text2emotion as te
import scipy.sparse as sp

analyzer_anchor = SentimentIntensityAnalyzer()
mental_health_keywords = [
    'stress', 'anxiety', 'anxious', 'depressing', 'depressed', 'dark',
    'heavy', 'disturbing', 'triggering', 'trigger', 'intense', 'draining',
    'exhausting', 'overwhelming', 'comfort', 'comforting', 'wholesome',
    'uplifting', 'feel-good', 'relaxing', 'heartwarming', 'safe'
]

anchor_rows = []
for show_name, (mood, reviews) in anchor_reviews.items():
    for review in reviews:
        sentiment = analyzer_anchor.polarity_scores(review)
        try:
            emotions = te.get_emotion(review)
        except:
            emotions = {'Happy': 0, 'Angry': 0, 'Surprise': 0, 'Sad': 0, 'Fear': 0}
        anchor_rows.append({
            'show_title': show_name,
            'review_text': review,
            'mood_category': mood,
            'sentiment_neg': sentiment['neg'],
            'sentiment_neu': sentiment['neu'],
            'sentiment_pos': sentiment['pos'],
            'sentiment_compound': sentiment['compound'],
            'emotion_happy': emotions['Happy'],
            'emotion_angry': emotions['Angry'],
            'emotion_surprise': emotions['Surprise'],
            'emotion_sad': emotions['Sad'],
            'emotion_fear': emotions['Fear'],
            'review_length': len(review),
            'review_word_count': len(review.split()),
            'mental_health_keywords': sum(1 for kw in mental_health_keywords if kw in review.lower()),
            'exclamation_count': review.count('!'),
            'question_count': review.count('?')
        })

df_anchors = pd.DataFrame(anchor_rows)

# Load the existing training CSV and merge anchors in
df_existing = pd.read_csv('tv_reviews/imdb_tvshows.csv')
df_existing = pd.concat([df_existing, df_anchors], ignore_index=True)
df_existing.to_csv('tv_reviews/imdb_tvshows.csv', index=False)

In [71]:
balanced_anchors = {
    "Stranger Things": ("Balanced", [
        "A brilliant blend of nostalgia, mystery and adventure. Scary in places but never overwhelmingly dark.",
        "Stranger Things balances horror elements with heartwarming friendship and humour really well.",
        "It has tense moments but also a lot of heart and fun. Not too heavy to watch.",
        "The perfect mix of scary and feel-good. Emotional without being draining.",
        "Thrilling and exciting but also warm and funny. A really well balanced show.",
        "Some dark moments but overall an uplifting and exciting adventure series.",
        "Manages to be both creepy and wholesome at the same time which is impressive.",
        "Great mix of tension and warmth. Never leaves you feeling emotionally wrecked.",
        "Exciting and occasionally scary but balanced out by humour and heart.",
        "A fun, engaging watch that doesn't leave you feeling drained or disturbed.",
        "The friendships and humour balance out the darker elements perfectly.",
        "Thrilling without being traumatic. A very watchable and entertaining show.",
        "Tense at times but the lighter moments stop it from ever feeling too heavy.",
        "A good mix of emotions. Scary, funny, sad and uplifting all in one show.",
        "Perfectly balanced between dark and light. Never tips too far either way.",
    ]),
    "Black Mirror": ("Balanced", [
        "Thought provoking and occasionally unsettling but never gratuitously dark or traumatic.",
        "Each episode makes you think without leaving you feeling completely hopeless.",
        "Dark in concept but balanced by clever writing and occasional optimism.",
        "Some episodes are heavy but others are surprisingly uplifting and hopeful.",
        "Unsettling but in a way that makes you think rather than just feel bad.",
        "A mixed bag of tones. Some episodes dark, some surprisingly warm.",
        "The best episodes balance darkness with genuine emotional resonance.",
        "Occasionally disturbing but always thought provoking rather than just bleak.",
        "Dark themes handled with intelligence and nuance rather than pure shock value.",
        "Not always comfortable viewing but never gratuitous or needlessly distressing.",
        "A balanced anthology - some bleak, some hopeful, all thought provoking.",
        "Makes you uneasy in a productive way rather than just being depressing.",
        "The darkness always serves a purpose and is balanced by clever storytelling.",
        "Challenging viewing at times but rewarding and never purely nihilistic.",
        "Some episodes feel heavy but the variety across the series keeps it balanced.",
    ]),
    "Ted Lasso": ("Balanced", [
        "Warm and funny but with real emotional depth and occasional sadness.",
        "Mostly uplifting but tackles serious issues like mental health and divorce with care.",
        "A feel good show that isn't afraid to go to darker emotional places occasionally.",
        "Funny and heartwarming with enough dramatic weight to feel meaningful.",
        "Mostly positive but with enough conflict and emotion to feel real and balanced.",
        "A genuinely warm show that still manages to feel emotionally honest and complex.",
        "Uplifting overall but not afraid to deal with sadness, anxiety and failure.",
        "The perfect balance of comedy, warmth and genuine emotional storytelling.",
        "Feel good without being saccharine. Has real emotional depth underneath the warmth.",
        "Mostly cheerful but the show handles darker themes with surprising sensitivity.",
        "A balanced mix of comedy and drama. Makes you laugh and occasionally tear up.",
        "Warm and optimistic but grounded by real emotional struggles.",
        "Not as light as it looks. Balances feel good moments with genuine emotional weight.",
        "A show that makes you feel good without pretending life is always easy.",
        "Funny, warm and emotionally honest. A really well balanced piece of television.",
    ]),
}

# Add these to your existing anchor injection code the same way
anchor_rows = []
for show_name, (mood, reviews) in balanced_anchors.items():
    for review in reviews:
        sentiment = analyzer_anchor.polarity_scores(review)
        try:
            emotions = te.get_emotion(review)
        except:
            emotions = {'Happy': 0, 'Angry': 0, 'Surprise': 0, 'Sad': 0, 'Fear': 0}
        anchor_rows.append({
            'show_title': show_name,
            'review_text': review,
            'mood_category': mood,
            'sentiment_neg': sentiment['neg'],
            'sentiment_neu': sentiment['neu'],
            'sentiment_pos': sentiment['pos'],
            'sentiment_compound': sentiment['compound'],
            'emotion_happy': emotions['Happy'],
            'emotion_angry': emotions['Angry'],
            'emotion_surprise': emotions['Surprise'],
            'emotion_sad': emotions['Sad'],
            'emotion_fear': emotions['Fear'],
            'review_length': len(review),
            'review_word_count': len(review.split()),
            'mental_health_keywords': sum(1 for kw in mental_health_keywords if kw in review.lower()),
            'exclamation_count': review.count('!'),
            'question_count': review.count('?')
        })

df_existing = pd.read_csv('tv_reviews/imdb_tvshows.csv')
df_existing = pd.concat([df_existing, pd.DataFrame(anchor_rows)], ignore_index=True)
df_existing.to_csv('tv_reviews/imdb_tvshows.csv', index=False)

print(f"✅ Added {len(anchor_rows)} Balanced anchor reviews")
print(df_existing['mood_category'].value_counts())

✅ Added 45 Balanced anchor reviews
mood_category
Intense     5466
Balanced    1541
Heavy       1387
Positive      45
Name: count, dtype: int64


In [72]:
#11
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

df = pd.read_csv('tv_reviews/imdb_tvshows.csv')
df = df.dropna(subset=['show_title', 'mood_category'])

feature_columns = [
    'sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound',
    'emotion_happy', 'emotion_angry', 'emotion_surprise', 'emotion_sad', 'emotion_fear',
    'review_length', 'review_word_count', 'mental_health_keywords',
    'exclamation_count', 'question_count'
]

# Fill missing features with 0
for col in feature_columns:
    if col not in df.columns:
        df[col] = 0

X = df[feature_columns].fillna(0)
y = df['mood_category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data prepared: {X_train.shape[0]} training, {X_test.shape[0]} test")
print(f"Categories: {y.value_counts().to_dict()}")

✅ Data prepared: 6751 training, 1688 test
Categories: {'Intense': 5466, 'Balanced': 1541, 'Heavy': 1387, 'Positive': 45}


In [73]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import pickle

# TF-IDF
tfidf = TfidfVectorizer(max_features=100, min_df=2, max_df=0.7, ngram_range=(1,2), stop_words='english')
tfidf_train = tfidf.fit_transform(df.loc[X_train.index, 'review_text'].fillna(''))
tfidf_test = tfidf.transform(df.loc[X_test.index, 'review_text'].fillna(''))

X_train_combined = sp.hstack([X_train_scaled, tfidf_train])
X_test_combined = sp.hstack([X_test_scaled, tfidf_test])

# Train
rf_model = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, class_weight='balanced')
rf_model.fit(X_train_combined, y_train)

y_pred = rf_model.predict(X_test_combined)
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Accuracy: {accuracy:.3f}")
print(classification_report(y_test, y_pred))

# Save
with open('mood_classifier_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

print("✅ Model saved!")

✅ Accuracy: 0.823
              precision    recall  f1-score   support

    Balanced       0.97      0.74      0.84       308
       Heavy       0.65      0.49      0.56       278
     Intense       0.83      0.93      0.88      1093
    Positive       0.67      0.67      0.67         9

    accuracy                           0.82      1688
   macro avg       0.78      0.71      0.73      1688
weighted avg       0.82      0.82      0.82      1688

✅ Model saved!


In [74]:
#8
import pandas as pd
import pickle
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import text2emotion as te
from collections import Counter
import scipy.sparse as sp

analyzer = SentimentIntensityAnalyzer()

with open('mood_classifier_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('feature_columns.pkl', 'rb') as f:
    feature_columns = pickle.load(f)

mental_health_keywords = [
    # Heavy/distressing indicators
    'stress', 'anxiety', 'anxious', 'depressing', 'depressed', 'dark',
    'heavy', 'disturbing', 'triggering', 'trigger', 'draining',
    'exhausting', 'overwhelming', 'uncomfortable', 'traumatic',
    'devastating', 'haunting', 'bleak', 'nihilistic', 'hopeless',
    'brutal', 'gore', 'gory', 'violent', 'graphic', 'nightmare',
    'dread', 'horrifying', 'scarred', 'horrific', 'grim', 'unsettling',
    'distressing', 'disturbed', 'harrowing', 'tragic',
    'heartbreaking', 'soul-crushing', 'emotionally draining',
    # Intense indicators
    'intense', 'gripping', 'tense', 'suspenseful', 'edge of my seat',
    'nail-biting', 'thrilling', 'shocking', 'jaw-dropping',
    # Positive/comfort indicators
    'comfort', 'comforting', 'wholesome', 'uplifting', 'feel-good',
    'relaxing', 'heartwarming', 'safe', 'cheerful', 'joyful',
    'lighthearted', 'cozy', 'fun', 'delightful', 'feel good',
]

def add_show(show_name, reviews_list):
    df_existing = pd.read_csv('tv_reviews/imdb_tvshows.csv')
    new_rows = []
    for review in reviews_list:
        sentiment = analyzer.polarity_scores(review)
        try:
            emotions = te.get_emotion(review)
        except:
            emotions = {'Happy': 0, 'Angry': 0, 'Surprise': 0, 'Sad': 0, 'Fear': 0}
        new_rows.append({
            'show_title': show_name,
            'review_text': review,
            'sentiment_neg': sentiment['neg'],
            'sentiment_neu': sentiment['neu'],
            'sentiment_pos': sentiment['pos'],
            'sentiment_compound': sentiment['compound'],
            'emotion_happy': emotions['Happy'],
            'emotion_angry': emotions['Angry'],
            'emotion_surprise': emotions['Surprise'],
            'emotion_sad': emotions['Sad'],
            'emotion_fear': emotions['Fear'],
            'review_length': len(review),
            'review_word_count': len(review.split()),
            'mental_health_keywords': sum(1 for kw in mental_health_keywords if kw in review.lower()),
            'exclamation_count': review.count('!'),
            'question_count': review.count('?')
        })
    df_new = pd.DataFrame(new_rows)
    X = df_new[feature_columns].fillna(0)
    X_scaled = scaler.transform(X)
    tfidf_features = tfidf_vectorizer.transform(df_new['review_text'].fillna(''))
    X_combined = sp.hstack([X_scaled, tfidf_features])
    predictions = model.predict(X_combined)
    df_new['mood_category'] = predictions
    mood = Counter(predictions).most_common(1)[0][0]
    print(f"✅ {show_name} predicted as: {mood}")
    df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    df_combined.to_csv('tv_reviews/imdb_tvshows.csv', index=False)
    print(f"✅ Added {len(reviews_list)} reviews for {show_name}")

print("✅ add_show ready!")

✅ add_show ready!


In [75]:
#9
the_boys_reviews = [
    "I've had big expectations about this show, because I thought that is going to be exactly that kind of show, in which the good guys (the superheroes) fight the bad guys (the villains) and of course, they win. I couldn't have been more wrong, especially because \"The Boys\" it's not about a desperate attempt of superheroes to save the world from evil, instead this show takes a much more realistic approach, about what is going to happen if superheroes had really existed. I'm saying that I had big expectations in the beginning, because I thought that it would be the same usual show with superheroes and villain, that is most likely to find everywhere. In the first couple of episodes I've been a little bit disappointed, or rather shocked, because like I've already said, \"The Boys\" describes what the MCU never did. Of course, the movies and shows from Marvel are destined to other public's category, but this show is tough. It has all you could not expect from a TV series with superheroes. It has violence, gore, nudity, astonishing CGI effects and a very unpredictable and solid storyline. In the end it seems that this one delivered what I didn't expect it to and I am pleased with that. The show is definitely worth watching.",
    "There's nothing more that I can really say about The Boys that hasn't already been said. It's just a flat out awesome show! It's easily become one of my favorite new shows in years! I binged the entire first two seasons in just a few days when they first came out and rewatched them again for the second time before Season 3 came out.",
    "The Boys is one of the best Superhero shows I've ever seen. While Season 1 was the best season of the series, Season 2 and 3 were also both very good and absolutely worth watching. Season 3 was fantastic, Jensen Ackles was the perfect actor to add to this already incredible show!",
    "I'm a huge Legion fan because of the way they did things - The Boys landed like a punch to the face, its dark, its funny (that dolphin scene!) and most of all its one of the most entertaining shows I've seen lately... every character is developed to perfection.",
    "Predictable. One shocking scene after another. This is the layziest writing and directing ever. Talking heads, bs dialogs and constantly people sitting and talking. And talking. No dynamic. This season is so off, it looks like they cut the budget in half.",
    "The boys started really well, but from the last episode of season 3, the show has been going downhill. The story has stopped progressing, and the creators are only adding stuff to generate shocking reactions from viewers.",
    "Want to see what it would really be like with super heroes around? Check this out! No kids play here folks....not for kids. Best damn series on right now....about time!",
    "This Amazon series is set in a world where there are superheroes; they aren't typical superheroes though; they work for the powerful Vought Corporation and consider ratings and profit to be more important than saving lives.",
    "When I first heard of 'The Boys', I thought it would be your standard superhero series, but when I actually watched it, I discovered I was dead wrong. After 15 minutes of the first episode, it was impossible to stop watching.",
    "The Boys initially captivated audiences with its first two seasons, offering a fresh, irreverent take on the superhero genre that was both darkly humorous and thrillingly intense. However, starting with season three, the series began to lose its way.",
    "The Boys really turns the superhero biz inside out. Hilariously disgusting adventures abound, yet there's still a lot of heart to the show. Great characters & character dynamics. Antony Starr as Homelander is brilliant.",
    "This series is dark, gritty and gripping. From the very first scene you know right away that this isn't the MCU or DC or even the Watchmen. It's somehow darker and even more twisted and yet, still inexplicably grounded in reality.",
    "Hughie Campbell's life is turned upside down when the love of his life is accidentally killed by a superhero. The more he investigates, the more he discovers they're far from perfect.",
    "Give a celebrity super powers and you have The Boys. Narcisistic and arrogant. Above the law. The Boys shows the real and darker side of humanity.",
    "Ridiculously good series. It's like your favourite Guy Richie movie, only with superheroes. Antony Starr's performance as Homelander is second to none.",
    "I can't stand Marvel and DC and all the comic book stuff, BUT THIS IS DIFFERENT! This is one of the best shows around, best show since Game of Thrones.",
    "Season 4 of The Boys is close to unwatchable. The biggest problem is that it is way too political. Hollywood needs to understand that the audience does not want political views shoved down their throat.",
    "Everything about this show is freaking amazing. The acting and writing are some of the best I've seen on a show.",
    "Love the show, best thing on Prime for a good while.",
    "It's a sad year for The Boys. What started as a captivating and entertaining show, has devolved into a painful hour of viewing each week.",
    "Finally something new and fresh from the stereotypical drivel being jammed down our throats for the past couple of years. Dark, gory, hilarious, ethically mind bending.",
    "This show is awesome. The series has a great story, great characters, crazy twists, awesome special effects.",
    "Incredibly well written. Incredibly acted. Incredibly gory. This is a show with substance.",
    "Truly an antidote to Marvel and Co. Having American super heroes forced down our throats 24/7 is so tiring.",
    "This once brilliant show is now like Spider Man 3. Every single character is going through an existential crisis.",
    "Season 1 and 2 were great and season 3 was pretty good. However, season 4 is garbage.",
    "Great but it's not a dystopian reimagining - it's quite literally an allegory for America and the world as it exists right now.",
    "First 2 seasons were great. Season 3 was good. Season 4 is just going downhill in a hurry.",
    "When it premiered, The Boys was a crazy comic come to life and a blast to watch. But after 4 seasons it has become repetitive.",
    "This show has the potential to be another gem such as GOT or Breaking Bad. The acting is superb.",
    "One of the most striking aspects of The Boys is its bold storytelling. Unlike traditional superhero narratives that celebrate heroism and justice, this series dives into corruption, greed, and the abuse of power.",
    "I've only watched the first 2 seasons and a little bit of the 3rd but it's already so good. Every 5 minutes I'm jumping out of my seat.",
    "One of the most impressive things about The Boys is how it balances entertainment with meaningful commentary.",
]

add_show("The Boys", the_boys_reviews)

✅ The Boys predicted as: Balanced
✅ Added 33 reviews for The Boys


In [76]:
the_last_kingdom_reviews = [
    "Incredible The best way to describe The Last Kingdom is that it is just a flat out awesome show and a must watch. I never read the books so I didn't really know what to expect but I'm so glad that I did start watching it because it's become one of my favorite shows of all-time. As you can tell from the reviews and the rating this show is loved by just about everyone who's seen it. It currently has a 92% on Rotten Tomatoes and 8.5 here so both the audience and the critics love it. I can not overstate just how good this show really is. Every season is just as good as the others. There really isn't much more I can say about this incredible show that hasn't already been said so just go watch it. I promise you'll love it.",
    "Bright The characters and plots are spot on and unlike Vikings, every season is brilliant. On top of the brilliant production, the soundtrack fits perfectly and gives me goosebumps every time I watch it. Hats off to everyone involved, this is hands down my favorite series. The actors are so good it's hard to single out one in particular, although Alexander Dreymond's work is impressive, but the interaction with David Dawson who plays King Alfred is outstanding. And that doesn't take away from the excellent performances from all the main cast and the casting did an excellent job with Millie Brady playing Dreymond's love interest. I want to see more of their future performances. Every season is just as good as the next. There really isn't much more I can say about this amazing show that hasn't already been said, so go watch it. I promise you'll love it.",
    "Amazing and captivating This series is one of the best ever. As one of the characters said, \"It is life.\" So much happens.... characters live and die, sometimes unexpectedly and certainly not what you wanted to happen. Despite being set in the 9th century, the series makes it clear that politics and power have not changed and The Last Kingdom amply demonstrates this. The role of religion and the power of the church also are treated as they should be.... sometimes good and sometimes evil, just as humans are. Watch it. I can hardly wait for season 4.",
    "BRILLIANT! I've watched this at least 4 times, maybe even 5 and having just finished watching it again I genuinely look forward to watching it more in the near future. The characters and story lines are on point and unlike Vikings every season is brilliant. On top of the brilliant production the sound track fits perfectly and gives me goose bumps each time I watch. Hats off to everyone involved this is by far my favourite series with the ending to the film (Seven King's must die) finishing this epic story in the most perfect way possible. If you haven't watched it yet stop floundering and get watching. You won't regret it!",
    "So well done on many levels Granted there with some license taken and that makes sense for production. However none of that detracts from what is for me one of the best series out there. The production design was great. They put a lot of effort into it and it shows. Uhtred is fantastic character vehicle guided by a well-written although fast-paced adaptation from the book series. The actors are so good that it's difficult to single out any particular one although it's an impressive job by Alexander Dreymon but the interplay with David Dawson who plays King Alfred it's outstanding. And that is not to take away from excellent performances by all of the leading cast and the casting did an excellent job with Millie Brady playing the love interest of Dreymond. I want to see more from her future performances. She demonstrated a complete understanding of her character and how it would need to navigate the gender power differences and still act as Queen. This series did suffer an variance in quality of writing and direction from the early days when it was produced by the BBC to Netflix's oversight. But once you're swept away the narrative it's not as important.",
    "Love it - BUT... I came upon this late - in 2025 - but I'm already on Season 4 and I'm hooked. Can't seem to stop at one episode and have to force myself to turn it off and go to sleep. BUT... what's up with Uhtred's \"accent?\" With each subsequent episode it becomes more and more bizarre. Right now, (S01E01) Alexander Draymon has decided to enunciate each syllable in each word all the way to infinity and to add a \"T\" to the end of every last syllable. And no director has told him to knock it off! So, that being said, the series is great. Kind of hard to keep up with the names, but the internet comes in handy. Overall, well worth a watch.",
    "The Last Kingdom - Between Ragnarök and Roast Beef, Uhtred Swings His Axe Right from the start, The Last Kingdom throws you into the thick of it: a noble kid gets kidnapped by Vikings and grows up as a cross between Ragnar Lodbrok and a British aristocrat. Uhtred is the guy stuck juggling two worlds, like an episode of Kaamelott rewritten by Bernard Cornwell. Raised by barbarians, destined to serve the Saxons-this dude hit the jackpot of misfortune, and honestly, it's glorious watching him struggle. We're talking Vikings, English kings, and epic battles, but the show spares you a boring PowerPoint presentation. Instead, it serves up places, dates, names, and improbable mustaches, all wrapped in a series that tricks you into thinking you're brushing up on your history. Yes, Alfred the Great was real, and no, Uhtred doesn't show up in history books. But let's be honest: this fictional guy is the missing link between William Wallace and Tony Stark. If you're into battles louder than a karaoke night after three pints, The Last Kingdom is your holy grail. Warriors trade leather Viking jackets for gory axe wounds, filmed like a medieval Doom Eternal. The action doesn't reinvent the wheel, but it delivers visceral combat scenes that keep you glued to your seat-reminding you that Saxons couldn't form a proper battle line to save their lives. Alexander Dreymon as Uhtred? He's got the looks of a heartthrob but the demeanor of someone repeatedly dying in Dark Souls. He carries the show like a berserker lugging a shield. Alongside him, the supporting cast oozes credibility: fragile kings, badass queens, and priests nastier than a The Witcher 3 final boss. They add depth, even if a few look like they wandered in from a half-baked cosplay contest. Uhtred's eternal dilemma-team Ragnarök or team God Save the King-is the heart of the show. Torn between his Viking upbringing and his Saxon destiny, it's like choosing between Game of Thrones and Vikings: you don't want to pick a side. This duality fuels the series' dramatic tension, hooking you like a Nordic fish on a barbed hook. The Last Kingdom is a medieval slap in the face, reminding you that history isn't just dates and wars. It's about people like Uhtred, caught between two cultures yet still finding time to drop lines sharper than Jean-Pierre Coffe with a battle axe. If you've missed this, you've skipped the best mix of action, drama, and axes since Braveheart. There, I said it.",
    "My Goodness! This a world class tv series with a brilliant cast. Thank you Netflix for Season 3, it's simply outstanding!",
    "Most underrated Tv show Binge watched 3 seasons of TLK.Tbh,i thought that this show will be good.But after some posts in the group, i tried the pilot episode and from there i have binge watched all 3 seasons in just 3 days. I kinda like and prefer more realistic films/series and that's the reason why i haven't gave TLK a try.But, it is so realistic and I regret it now.It was the best series i have watched in the recent times. It was fast paced and gripping from the pilot itself.I haven't felt a single low movement in the whole 3 seasons.Mainly, the strong characterization and the guy who played Uthred(Alexander) is just perfect for his role.I like his voice very much.Every single time he says\"DESTINY IS ALL\", i get goosebumps.And,the whole cast are good. I loved the relationship between Uthred and Alfred and the way they presented it. Cinematography and editing can't be better.Season 3 was much better technically when compared to 1 and 2.Special mention to the Soundtracks of the series.Ahhh! They are so addictive. Still can't get over with those characters and the kingdoms (:",
    "The boy who walks with the Vikings For a medieval adventure, The Last Kingdom moves at a strikingly fast pace with unhinged ferocity and undiplomatic cultural clash. Audience might be familiar with The Vikings and it does have similar characters. Both series sets in the same timeline and world, but this one is told from the perspective of an England boy. This is a fresh rendition of epic saga that creates solid characters in highly perilous time, relying less on stylish ambiance and more on gritty dusty violence. Uhtred, son of Uhtred, is a boy meant to inherit an earldom. Then a series of unfortunate events carries him from luxurious to eventual warrior life. All of which happen in far greater pace than most television series. It introduces characters and sends them off to whatever fate tomorrow may hold for them, it has no place and time for wallowing. Admittedly, this grants a much cinematic presentation, it also keeps the plot and characters very well refined. Script presents just the right amount of bantering for the personalities to be identifiable, establishing the relationship without wasting a moment. As its peers have done, it also brings skirmishes and brutality. Action is straightforward mayhem, grounded in presentably realistic warfare of the time. There's no shortage of carnage or blood here. However, it might encounter a few issues later on. The transition could feel too brief, audience barely has enough time to invest before it shifts the plot pretty significantly. While it doesn't gloss over details, some subplots could've been more thorough. There are plenty of characters passing by even in span of one episode, that it simply has no time to cover them all and this rapid pace might not match runtime of a series. The Last Kingdom is an intriguing epic tale. Granted, comparison could be made with Game of Thrones or The Vikings, be it a blessing or a curse, but it's strong enough to stand on its own and definitely a worthy foray to another medieval realm.",
    "Captivating! With the plethora of mediocre historical series out there, I did not expect much from this show. I, however, was utterly blown away by how compelling it is, and by the excellent acting by all. The subject matter is fascinating, and it depicts well the harshness of living, and surviving, in a barbarous world, with an emerging, equally barbarous \"new\" religion. The casting was terrific. The actor who played Scorpa, the Dane of the White Horse, was beyond fierce. An unforgettable presence. I was especially, impressed by the lead, Alexander Dreymon, whom I had never seen before. As the series progresses, you really see this guy's acting chops. His range is from the tender to the ferocious. He has a bunch of us hooked, and we cannot wait for the next season. Thank you for entertaining us!",
    "Great series, greater books but the casting is questionable at times.",
    "From ahistoric to a straight up fantasy. The novels are brilliant, but this falls short. Season 5 cements it as a total failure for a historic setting, with diversity casting and sign language for a deaf servant girl, it's gone the same way all TV shows go once the wokerati get their hands on it. It is an insult to the ancestry of the people whom it mocks. They would never dare do this with any other demographic besides Europeans because they know they would face real consequences for such mockery.",
    "After four seasons. Best show currently running. I reviewed this after the first season in 2015. I'm doing it again after season 4 in 2020. I said this show had the potential to be better than GoT and Vikings. In my opinion it has done that. You think GoT had unpredictable deaths? Well buckle up for this show. It happens a lot and at times it's completely out of nowhere and not a spectacle at all. You're like you know I like that character. Boom. Well sh&t they're dead! The story is absolutely fantastic. The characters are likeable and some are downright unforgettable. How this show doesn't even get any buzz around award time boggles my mind. The performance of Alfred in season 3 should have been Emmy and Golden Globe nominated. The show in general should be. This show has turned me into a fanboy and not many shows or movies have ever done that to me. Breaking Bad, The Witcher books and now this. So take this review as the words of someone who loves this show. If you watch all four seasons I think you will feel that this is one of the best shows currently running. I've been wrong before, but if you like swords, shields, battles, violence, betrayal, revenge, moments that make you want to jump off your couch run through a wall and high five the first person you see then WATCH THIS SHOW!",
    "The most under-rated show on TV I stumbled across this hidden gem and binged-watched it in just a few days. I cannot express how much I enjoy the story, the actors - the sense of realism. There's nothing like it on television! Wished that Netflix would promote it more. Some compare it to Vikings, Game of Thrones, and Witcher, but once I got a taste of The Last Kingdom, I couldn't even watch those others because TLK makes them look comically over-produced. So if you're looking for a realistic historic drama with great acting, this one's for you!",
    "An amazing show that will keep you wanting more. Without going into too much detail, this show will slowly but surely get you attached to the main character Uhtred son of Uhtred. The immersion that this show provides leaves you stunned at times and it never fails to disappoint in terms of character development. It shows how a man who carries himself with honor and conviction doesn't always get what's due never the less that doesn't hinder him in pursuing his dreams and his birthright.",
    "I'm obsessed with Uhtred Have watched Game of Thrones, Vikings, Peaky Blinders- etc etc..but I had never even heard of until a few weeks ago has blown me away. The characters are well developed and you feel like you get to know each of them. Uhtred is by far the most interesting character that you he root for every episode. It makes me sad that with all the garbage out there, that the series doesn't receive the attention and accolades that it should. It's an excellent show, excellent production value, excellent storylines. Each season gets more intense and storylines are seen through. I highly recommend it.",
    "Great TV Show Just happened to find this show scrolling through Netflix one day. So happy I found this series. The first episode was good, but it took 2-3 to draw me in. I think the plot and story are just great. It is a twist on the Viking world that is currently taking TV by storm. The actors do a phenomenal job with their characters and interaction with one another. I love how the relationships have developed between all of the characters in this world, between Uhtred/Alfred/Ragnar and just the many many other characters how have strained and strengthened relationship. I have turned several friends onto this who are GoT fans. I think this series is right there with GoT. I can't wait for another year of it and I am interested in where they take the story and the characters!",
    "Rip-Roaring Combination of Familiar Adventure Series Elements I will not summarize the plot of THE LAST KINGDOM, as it follows a familiar quest-pattern and has been amply described by previous reviewers. Suffice to say that the source-text's author is Bernard Cornwell, an established writer of historical adventures set in various historical periods that have translated successfully to television in the past (remember SHARPE (1993-2008))? Directed by four different people (Anthony Byrne, Ben Chanan, Peter Hoar and Nick Murphy), THE LAST KINGDOM is basically SHARPE transposed to the Saxon/ Viking era, with the same combination of familial politics, intrigue and rivalry. The moral priorities are determined from the first episode onwards: the Sharpe-character is personified by Uhtred of Babbanberg (Alexander Dreymon) who must negotiate several obstacles before he can assume his rightful place in the formation of a new kingdom. Good guys and bad guys are easily distinguished, chiefly by their hirsuteness; the bad guys have bushier beards and longer hair than their rivals. Each episode has its fair share of incident, romance, and a touch of pathos; the visual style owes a lot to Ridley Scott's KINGDOM OF HEAVEN (2005), with its lengthy pans of lonely isolated, almost primeval landscapes contrasted with dimly-lit interiors illuminated by candlelight. Considerable emphasis has been placed on the elements - sunrises, sunsets, storms, wind and rain - making us aware of how difficult life can be for those trying to establish social and political order. Hence their acts of heroism become all the more noteworthy. THE LAST KINGDOM might be hokum, but it is nonetheless highly entertaining: we care for the characters and their futures, even if we are well aware of what will happen in the end. As with most adventure novels, there is little doubt that it will have a morally definite ending.",
    "Everything Game of Thrones wasn't I put this on in a moment of boredom. After two episodes I was hooked. Great story arc from a fascinating and horrific period to be alive. Just about every actor smashes it out of the park. Finan and Ardhelm both strong but very different stand out. Breeda was the only storyline that lost any sense of reality, it's like they didn't really know what to do with her. Alfred's wife/Edwards mum seemed ageless and became annoying and almost added comedy. Father Pyrlig also surprised me, ive only ever seen him in small roles like the inbetweeners, but I look forward to seeing him again, Ian Hart as Beocca was the pin that hooked me from the start. Great production values from the first episode that got better Honestly it's been the best two weeks binging this. So pleased I gave it a go.",
    "Series 1-3 Great. Series 4 Pap. It's always been a bit far-fetched and borderline silly but The Last Kingdom was always very watchable and compelling. That is until the latest series (four), which is absolute dross. Did they sack the real scriptwriters and just let the actors improvise?? If this is the best the can do I hope they don't try to eke it out any longer. For their sakes.",
    "Incredible Series I can watch many times Absolutely love this series, never wanted it to end! Read all books many years ago would have loved at least 4 more Episodes. Acting brilliant. I felt a part of the whole story line. Characters appeared so real. Alexander's emotions in all scenes was so moving he displayed passion, sadness and anger with such feeling. I felt like I know all of them. Watched it now 8 times! Corruption with Kings always present and political exploits. England that land of my birth! Briton & Anglo Sax Norse with some Irish in for the mix. All the cast far out did any well known H/wood actors. Thank you all for making it so real !",
    "You will lose braincells watching S5 Woke killed the show, not even the 10/10 bots can save it. I don't think that it's a result of incompetence, you simply can't have nice things any more.",
    "Best medieval T.V. Show so far I love every episode of this show, there is always an action, it does not bore even for a minute. This show is the reason why I am still paying for Netflix, it reflects the era very well. I hope they do not cancel and continue for years.",
    "Keeping the soul of the books intact, The Last Kingdom delivers a extraordinary historical tale. In a way, BBC's The Last Kingdom is a first ever experience for me. It's certainly not the first series I watch, nor it's the first series based on books, not even historical fiction books. But it's the first series which I'd read the books it is based on way before it was even announced. So seeing it rumored, then officially announced and finally premiered was very interesting and exciting. I had a lot of hopes and fears for this. And I'm so gladly to say that based on the first two episodes of the series, my hopes were made into reality and my fears into joy. And so, inspired me to write my first review here. I'll try to keep this as spoiler free as possible, but some are impossible to avoid. The Last Kingdom tells the history of Uhtred of Bebbanburg, a Saxon noble who is captured and raised by the Danes, first as a slave and then as a son. His future will be one of battle within and without, fighting against powerful warlords and struggling between his love for the Danes and their culture and his duty for the Saxons. In many ways, talking about how this series make this simple enough premise into a engrossing story full of nuance is talking about how the books do it. Here there is a commitment to reality that goes beyond most historical shows, and I'm not talking only about the clothes, weapons or history itself, but characters, their motives and relationships. At the center of The Last Kingdom is its characters and the show presents and develops then masterfully. Take Ælfric (Joseph Millson) for example, Uhtred's uncle. When the young Uhtred (Tom Taylor) wakes up and realize his father went to war without him, he promptly go and ask him why he was left behind. His uncle barely talks to him and say he has to go and clear the stables. But the directing and acting in the scene makes it clear Ælfric has something sinister in mind. Uhtred get his pony and goes to join his father, and then you realize that that was Ælfric's plan all along. It's subtle and skillfully done, and with something that feel effortless you have a lot shown to you what Ælfric is all about. When it comes to characters there is no compromise, they feel like real people that are just living their lives. Doing what they think is right and/or what is best for them and their loved ones. You won't find good or bad here, nor one dimensional characters. By the end of the first episode, you will find you will have feeling for these people, both positive and negative. And in the end, that is what keeps you interesting in a series. And you will care for the fate of Uhtred (Alexander Dreymon), Brida (Emily Cox), Alfred (David Dawson) and the many other great characters that are present in the first two episodes. And from them stem all the other conflicts in the series. The political plotting and machinations are ever present in the fractured world that these people inhabit, and the leaders of both the Saxons and Danes are constantly trying to get things working in their favor. Together with political issues, there is religion. Christianity is a defining characteristic of the English, and because of the Norse paganism, they view the Danes not only as enemy of the Saxons, by enemies of God. The Danes of the other hand, view Christianity with a disinterested curiosity. In one opportunity Uhtred explain Heaven as Valhalla without all the drinking, fighting and humping. This passage helps to illustrate a kind of 'lightness' that the series, just as the books, have. Maybe lightness is not the best word, since there is plenty of violence and moments that will leave you shocked, but the story is told in a way that feels amiable and pleasant. That is done is the series by skillful directing , writing (in the first two episodes done by Nick Murphy and Stephen Butchard respectively) and editing. That, along with its character, might be the two most appealing things about The Last Kingdom. Add to that a great score, beautiful scenery and great acting (although a little bit inconsistent in some moments) and you have the recipes for a series that stand tall and deserves to be seen. No matter what you like, The Last Kingdom (the first two episodes together are a nice demonstration of everything good about this story) might offer something for you. For the books fan in particular, this is a series that even when it changes the story of the book it feels it belong, Bernard Cornwell's tone is ever present in it. Not to mention many small moments that shows the people responsible for the series really cared about the source material, like in shield wall battle (it doesn't reach the greatness of the book, but then, what battle scene anywhere does?), there is a scene of people walking over a dead body, something that feels taken right from the book. Or a sword strike that comes from bellow. Even some of the more questionable changes feel justified. Like giving the Saxon rectangular shields. In the end it proves extremely valuable as a tool to differentiate Danes from Saxons. But without a doubt, this is a adaptation that work is all levels and will delight most readers.",
    "Absolutely Amazing!! The Last Kingdom is absolutely, without a doubt, one of the best shows I've ever seen! I'm a big fan of this genre show I was excited to start watching it and once I did it quickly became one of my favorite shows of all-time. It has enough violence and drama to satisfy anyone watching it. It gets compared to Vikings and Game of Thrones a lot and as good as those shows are this holds up against either of those . It's one of the rare shows where every episode of every season is fantastic. If you're looking for a great show that will keep you entertained throughout the entire series then give this a try. You won't be disappointed!",
    "'The Last Kingdom' is a great binge!! One of my favorities! This Netflix series is a period piece during the early years in England highlighting the battle between the devoutly religious and non-religious, good and evil (sometimes in reverse), and is full of action, drama, sensuality and a bit of fantasy. The main actor, Alexander Dreymon who plays \"Uhtred of Bebbanburg\" is mesmerizing and entertaining to watch! Great supporting cast, great storyline and a great escape! (It's on a smaller scale of 'Game of Thrones', but very enthralling and it will keep you hopping and binging!) If you haven't seen this one yet, it's a true hidden treasure!",
    "History with a twist. Every scene, every season - Amazing. Once you start, you won't want to stop. Really hoping for a season 4.",
    "Absolutely brilliant until you get to Season 4 Loved season 1-3. Great story and character building Season 4 is filled with disappointing character decisions and choices Not that season 1-3 has realistic scenarios but season 4 takes this to another level of unbelievable Hope season 5 can redeem the show",
    "I Upped this from a 9 to a 10 (Flash Review) This series is not known as well as it should be. It blazes its own trail without being pressured to follow the path of Games of Thrones. The characters are still full of depth, emotion and moral quandaries. Season 4 watches a young and inexperienced King of Wessux, which seems to be the head kingdom of the four primary kingdoms, struggle to make decisions, earn respect and maintain true allies. There's an abundance of intertwined story lines and with moderate viewer focus can be clearly followed that are all rich with drama. As with most Mid-Evil shows, many a battle is fought, much flesh is slashed or punctured and blood spilt. Great production value, unique musical score and honest acting make this a must see if you like Mid-Evil era shows.",
    "Seriously good, don't listen to the hate If your patient with this show it can be a great watch. season 1 is brilliant, season 2 does start off slow but gets good during the middle and end. If you love game of thrones you'll enjoy this no doubt.",
    "It's pretty good - but not groundbreaking at all. Ok, I'll start by saying the show is quite entertaining and that the script is quite well written. It's a show you can't miss a moment of or you'll be lost , as there is always something happening. With that said I'll admit I understand the times the show takes place in quite well; but, I find Uthred actually a rather unlikable and seemingly predictable character. His personality is actually rather offputting and I'm not sure if this is Dreymons true accent; but it seems rather forced in parts and even jarring; though not in a fun way. Also, Dreymon is a good actor ; but he doesn't quite seem to fill the role of Uthred at all, and would have been better suited to be in films like Kingsmen or The Gentlemen- not as a blood thirsty half Dane / Saxon warrior. There isn't really a character on the show that differs from others. The Danes are all pretty cookie cutter, with even Brida being a good deal of the time a bit too much and again seeming overacted. I love a nice complicated storyline and found that the names also being very close in sounds (Uthred, Guthred) made it at times a bit silly trying to figure out whom was being spoken of. The show seems to have a repetitiveness to it in knowing that Uthred will always come out alive and ok somehow and that there will be one big battle in the last episode of each season. Also, the soundtrack was ok the first episode or two but hearing the same three songs repeated EVERY EPISODE made it feel lazily put together. Overall it's worth a watch and at times quite entertaining - especially those battle scenes , very well done. But it certainly isn't a masterpiece. Some more character range , and better casting and soundtrack would VASTLY improve this series.",
    "the last season is a total mess, from great to ridiculous",
    "It just gets better and better If you haven't starting watching this yet and you're are thinking about it - think no more and just watch it. You won't be sorry. This series is sensational with brilliant writing and phenomenal acting. There is so much going on but it is easy to follow the storyline and it keeps you interested and excited and anxious for more. When season 2 ended it was tough. I am now started to watch season 3 and so far I am not disappointed. Thank you Netflix for another amazing series. I am rooting for a season 4 and more please.",
    "Doesn't take tens of millions of dollars to put on a great show! Less polished than GOT, yet thoroughly riveting and entertaining. Just goes to show that showmakers don't need to spend tens of millions per episode to put on a great show. This show has some great actors and an obviously brilliant director",
    "Top notch! Start the finish the storyline and the characters are fantastic and it never got boring. Every season was better than the previous one. Wish they made more. The Last Kingdom is a captivating historical drama set during the Viking invasions of England. Based on Bernard Cornwell's The Saxon Stories, it follows Uhtred of Bebbanburg, a man caught between his Saxon roots and Viking upbringing. The series stands out for its complex characters, intense action, and rich historical setting. With strong performances, especially from Alexander Dreymon, and impressive visuals, it offers an immersive experience full of intrigue, loyalty, and personal conflict. A must-watch for fans of historical drama and epic storytelling.",
    "Best ever Last Kingdom is a perfect series! Here you see a wronged main character who fights for his rights throughout his life, he is a wonderful warrior but is always at a disadvantage against his enemies, he is respected and idolized wherever he goes, he is faithful, he fights for everything he believes in, he fights for his family, for his friends, for the people he loves... The main character is an inspiration for my life!!! I want my children to be warriors, to be respectful, to love and be loved like the main character in this series with so many remarkable characters! Last Kingdom is not just a TV show it is a lesson in life, morals and love.",
    "The birth of England. Absolutely loved this series, one of those I looked at beforehand and thought I wouldn't like, it looked like another Game of Thrones themed series, along the same lines as Vikings. It's a period in history that I know relatively nothing about, so it's an interesting theme for a drama to explore, I can't comment on the accuracy of the series as I don't know enough about it. The series looks beautiful, the locations used are utterly tremendous, the cinematography was awesome. Fantastic clothes and make up too. It's fairly violent in parts, but it's there for a reason, it never seems done unnecessarily, it's there to add to the story. The acting is fantastic, quite a few big names, none of them last too long it seems, how mean to get shot of Matthew Macfadyen, Jason Flemyng and Rutger Hauer after just a single episode. Alexander Dreymon is absolutely fantastic in the lead role of Uhtred, he's perfect for the part. Loved it 9/10.",
    "Incredible series Best series on Netflix by far can rival Viking's any day of the week",
    "Nice! First three seasons are very good. Fourth season is meh.",
    "Outstanding. So this is how it goes. Season 1. You hang in for a couple of episodes, then you start thinking, wow, this is really good. Then you think, heck, I might as well move right on to Season 2 and you say, whoa, this is awesome. Of course you'll immediately plunge right into Season 3 which of course, you CAN'T STOP WATCHING because it's EPIC. Script writing, cinematography and actors are all perfectly in sync. The soundtrack is mesmerizing. There's blood, battles, kings, queens, Saxons, Danes, love, sorrow, humor and lots of The Unexpected. And history. Real history. David Dawson deserves award after award for his portrayal of King Alfred, Alexander Dreymon, besides being quite breathtakingly hunky, has perfected the conflicted warrior-hero, Harry McIntire - well, you just gotta watch. The entire cast simply excels at their craft. Netflix signed on for Season 4 - and for good reason. You won't be disappointed with The Last Kingdon. Destiny is all.",
    "FIRST 3 SEASONS = 9/10. SEASON 4 = 5/10 In season 4 everything has become boring and some scenes have no sense. Why Uhtred never learns how to have freedom after all that happened to him? His enemies always have a huge army and he doesn't, why?? Isn't he famous after all the battles he fought? Overall rating: 8.0 I didn't spoil anything because everyone knows that he always get arrested lol. I watched it until the end of S4.",
    "S05: another fake woke history Again rewriting history and doing social engineering with black people un S05. We already know your plans but please leave The Kingdom Alone alone. It was the last series you hadn't destroyed.",
    "One of best shows ever I love this show. It guides you through history, humanity, and religion. All characters have amazing plots and are respected in the story. Every event has it purpose for the future and nothing is taken without historical reasons. All human emotions, are portraited equally, like a canvas an its colors. You can not define the show with only one label. Each character invite you for understanding. In this show, there is not good or evil, it is just \"Humanity\". I love it so much!",
    "Crikey what a show! Easily one of the best show going right now. Better than Viking's, and dare to say as good as GOT. Netflix dont screw this up and cancel this show. So much more to do and give us viewers... Acting is top notch and is just an all around great show. Give it 4 episodes and then make your own decision. If you watch all seasons you will be screaming for more. Uhtred is one of the most original type characters on TV and is just full of pure bad-assery!!!",
    "Hooked me superb show I love this show It has it all history drama comedy great writing along with settings and acting I have just started to watch the series from the first season again it is that good . I think this show and Vikings are 2 of the best action packed captivating and superb historical dramas of the decade. Such a shame as I think it is on its last season as is Vikings I will miss such excellence",
    "Alexander Dreymon I'm a sucker for period pieces. Anything about Vikings is likely to grab my attention. The Last Kingdom is remarkably well done. I don't know enough about movie making to say with confidence but it seems like the director knocks it out of the park. Alexander Dreymon is a name I hadn't heard before but he absolutely breathes life into the main character. The writing helps for certain, Uhtred is everything you want in a protagonist. I have no clue how historically accurate this show is but it entertains me in a way only the best can.",
    "Binge watching into S4 and apprehensive about S5 At first, I was wondering how comes I am six years late with watching this show. I thought it's a single season thing. Finishing S1E8 and being somewhat sad it was over, I saw it wrote \"next episode\" in the Netflix lower right corner. What?! Imagine my joy - there were 3 more to see (and hopefully Season 5 on the way). Now I am literally living with the heroes for days and enjoying every moment of it. Brutality was a bit of an issue in the initial episodes, but such were the times, and we have no less of brutality in the modern days - so I got over it. Phenomenal epic drama - thrilling and captivating. Formidable characters brought to life by excellent acting. They make you love them, despise them, wish them a painful and humiliating death, fear for their lives, cheer for their victories big and small... The dialogues are natural, yet full of wit, fun and wisdom. In my experience of the show, the only bump on the road was the everlasting baby face of King Alfred's plain and irritating wife, whatever her name was (wife I call her, since she could as well wear an apron, chop onions or hold a ladle to no one's surprise). Even after Alfred's death, she looked younger and fresher than her 30-year-old daughter. That said, I strongly recommend this show. Personal favourites - Uhtred and Beocca - but there are many memorable characters in both the heroes and the villains, and the ones in between.",
    "Historical, Addictive, Heartbreaking, Excitement. I didn't discover this show until 2020. It is without a doubt the only tv series that I can watch over and over and still be entertained. The show although not 100% historically factual still shows battles that did take place and how King Alfreds dreams of a united England came at a cost. It is a fantastic show based on the novels by Bernard Cornwall.",
    "A Fair Stab At A Brilliant Book Series The Last Kingdom comes as welcome relief and an antidote to the likes of \"Vikings\". Where the aforementioned series portrays the English as terrified buffoons virtually bending over to receive the Norsemen's thrust, The Last Kingdom gives a more balanced and, almost, historically accurate take on the era. The gradual coming together of the kingdoms of Aengland and the near 300 year struggle against the Vikings could fuel an epic series of full length movies, but Cornwell's books concentrate on a 50 year period with Alfred the Great at it's centre. This TV adaptation is a fair attempt at bringing the much loved books to life, however I would have liked to see more of the BBC's gigantic budget reserves used to greater effect in some of the scenes. More battle scenes, a little more gore and a few more players would have improved things. The books describe the fear, the nerves, then the stench of intestines and their contents spilled out, teeth, fingers littering the floor, you're right in the action - I would have liked to see more of that! I'm also not convinced by Mr Dreymon. Too much \"Legolas\" about him, not enough \"Aragorn\", if you'll excuse the Tolkien references. Uhtred in the books grows wiser with every day, he soaks up experience and thinks fast, our BBC man just doesn't quite have the presence to flesh out that character. All in all, an enjoyable series, great education for Americans and others in that it shows that the English were as tough as anything the Danes could do, special mention must go to David Dawson for his amazing portrayal of Alfred the Great...just a brilliant performance playing this complex and misunderstood king.",
    "Best show available The history and great storyline keep you always wanting more. This show has some of the best acting and character development going. It is truly a show that everyone will love watching!",
    "Season 4 Ruined It For Me I watched the first 3 seasons 3 times and enjoyed every minute of it. Season 4 on the other hand was just bad. Too much Drama and and too much feminism and totally unrealistic time sprinting through the events..",
    "The show that makes Game of Thrones season 8 look like a masterpiece!",
    "10/10 Exceptional TV Series, that seems to get over looked. Exceptional TV Series, that gets over looked possibly because of Vikings, but which is in my opinion is quite possibly one of the best tv series to grace our screens in recent years. I first watched The Last kingdom around 2018 so I missed the first couple of seasons due to being out of the UK, and I have to say on first viewing it didn't initially draw me in, and I've never left any reviews before for what ever reason, but subsequently having binge watched all the seasons I was hooked, and I since binged several times since, it just gets better and better. It's amazing how much you miss initially, then as you watch again more is apparent, all I have to say after several viewings it is just an enormously well acted, well directed, well written production, one which all the cast and crew should be immensely proud to have been a part of. In my all time top 10 of great watches.",
    "Addictive Great show! Better than Vikings! Nipping at the heels of GOT!",
    "A must-see for historical fiction fans The Last Kingdom is my personal favorite show of all time. Set during the invasion of the Great Heathen Army and its aftermath, it follows Uhtred, a warrior born as a Saxon but raised as a Dane, as he treads the line between the warring factions while going after his own stolen birthright. It's also based on the Saxon Stories by Bernard Cornwell and follows the storyline quite faithfully by most accounts, though I haven't read the books (I can't get through a Cornwell book without being bored to tears). Gratefully, The Last Kingdom is far from boring. It treads many years of political strife, the rise and fall of many well-developed characters of all moral persuasions, and showcases many heart-pounding battles with historical relevance. The culture of both Saxons and Danes is highlighted here, as is the era's Christian influence and Norse mythology. Never before have I been so immersed in any show, let alone one with a historical context; every time I binged episodes I found it hard to get into anything else. Aiding this immersion is a tear-inducing soundtrack, characters you learn to love, gorgeous vistas, and a sense that you've been uprooted and transported to a vastly different time. Let me double down on the \"great characters\" bit for a minute: the characters here are what differentiates The Last Kingdom from many shows that try and fail to reach its heights. I could be here all day listing characters and why they're enticing, but I'll settle for this: Uhtred is a tragic hero that is also a puzzle, Brida's character arc is as full of rises and falls as she is of powerful warrior moments, Aethelwold is the sneakiest snake that ever sneaked and was a joy to love (or hate), Alfred the Great (played to perfection by David Dawson) is an intriguing yet troubling juxtaposition of kindness and betrayal, and each and every antagonist of the series is interesting, even if they don't last very long (Haesten, Aethelred, Cnut, Skade, Erik, Bloodhair, the list goes on). The Last Kingdom is notorious at killing off its characters, but it develops them so well that with each death the viewer feels either grief or the sense of the end of an intriguing era. This show is simply magic. My only criticisms of the show are nitpicky. At times it's obvious they use a filter to darken scenes with a gray-blue tint because natural lighting didn't fit the mood. Some fighting scenes were filmed in such a way that it's hard to grasp what's happening (usually during the large-scale battles involving clashing armies). However, this lasts for a few seconds out of the occasional episode, so it's not an issue. Another reviewer mentioned how frustrating it is that Uhtred is constantly treated horribly by the people he tries to aid and ally with, and I agree it can get frustrating. But without conflict, the show wouldn't just be boring, it wouldn't exist at all. The Last Kingdom is a masterpiece of historical fiction that all lovers of history owe it to themselves to try. Many have likened it to Game of Thrones despite the genre difference because of its nasty game of politics, but the difference here is that The Last Kingdom keeps its quality consistent throughout its seasons. As old characters die off, those who have been rising in the background take center stage until the cycle begins anew. Watching this show feels like a journey because its plotline continues to develop in new and intriguing ways, mercilessly pulling its transformative cast along with it. Netflix, thank you for bringing this show back for a third and fourth season, but Uhtred's story is not over. You are lucky to have such a show under your control; don't drop it. Amazing show, and the only one I can in good faith give a 10/10. Please support it!",
    "Some light in the Dark Ages I really like this which is another hit rather than a miss for the BBC. The Last Kingdom now joins Wolf Hall and Poldark in its ability to transport me back in time. Obviously this depends on how well you know any given time period. I am more pedantic about the last two world wars so not every BBC effort achieves this. People might love The Crimson Field but in no way did people speak or say things in WW1 that they did in that particular drama. For a reality check watch The Great War: The People's Story. Leaving that era aside we have now watched four episodes of The Last Kingdom and all have been excellent and \"believable\" if that's the right word for what remains fiction. It must be difficult to get it right hence some people on here slating this series. Only slate it for historical inaccuracies if you are that obsessed but don't knock the overall effort for what remains a \"Dark Ages\" drama. Yet it actually feels like it was way back then. This has been achieved in far less stylised way than say Game of Thrones which is obviously not really grounded in any time period yet remains an excellent drama most of the time. Also, with The Last Kingdom there is no gratuitous violence just for the sake of it as there is with some other offerings set in this same time period. You only get occasional violence within the story line so this gives you the opportunity to get interested in the love scenes too and these are not over the top. Also people are allied in all sorts of ways. The end scene in Ep4 was brilliant and sums up what I mean. You can't expect killings all the time. If you want more detail buy a copy of the Anglo Saxon Chronicle. Apparently there are plenty of deaths mentioned in that!",
    "The absolute best if Viking era shows!! Loved this show since series 1, fantastic mix of historically correct detail and fiction played by amazing actors. Its gripping, a must watch!",
    "Don't let this show go under the radar!!!! Incredible cast, plot, battles So, 2 years ago I let my husband convince me to watch this show, TLK. I love a battle scence so agreed. A charismatic main character, betrayal, family feud, blood shed and epic battles. I was hooked. What sets this show aside is the talented cast and brilliant writing. The plot is detailed and takes account of each character's development. It becomes just as much about Lord Uhtred as any other character. In fact, my favourite character changes each time I rewatch. Incredible soundtrack adds to the joy. I love that Uhtred Ragnarson is brave, a true warrior and head strong but also flawed. The villains you love to hate and throughout the one liners are witty and quick. Don't miss out. Do yourself a favour - join The Last Kingdom world.",
    "Best show ever! This is the best series I have ever watched! It has an an amazing cast. All the characters bring their own A+ skills to this series! I am obsessed with this show! I love love the character \"Uhtred\"! Not to mention the actor who plays him- Alexander Dreymon! HE is sexy/gorgeous! I have rewatched this series about 6 times now! I just cannot get over how good it is! I dont want it to end!",
    "It started as a another classic Cornwell must watch.series.... I enjoyed series one and two (the BBC series) they were dark and well thought out. I started series 3 with high hopes and was slightly disappointed, not as well written, far more soap than epic. But series 4 is pure soap, it doesn't hold a candle to the first 2 series. I'm grinding through S4 but I'm not enjoying it nearly as much. Thanks Netflix. ;~(. Gone from a 9+ to a 7-. Too much chatter and sex, not enough quality historical epic.",
    "The gold standard for me This is *the* show I compare other shows to. I can't emphasis enough how much I appreciate the direction this show has gone and the path it has followed since the beginning. This show sets a high standard for others to follow. Every bit of this show is intriguing, interesting and exciting. There's not a moment of boredom with this show and the setting, story and immersion is just the best in the industry from my peasant point of view. With each season I've watched the show from the beginning and not once have I felt even slightly bored or \"I know what is going to happen so meh\". It just grips you and takes you with it to an adventure and every time you feel this sparking within you as you follow the, admittedly ridiculous and cheesy, journey of Uhtred, son of Uhtred.",
    "This show will pull you in if you give it a chance, a well done historical drama I've heard of the books but have yet to read them. I binged watched the 1st 3 episodes so far, and was surprised that I really liked the adaptation. I've seen other pieces of this period (The Vikings, The Bastard Executioner) recently and liked this much more. I'm sort of glad I didn't read the Cornwell's books before seeing the show, as I did with the Outlander Series. I was constantly comparing the Outlander books to the show and disappointed at times that it was not same as the books which I loved. I'm sure the readers of Bernard Cornwell's books will be doing the same. However, not knowing the original story, I can just judge the show on what I see presented. So far, I really like the story. I like the acting all the characters are doing a great job. I very much like Alexander Dreymon's portrayal of Uhtred. He brings the character to life. His acting makes me root for the character, and makes me want to see him succeed in gaining his land's back. I also liked Emily Cox portrayal of Brida, Uhtred's strength, I'm curious as to why Uhtred let her go back to the Danes when he obviously loves her. Their story has captured my interest and I'm looking forward to the rest of the episode. My one disappointment, was that I was sorry to see Matthew Macfadyen was only in the 1st episode. I really like Mathew's acting and would of liked to see more of him. But that being said, the casting is brilliant. All the characters seem to fit my impression of who they should be. BBC does a great job with its historical dramas. I don't think you can go wrong watching this show. I'm glad I've something else I like now that I have to wait for more of the Outlander series. This will do nicely. Watch out Sam Heughan, Alexander Dreymon has that same sex appeal.",
    "Destiny is all !! Stumbled across this beauty on Netflix while looking for a new series to watch. After the first episode I was hooked, you instantly want to watch episode after episode! Great story line, wonderful acting... Not one episode across all 4 series was boring, it kept you on the edge at all times. It's well worth watching!",
    "Hated the main character for a few episodes, then fell in love.",
    "Love this series I began watching this series when it first came out in 2015. However, life happens and I was unable to continue watching. Just recently I discovered the series and binged watched all five seasons. I can't wait until the 2+ hours movie they are filming comes out, completing the series. Set us hope it will not take two or more years!",
    "By far the best! This show will definitely go down in my books as one of the best! It is filled with suspense, adventure, great dialogue, a little romance and humor! Based on the historical novel series \"the Saxon stories\" this show is set in the 9th and 10th century. It showcases the clash between Saxon Noblemen and Viking Danes, with a Saxon king desperate to hold on to his kingdom and protect the vision he has for its future from invasion and destruction from the Vikings, all whilst trying to stay within the boundaries of his beliefs. With every season new and exiting characters are added which keeps the storyline fresh and exciting, and helps the main characters evolve. Definitely a worthy watch for those who enjoy history based, old world or Viking themed interests.",
    "Fans of the Klingon Empire OK, once in a while, some of the choices made by some of the characters in this historical drama/action series make you want to scream at the TV but, films and TV have always been that way. And without that kind of thing going on, we wouldn't have the cliffhanger. I started watching this as I liked the Vikings but, apart from that strange accent they do to indicate that they are speaking in old Norse, I think this was a pretty different series. Historical action/drama, yes, but more Kilngon warriors kick ass than Vikings. Instead of Sto-Va_ka, we got Valhalla. The clothes and hiarstyle were pretty much the same for the Danes. And the dialogue was sopt on. I suspect that \"qapla\" in English must be something like \"destiny is all!\" It would have been nice to see Michael Dorn do a cameo as the character of Uhtred was clearly a carbon copy of Worf only in reverse. These are not criticisms, by the way. Absolutely loved it from start to finish. I just hope we get something like this from the Star Trek franchise one day. Wonderful!",
    "You can tell Netflix is at the helm in the end Season 5 starts with comedic scenes and then there's a black priest. Trust me, despite what some might have you believe, those did not roam the 9th or 10th century English countryside. That's just 21st century Hollywood neuroses.",
]

add_show("The Last Kingdom", the_last_kingdom_reviews)

✅ The Last Kingdom predicted as: Balanced
✅ Added 69 reviews for The Last Kingdom


In [77]:
dora_reviews = [
    "I think this show is wonderfully entertaining and educational for preschoolers. My child learned to count and picked up many skills just from watching it.",
    "A really great show for toddlers that teaches Spanish and basic problem solving in a fun and engaging way.",
    "Dora is fun, colorful, and interactive. Kids love participating and repeating phrases while learning new words.",
    "The show is highly repetitive, but that repetition actually helps young children learn and remember things.",
    "It teaches language, counting, and problem solving in a way that is simple and accessible for very young viewers.",
    "A fun and educational show that encourages kids to think and interact rather than just passively watch.",
    "My child became more curious and even started using Spanish words after watching the show.",
    "The interactive format is great for young children and helps them feel involved in the story.",
    "It’s clearly designed for preschoolers and works very well for that audience, even if adults find it annoying.",
    "The show is simple, colorful, and engaging, making it perfect for toddlers learning basic skills.",
    "Some parents may find it repetitive and loud, but children respond well to the format and enjoy it.",
    "It promotes learning through repetition, songs, and participation, which is effective for early development.",
    "The show encourages kids to solve problems and follow steps to reach a goal.",
    "It’s not meant for adults, but it does its job well as an educational children’s program.",
    "The repetition and slow pacing help children understand and follow along easily.",
    "The show teaches basic Spanish vocabulary and introduces children to another culture.",
    "It’s engaging for very young children and keeps them entertained while learning.",
    "The animation is bright and appealing, which captures children's attention effectively.",
    "It encourages participation and builds confidence in young viewers.",
    "The show provides a safe and non-violent environment for kids to learn.",
    "It promotes kindness, teamwork, and helping others through simple stories.",
    "The bilingual aspect adds strong educational value.",
    "Children enjoy the songs and characters, even if adults don’t.",
    "It’s effective for very young kids but loses appeal as they grow older.",
    "The structure is extremely repetitive, with nearly identical episodes.",
    "The constant repetition and shouting can become irritating for parents.",
    "The characters often ask obvious questions, which can feel unnecessary.",
    "The show can feel overly simplistic and lacking depth.",
    "Some viewers feel it underestimates children's intelligence.",
    "The format is predictable, with the same structure every episode.",
    "The constant audience interaction can feel forced.",
    "It can be frustrating to watch as an adult, despite being effective for kids.",
    "The storytelling is very basic and formula-driven.",
    "Some episodes feel slow due to repeated pauses for responses.",
    "It relies heavily on a fixed pattern rather than creativity.",
    "The voices and tone can become annoying over time.",
    "The show sometimes rewards answers regardless of correctness.",
    "It lacks complexity, making it unsuitable for older children.",
    "The simplicity can be both a strength and a weakness.",
    "Some parents find it overwhelming due to constant noise and repetition.",
    "It is best used in moderation rather than extended viewing.",
    "The educational content is clear but not very deep.",
    "It succeeds in teaching basics but doesn’t go beyond that.",
    "The characters are memorable and recognizable for young audiences.",
    "It creates a sense of participation that many kids enjoy.",
    "The show has a very specific target audience and sticks to it.",
    "It’s harmless and generally positive in its messaging.",
    "While repetitive, it achieves its goal of reinforcing learning concepts.",
    "It’s not entertaining for adults but is effective for early childhood learning.",
    "Overall, it’s a simple, structured, and educational show with clear strengths and limitations."
]
add_show("Dora the Explorer", dora_reviews)

✅ Dora the Explorer predicted as: Balanced
✅ Added 50 reviews for Dora the Explorer


In [78]:
you_reviews = [
    "Exciting Thriller! You was so much better than I thought it was going to be. I know it's been hyped up ever since the first season and has gotten good ratings and reviews from nearly everyone but I still didn't think I was going to like it. I was made to watch the first season and loved it. After that I binged the rest of the seasons on my own in record time. I'm not saying it's my favorite show or even close to that but it's one of the most surprising shows I've seen. It was originally suppose to be a miniseries but is such a popular show they keep renewing it. Season 4 drops early 2023. Penn Badgley does such a great job with this character, he's very believable in being a lovable psychopath.",
    "Better than expected! You is every bit as good as people have said it is and is pretty addicting once you start it. I've been putting off watching this since it first came out but finally found the time and now I'm mad I waited so long. For some reason I thought this was a show who's key demo was women. I was wrong. The thing that surprised me the most was how much humor is in it. It made me laugh a life more than I expected. Now, people saying that this isn't as good as Dexter are absolutely right (not much is), it isn't, but it definitely holds up on its own and thought it was overall a pretty good show. Penn Badgley does a fantastic job as the lead and Elizabeth Lail and Victoria Pedretti are both great in their roles as well. Now that I started it I can't wait for more seasons to come!",
    "Oddly Uncomfortable I watched this without expectation. I hadn't heard of the book. I hadn't seen any previews or descriptions. What i found was a pleasantly dark show. It's uncomfortable and unpredictable and has what is hopefully an amazing story to tell from a unique perspective. We're put into the head of a stalker who seems to be a hair's width away from violence. The tension is held well. But what REALLY made it feel uncomfortable for me was a rather odd choice from the network. Swearing is censored out. There are these conspicuous blanks in the dialogue. It had me obsessing. When you've got a show this dark, who the hell do they think they're going to offend with a little swearing? Are they expecting kids to be up watching it? It's content is far more offensive than swearing could ever be. Apart from that one VERY annoying point, it's a brilliantly put together show. I'm looking forward to watching more (though I'll probably find myself swearing at the TV about the missing dialogue).",
    "Made for Penn Badgley I love this! It's dark but yet sexy at the same time. Penn does an amazing job narrating and he makes creepy seem alluring. I can't think of any other actor who'd do a better job.",
    "First 3 seasons were great, but season 4? I found the first three seasons to be very captivating and I was really looking forward to Season 4. Unfortunately Season 4 (part 1) has been a challenge to get through. I just can't relate in anyway to this friend group he has gotten into so quickly, and not sure how anyone can get attached given their attitudes. Truthfully this 4th season part 1 is just plain boring as a result. If you end up not really caring for the characters or connecting with them, then what does a show leave you with? But let's not forget seasons 1-3 (particularly season 1!). That era demonstrated excellent writing, wild plot twists and just an all around excellent viewing experience. 6 out of 10 overall as a result. Ps. Update - oh boy part 2 of season 4 is even more unfortunate.",
    "Season 5 is the BIGGEST disappointment of the year The story really does shift this season-everything about it feels different. Joe this season isn't the same. He's not even trying to think. He's just an ordinary killer now, like a teenager who can't control his lust. There are a lot of wasted elements and a lot of characters who aren't used in a meaningful way. Well, I'm sorry for whoever liked this season, but it feels like a completely different show.",
    "Season 2 is even stronger than the first one. Main characters were much more convincing than the ones in the first season. Thrilling again and hoping for a next season.",
    "A great show with an incredibly underwhelming ending The show started to go off the rails after season 2, but it was all still enjoyable due to the consistently amazing performance of Penn Badgley. Season 5 started off with a good premise with a few twists that will be predictable to many, but were still entertaining to watch. Around the midpoint it started to lose its footing. The best way I can summarise season 5 is that it's a good season with many high points but a very bad final season, lacking the finality this show deserved.",
    "Should Have Ended at Season 3 As a long-time viewer, it's painfully clear that You ran out of meaningful story after season 3. What once felt fresh, suspenseful and cleverly twisted has now spiralled into something more akin to a soap opera. This final season is a mess of convoluted storytelling and miscasting. It leans too heavily on shock value and forced drama, rather than character development or logical progression. A once-great series diluted by the need to keep it going.",
    "Used to love it First 3 seasons were amazing. I loved it. S4 is just so boring. I really want to like it but all the characters are so unlikeable and the storyline is awful. I find myself drifting off or looking at my phone and that's when I know it's lost on me.",
    "Freaking. Excellent. Great acting. Inventive storytelling. Truly unsettling and underlying creepy. I can't wait to see where this show goes.",
    "Season 2 is quite good Season 1 was okay, entertainable. Season 2 is more mature and Love is a great character. The season is more thrilling and the story isn't that cheesy. Really enjoyed it!",
    "great series apart from some careless moves Great series, reminding me of the iconic Dexter. Acting is top notch, specially from Penn Badgley, completely unknown to me, but capable of portraying the complex, lethal and sensitive character of Joe. Victoria Pedretti on seasons 2 and 3 was just perfect. All 3 seasons delivered, with enough twists and last episode chaotic situations.",
    "From Masterpiece to Mess: How You Lost Its Way You has been the center of my attention since 2019. Season 2 took my experience to another level. Victoria Pedretti entered the scene and completely stole the spotlight from Penn Badgley, turning You into a masterpiece. Season 3 had weaker writing, but it still delivered solid characters. Then came Season 4 which felt like a major step backward. The fifth and final season started off well, but it went downhill fast.",
    "Season 2 lifted up the series! Penn Badgley's superior and creepy acting as the enigmatic and psycho lead character is outstanding. Victoria Pedretti who shone in season 2 almost singlehandedly lifted that season all by herself. You will stand the test of time due to its freshness, subject, superior acting and unique approach. Highly recommended series to watch.",
    "Whoa. I read the novel a year ago. The book is dark and has a comedic shade to it. The TV show is dark but the darkly comedy shade is still missing. The show is great. The actors are faithful in portraying the characters. Pen Badgley as Joe does a great job. I hope that the show sticks to this level of superb and gets better.",
    "CREEPY BUT EXCELLENT Wow I loved this! I finished it in 2 sittings. Penn played this part BRILLIANTLY! He convinced me to love him at times yet hate him at others. Each episode something more creepy and messed up happens yet you just need to keep watching!",
    "You: Not perfect, but not far off. Penn Badgley was casted as Joe which I think was a perfect casting decision. Badgley makes Joe seem sinister and imperfect, which is exactly what makes the show so intriguing. The way the show alters by Joe's thoughts is what keeps it so raw. The writing is formed in a way which makes you sometimes side with Joe, which is why Joe is such an important and interesting character.",
    "Relationships, defined. This is an incredible show. Every season has kept me intrigued for what's to come. I also believe Penn Badgley was MADE for this role. The voiceover actually helps grip you into understanding his point of view and helps carry the show extremely well.",
    "Season 4 is disappointing Season 4 is riddled with strange dialogue choices, poorly thought out plot lines, and an uninteresting and lazy mystery. It feels somewhat lazy and rushed. Joe isn't as vile, the side characters aren't as interesting, and I find it hard to care about anything in this season.",
    "Season 4 is a Bust The first two seasons of this show are strong. Season 3 is up and down. Season 4 is almost ridiculous. By season 4, the show is a parody of itself, recycling plot points, going into soap opera territory. Terribly disappointing by the end.",
    "Season 4 has ended it for me Season one - absolutely LOVED it. The story, the characters, the acting. Season two was very good as well. I feel that I sort of forced my way through season three. Then comes season four. I couldn't even make it through the first episode.",
    "Skip the 4th season Hands down, I absolutely loved the first season. Season 2 introduced Love Quinn who is even more intriguing. I can't even describe how terrible the 4th season is. Nothing is worth your time here.",
    "to hate YOU is to LOVE it What makes YOU so different is its social commentary to our generation. This is one of the very few shows which changes its setup, adds new cast and all three seasons can stand on their own thanks to brilliant performances from the leads.",
    "A little like Dexter, less predictable You is Dexter updated to a modern age. It's a new idea and fresh, and highly watchable, both for lovers of modern romance and for lovers of psychological thrillers. Fantastic acting, realistic look at modern phones, and a subdued yet perfect modern soundtrack.",
    "Good premise The show has a hint of Dexter and is so good. The acting is spot on from all characters while Penn Badgley is outstanding. Penn's performance of a dual personality is riveting.",
    "What happened? Absolutely loved season 1 of this show. It was so clever, innovative, creative, like a book you can't put down. Toying with ideas of morality and which characters drew our sympathies. Season 2 was throughly engaging. Then season 3 was just the complete opposite. No cleverness, predictable twists, bizarre storylines, plot holes everywhere.",
    "A thriller with a compelling plot Season 5 delivered a good plot showing Joe who he really is and what he is willing to do to get what he wants. Season 5 felt like season 1 with its quality. The ending of Joe was quite satisfying. You has a strong final season.",
    "Downhill After Season 2 The first season was incredibly well written and cast, but every season has gotten worse since then. Season 4 in particular uses familiar and unbelievable storylines. The character of Kate is particularly annoying. This season grasps for ratings and makes you hate Joe Goldberg, whereas before you kind of used to root for him.",
    "Could do without season 4 The series in itself is amazing and seems super unique. A man who believes in love but loves obsessively to an unhealthy degree is a great premise. Season 1 was a good introduction. Season 2 was amazing. Season 4 wasn't great at all and ruined the ending.",
    "Weak finish to a classic Like anyone who got hooked on season 1 and binge-watched every episode that followed, we were looking forward to the closing season. Unfortunately, it fell rather flat. The magical appearance by the end of three people who earlier appeared to have died was also difficult to swallow.",
    "Smart Writers could've made this sooo good Season 1 is a really cool plot with a really cool protagonist. Season 2 is the best season of YOU. It introduced the best love interest for Joe Goldberg, Love Quinn. Season 3 hit rock bottom. Season 4 surprisingly I was liking very much. Overall Penn Badgley is a hell of a great actor.",
    "Lifetime's Best Show Penn Badgley's portrayal of Joe carries the show. His dark, witty performance encapsulates the entire show. He walks the antihero tightrope beautifully and it makes for engaging cinema.",
    "Love Hate Relationship The series is very cleverly written along with Joe's point of view to everything, it's beyond twisted that it becomes intriguing. Penn Badgley made this series through and through. The way he can look so innocent with his eyes and facial expressions is remarkable. He is a great actor with lots of talent.",
    "Unique series I personally liked season 1 much better than season 2, because Joe's character was just so new and unique. I was just so invested in finishing the season. I still would like to watch season 3.",
    "Looks promising Season 2 tops the first one massively. All the twists and turns in this season make up for a good tv show with an amazing finale. Penn Badgley was amazing as per usual. This season gave me very Dexter vibes. Superb 8/10.",
    "Why are y'all so negative? I actually think it's a good series. The series is about a man named Joe who stalks women and does everything to protect them. We keep getting deeper and deeper into his mind, which is very interesting. It becomes clear that he tries to justify every action. You also get flashbacks from his youth which slowly make clear why he gets so obsessed with certain women.",
    "Most underrated show ever! This show deserves at least 8. Engrossed! Binge watching it, and haven't got bored!!",
    "Oh my God! Such a great show! Loved every part of the plot, great performances!",
    "CREEPY BUT EXCELLENT Wow I loved this! Penn played this part BRILLIANTLY! Each episode something more creepy and messed up happens yet you just need to keep watching!",
    "Excellent Thriller It moves fast and unexpectedly. There are a lot of plot twists and I found it very interesting. Penn Badgley expresses beautifully. His dialogues are written very smartly.",
    "Read the book, as well as Hidden Bodies. Was not disappointed by Pilot! The book was so interesting, uncomfortable, funny, terrifying. Badgley is pitch perfect to the character in the book. Joe's dry humor, intelligence and compassion make him lovable, which the viewer must reconcile with a deluded sociopath.",
]

add_show("You", you_reviews)

✅ You predicted as: Balanced
✅ Added 42 reviews for You


In [79]:
#12
got_reviews = [
    "Till season 7 very likely the best TV show ever made in terms of suspense, character development, plot, effects, acting.... 10/10 with no doubt! But S6E10 is probably the point where you should stop and imagine your own ending. It is guaranteed better and has more character development. Otherwise you may end up rating it down by more than just one and are overcome by disappointment...",
    "Seasons 1-6: outstanding. Deep world-building, intelligent dialogue, epic set-pieces worthy of Hollywood, expert acting, detailed plotting, genuine surprises, great soundtrack used sparingly. GoT is worth watching for these seasons, definitely. Season 7: you suddenly notice the writing and plotting has taken a sharp dive downwards in quality. Season 8: this drop chasms down to depths you didn't think possible.",
    "I'm feeling so heartbroken to see everyone criticising my favourite show of all time. After all these years hardwork, creating and developing each and every character from bottom to top, memories, emotions throughout the process all went to vain for one bad making. Could you please delete the whole season 8 and make a new version of it?",
    "It was a master piece. It was written to the perfection. It was mesmerizing. It was gripping. But yet, I cant hate it enough after final season. It was not a let down, it was a BETRAYAL!",
    "Seasons 1-6 are truly outstanding television, right up there with The Sopranos, Breaking Bad, Mad Men, The Wire, best of the best. Season 7 was a drop in quality but still somewhat enjoyable, and then came Season 8 which was a blunder of epic proportions.",
    "The end ruined it all for me. I wouldn't recommend it to my friends knowing how bad they handled it in the end. I have never seen such a good show fall apart like this.",
    "Game of Thrones boasts an intricate plot with complex characters, stunning cinematography, and intense action sequences. Despite being disappointed by the final season's storytelling, Game of Thrones still remains a must-watch series.",
    "Game of Thrones is absolutely, without a doubt, one of the best TV shows ever created. Yeah, the last few episodes of Season 8 weren't that good but the first 7 and a half seasons were so amazing that it still gets a 10 from me!",
    "Not only is it a rare television show that does its original source material justice but it is on its own merits one of the finest, most addictive and consistently compelling shows in recent years.",
    "This is the first series I would recommend to anybody. It is an amazing piece of work, the most stunning TV series you will ever watch. Why i gave it 9/10 because of the last season, it's still good but very disappointing compared to all the seasons.",
    "Started off as the greatest series of all time, but had the worst ending of all time.",
    "It's universally thought of as one of the best series in the history of television. Obviously everyone knows the last episode was absolutely awful, even most of the actors have come out and said they didn't like it. Having said that, it doesn't take away from the show being amazing.",
    "Season 1-4 were practically perfect. Season 8 the biggest disappointment I have ever seen. I have never seen anything more terrible. Overall i still give it a 9/10 just because of the first half of the show.",
    "A series like never seen before which rocked last season on a flop!",
    "Legendary.... Best Thing to watch in entire Milky Way...... No words to describe....",
    "I loved this series so much. Then came season 8 episode 3 and 4. Such a disgraceful ending. Seasons 1-4 10/10 Seasons 5-7 9/10 Season 8 3/10.",
    "Game of Thrones is a beautiful anchovy sundae. You don't even want the sundae anymore, in fact, you can't even look back on how nice that sundae used to be without thinking about all the stupid stuff they did at the very end.",
    "I was a big fan of Game of Thrones ever since it first aired on HBO. However, after waiting for two years, I felt that the season didn't rise up the rest of the show. The writers and producers didn't do the show justice and rushed the story.",
    "As an avid reader, I am often disappointed by what is represented by the silver screen. I was absolutely shocked to watch the Pilot and realize that it followed the book almost to perfection.",
    "For the people who haven't yet seen it, is it worth it to watch anyways? Seasons 1-4: 10/10. Seasons 5-6: 9/10. Season 7: 7/10. Season 8: 3/10. MY WARNING: DO NOT get too emotionally attached to the show.",
    "This show is stunning. It is complex, multi-layered, surreal, vibrant, imaginative and it draws your eye in to the surprising level of detail. Even if you have never watched a fantasy show before, you should not miss this.",
    "At first I did not want to watch the series. But once I started watching it, I couldn't stop. It became more and more exciting from season 1-8. I was disappointed with the end of the 8 season!",
    "One of the best series I've ever seen, but the last season was ruined.",
    "The first season was a bit overwhelming, what with all the houses and characters thrown at you. You can feel a drop in quality towards the end. The conclusion is anticlimactic to say the least.",
    "The cast is great. The setting, the atmosphere is perfect. This is NOT Lord of the Rings. It's based around great families on different sides of the world, the stories their members have to tell and the secrets they hide.",
    "I am proud to be on the same planet as those who created and put this work into fruition. When one looks at the technical accomplishments and the multiple plot strains that have been kept in balance, it is truly a wonder. I doubt at this late stage of my life there will ever be another series to match this.",
    "It use to be my favorite show by far and i never thought that could change, but it did. Last season i watched only once but all the seasons before that i have seen 3 times. You really mess it up with the last season.",
    "Game of Thrones may belong to the Fantasy genre, but the world of Westeros has been so amazingly well thought through and is inhabited by characters that are so well drawn and credible that everything you watch feels real.",
    "Ever since I've watched the last available season of GOT I haven't been able to get into any other show. The way the multiple story lines and characters intertwine with one another is absolutely genius!",
    "For me it's one of the best series ever made. About the ending I think it was a good and expectable ending. You know how great this was when the big question is: Which series will become the next Game of Thrones?",
    "I believe the graph of the show starts at the lowest point and from ending season 1 gets a drastic rise. In my opinion it peaks at Season 4 and slowly declines from there. It remains my favorite show till date, and yet the perfect example of Messing up something absolutely beautiful.",
    "This is The Greatest thing I ever seen on screen. Cast, Story, Filming, Involved character, Friendly Screenplay - all fabulous.",
    "I wasn't into fantasy before I started to watch Game of Thrones, but then I started and after just a few episodes I found the series so great that I didn't do nothing but watch it. All seasons are great.",
    "First of all, I am a huge fan of the books, and I find it exhilarating to see a show that is so close to what I made in my mind while reading it. They keep true to the epically written book.",
    "For a series with such complexity and many-layered characters, you'd expect to get an epic ending but that wasn't the case. Shame on the writers to turn such a rich world into a pale and dumb pop-ish work at the last 2 seasons.",
    "It doesn't matter how it ends, it will remain the greatest tv show ever.",
    "Love, betrayal, greed, murder, corruption - all collides in mythical world of Westeros. George R. R. Martin's world is unlike any other fantasy world you have ever seen.",
    "Ultimately, this remains one of the greatest television series of all time, even with its complete failure of a conclusion. The cinematography, special effects, costumes, writing, acting, character development are all time great.",
    "I came late to GOT on TV. As an engaging and gripping series it blew me away. I've never binge-watched anything before and was completely obsessed with GOT. Hard to see what is going to replace it for my evening binge-viewing.",
    "This epic series is set in a fantasy world like no other. The cast does a phenomenal job. The special effects are fantastic. Overall I can only say Watch this as I loved it from start to finish.",
    "This is the BEST TV show I've ever watched. The CGI is top notch with masterful direction and twists and turns. Even though the ending is bit disappointing it still worth a watch.",
    "I watched all eight seasons within a week and was surprised at how many things I had forgotten. The show holds up very well.",
    "The ending of this series is a disservice to the public. The first seasons are great but the last season is devoid of content and value. The ending season ruins quite literally everything.",
    "Yes you can completely ruin a show with a terrible ending and that is what GOT will be forever remembered for: the worst final season of all time. An ending so bad it ruins the rewatchability of the show.",
    "Loved this series. Fabulous characters and story. Costumes and locations are exceptional.",
    "An epic series, most of the actors mastered their characters despite the bad season, it is still the best TV show in history.",
    "This show is just amazing, everything about it, even though its last season wasn't amazing, every other season was great. The action is just crazy and the episodes will just get crazy.",
    "Game of Thrones used to be one of my favourite shows on television. However Season 8 is one of the worst seasons of any TV show I have ever watched. It throws everything you ever cared about out the window.",
    "Have been delaying for years watching this as it is not normally my sort of thing. How wrong I was. Amazing show. By far the greatest series ever made. I loved the ending too.",
    "The first 5 seasons of Game Of Thrones are exceptional, a beautifully structured series. When it came to Season 5, the author and HBO had a disagreement and it is extremely evident from that point.",
]

add_show("Game of Thrones", got_reviews)

✅ Game of Thrones predicted as: Balanced
✅ Added 50 reviews for Game of Thrones


In [80]:
tvd_reviews = [
    "The Vampire surprised me as much as any show I've ever seen. I absolutely thought this was a show for High School kids but I couldn't of been more wrong. This show offers so many twists and turns that it will keep you entertained throughout the entire series.",
    "It started off that way for like the first half of Season 1 but around episode 10 or 11 it really picked up and just got better and better. This show is beloved by so many for a reason!",
    "Damon has the best chemistry with everyone. Vampire Diaries has taken me by surprise. It is a great show with great characters. Damon's character has exceeded my expectations and surpassed similar characters on other series by far.",
    "The Absolute Truth: The Vampire Diaries is infinitely better than Twilight. There are werewolves, hunters, witches, and even stronger vampires waiting in future episodes to make this one nail biting show.",
    "This is no doubt one of my FAVORITE shows of all time. I would like to thank the whole cast for giving us one of the most amazing, gripping, unbelievably FANTASTIC shows ever to be made.",
    "The most important thing to know about this show is the first four episodes are seemingly pointless. Then something magical happens. This is a show that takes risks: people die without warning, stories take only a few episodes, characters do things you wouldn't have thought possible.",
    "After watching episode 4 i got all excited and started to flap my hands wishing the next episode will come out already. Give it a chance. yes it may not be as good as the book series, but it beats twilight by far.",
    "It could've got 9 from me but the series was perfect till season 6 after that it is so boring.",
    "Even though this Series ended in 2017. Fans are still watching and obsessed with all the characters that make this show still one of the most watched Series in the world.",
    "If not for Elena's character this show would be much more enjoyable. One of the most annoying characters ever created. It's typical, unoriginal CW young adult drama, not Shakespeare people.",
    "Like most reviews have stated, after episode 4, the show takes a different direction and moves away from traditional vampire themes. It totally excels and turns into a very exciting and watchable television show.",
    "This show is good in the beginning where the characters must hide their supernatural nature from the normal people of the town. But as time goes, everybody either become supernatural or knows about it. Season 1 to 4 is worth watching.",
    "This series is simply epic. Damon's character was really great, one of the best written characters in TV history. I loved the characters of Elena Stefan Bonnie Caroline Enzo Alaric and everyone else.",
    "10/10 show. Way better than Twilight, while watching this you will cry and laugh. Some scenes are pretty emotional.",
    "This TV show is one of my favorites because first I'm a fantasy genre lover along with romance and this show got every detail about it. The way they added every type of supernatural creature without forgetting the drama and action in every scene.",
    "This show is absolutely amazing! I am a guy which may be a little weird, but this show is actually really good! It is bloody good fun and is great at keeping you locked into it!",
    "After watching The Originals, I watched this series. I found too many illogical things. I think the overall story is good but The Originals is one of the best series, and while this one doesn't quite measure up to it, it's still good.",
    "I have never read the books however I don't understand how anyone could expect for any TV series to be an exact replica of a book. I have been watching the show since the pilot episode and I absolutely love it.",
    "I can't remember how the show hooked me in but I have watched every episode since the show was aired. The show develops the relationships between all of the main characters in a manner that the book can only dream of.",
    "I avoided TVD for six seasons. One day while laid up in bed I decided to give a couple episodes a try, next thing I know I've binged through 6 seasons. The show was heartfelt, funny, irritating, intriguing.",
    "This is vital to a show like The Vampire Diaries, as the program is a typical case of once you get past the first few episodes, it evolves into something brilliant. Irrefutably, the best thing about this show is Ian Somerhalder as Damon Salvatore.",
    "It is one of the best teen drama present till date featuring all kind of emotions in one. The plot is just amazing and the music supporting the scenes makes the emotions more expressible.",
    "Like many tv series that stretches out for many seasons, the start to the series was interesting. But as the seasons roll along, it becomes repetitive and quite boring.",
    "I finally decided to watch the show last fall after it kept popping up on the front page of my Netflix account. Minus the supernatural topics, this show is real. It's got grit, romance, sex, violence, tragedy, and more.",
    "This show may have a fantastical theme with Vampires, werewolves, and witches, but beyond that the material is adult themed. Vampire Diaries, the spin offs The Originals and Legacies are much better than Twilight.",
    "The Vampire Diaries is like watching a train smash in slow motion. The characters are teenagers that interact with an impeccable middle aged rebellion. The dialog is expositional and rhetorical, lacking authenticity.",
    "This is my favorite of the CW lineup now. The cast is super attractive and there is lots of excitement each week. Elena is a much stronger and more modern woman than Bella Swan.",
    "TVD was a beautiful and interesting series as long as Damon and Katherine were in it together. Unfortunately, as soon as they forgot about Katherine, it went downhill after 8 seasons.",
    "This is a breathtaking TV-series. THE best Vampire TV-show to date! Better than Buffy, better than Charmed, better than Smallville and better than most supernatural series!",
    "The first three seasons of some of the strongest pieces of vampire media out there. Unfortunately some of the writers left and it became a mess of a show with ridiculous storylines.",
    "I watched all 3 seasons on Netflix over a few weeks recently and generally enjoyed it. Things started going sour somewhere near the middle of season 3 though and season 4 has been horrendous.",
    "This show is quickly becoming one of my favorites on TV. I love how each episode ends in a cliffhanger. The show always keeps me guessing with all its twists and turns.",
    "Eight seasons later I saw the end. The series is full of strong and dramatic characters made by actors who have shown talent. Nina Dobrev was very good as Elena and her disappearance from the seventh season was unfortunate.",
    "At first I thought this show wasn't as good. But after comparing it to other new shows this is almost a vampire classic right next to Buffy. Every episode is clever from start to finish.",
    "Don't predict it to be some kind of Twilight crap. Once you'll start watching it there's no going back at all. This show is so addictive that it'll hook you up to your seat.",
    "It's one of the best shows I watched. The chemistry between the characters is fantastic. I really like the first seasons, there are a few seasons that I think are not as good as the rest.",
    "TVD is a supernatural series which revolves around vampires, witches and werewolves. The series starts a bit slow but then picks up the pace and gives a truly entertaining ride. Top notch up to season 5 but after that it becomes a bit repetitive.",
    "Excellent casting, acting, writing, filming. One of the best series I've watched and one of the few that has compelled me to watch every episode. The storyline is fast moving, full of twists, and leaves me wanting more.",
    "I used to LOVE the Vampire Diaries when it first aired. Today when I tried to rewatch the show, I can see the cringe factor it actually has. After season 4 it really gets unwatchable.",
    "This is my fourth or fifth time rewatching this series since it originally aired. Every time I finish it I can't comprehend why they ever ended it. It's truly a tragedy. BRING TVD BACK!",
    "My favorite show to ever exist. It has drama, passion and great music. It really helped me get out of a depressing time in my life.",
]

add_show("The Vampire Diaries", tvd_reviews)

✅ The Vampire Diaries predicted as: Balanced
✅ Added 41 reviews for The Vampire Diaries


In [81]:
invincible_reviews = [
    "It's 2025 and I just started watching Invincible and now I'm mad at myself for putting it off for so long. This is so much more than another animated series. It's so brilliantly written with a plot you can't help but want to follow, the character development of a great movie and is beautifully animated.",
    "Invincible has been everything I expected and more. It currently has a perfect 100% in Rotten Tomatoes. The voice over cast has to be one of, if not the best, of any animated series ever. You don't even have to be a fan of animated shows to enjoy this either, it has something for everyone.",
    "If you were into The Boys you will definitely be into this show, a realistic version of what superior beings would actually be like. Blood and gore check, great voice actors check, fantastic soundtrack check, great story check and dialogue that actually feels important and emotionally powerful.",
    "Great voice cast and a very good story makes this series a must if you are a fan of the superhero genre. The story arc between mother, father and son is excellent. Just keep in mind that this is an adult show due to the violence.",
    "I bought the comics and i love both the show and the comics. Amazon definitely needs to pay more attention to invincible and give it the money to have super duper good animation. They already have a fantastic cast and all the voices perfectly suit the characters.",
    "Robert Kirkman's Invincible animated series is a gritty, subversive, and emotionally intelligent take on the superhero genre. The brutal twist at the end of episode one instantly sets Invincible apart. This level of complexity is rare and refreshing in the genre.",
    "Invincible really increases the standards in animated tv or web series by showing good character development, emotion, great ton of action and pg-18 violence. Best thing is, every episode is 40 minutes instead of 20.",
    "The writing in Invincible is one of its greatest strengths. The dialogue feels authentic, the pacing is tight, and the emotional stakes are consistently high. The series also doesn't shy away from challenging questions about heroism and the costs of power.",
    "I went into this show with minimal expectations and just expected it to be some cheap knock off superhero flick. Oh boy was I wrong. Nowadays you never really hear of shows that get better with each season. This show has me more and more locked in with each season.",
    "This show was very inconsistent. It seemed like it was made by multiple people who were all given a different scene to work on with no one person in charge. It alternated between plotlines seemingly intended for very young children and scenes intended for edgy teens.",
    "This series worth my time. It is so well crafted and has such good characters, but it seems that the core of the series cannot tell us anything new in this genre. The Boys has something different and new. This series absolutely defeats Marvel shows though.",
    "I love how Invincible explores the complexities of being a superhero and the responsibility that comes with it. The series is thrilling, emotional, and has a great balance of action and drama. Highly recommended!",
    "Violent, irreverent, entertaining. Seems like it will be able to hit all the same marks as The Boys but without budget limitations. The star power of this show is incredible.",
    "Invincible is everything modern superhero franchises wishes they were. It is a great subversion of what the audience has come to expect from the genre. The first episode sets up a world very familiar to other superhero shows, but then swerves to a new direction that is even better.",
    "Invincible isn't just another superhero show - it's a powerful deconstruction of the genre. The writing is razor-sharp. Mark Grayson is one of the most compelling protagonists in recent memory. What sets Invincible apart is its honesty.",
    "Invincible is hands down a 10/10 show. The characters feel real, with their flaws and struggles, which makes you actually care about what happens to them. The animation is stunning, especially during the fight scenes. Invincible feels fresh, exciting, and powerful.",
    "Season one of Invincible was probably one of the best debut seasons of any show ever. The voice acting continues to be some of the best I've ever seen. Steven Yeun really is an elite talent.",
    "I rated Season One 9/10, but Season Two was a huge disappointment. The storytelling and plot has become too convoluted and burdened down by whiny teenage emotional drama. Furthermore it was bad enough that this season is basically a soap opera.",
    "I have no clue of the source material but happy they kept it animated. Not what I was expecting but episode one had me hooked!",
    "Invincible is a great show but there were some aspects that didn't quite sit right with me. The dialogue can be corny at times and the animation isn't quite up to the quality I'd expect. I didn't enjoy Season 3 as much as Seasons 1 and 2.",
    "I watched the first episode and after the ending i just had to binge watch the next two. The plot is well written, characters are credible, voice acting is good. Mindblowing will be a recurring word as you watch this show.",
    "Invincible captures so much emotion. The soundtrack is absolutely immaculate. The characters are perfect in design. The storyline is perfect and not at all predictable. This show is so good it will force you to binge watch it.",
    "This is the best animated series I've ever watched. The battle scenes and character development are absolute nirvana. It has a unique and exceptional storyline that no other animation has come close to achieving.",
    "Invincible takes the classic Image Comics series and gives us the best mature animated superhero series since Spawn. This show is must-watch television.",
    "Season 1 of this knocked it out of the park. Season 2 is proving to be a bit of a chore to get through. The stiffness of the animation's really taking its toll, and there is an air of something missing.",
    "Without a doubt the best superhero franchise out there. There is so much more in it so I think that everyone should watch this standout in the superhero genre.",
    "Finally caught up on season 3, and the show hasn't diverged into bleak, repetitive cynicism like the Boys. Invincible has a brilliant conceit in how it deconstructs and reconstructs the Superman archetype.",
    "The Greatest Animated Comic Book Show Ever Made. From the very first episode, Invincible grabs you by the throat and refuses to let go. What really sets Invincible apart is its heart - the raw emotional weight behind every punch, betrayal, and revelation.",
    "I am completely unfamiliar with the comics and I don't really care. You gotta watch this show! The plot is extremely interesting. Omni man is the MVP without a doubt and Homelander can learn a few things from this fellow.",
    "This show is just so touching all over. This show explores themes of loss, betrayal, grief and on top of all that the desire to keep going. This is the kind of show you'd expect to move you as this sometimes exceeds beyond entertainment.",
    "This series is indescribable. Deserves all the hype. I couldn't imagine getting hooked by a show so quick just by the famous last minute of Episode 1. This masterpiece will definitely go down as one of the best animated series I have ever watched.",
    "Invincible takes something like the DCAU and decides to use it as a launchpad for uber-violence and character-centric world-building. It's character-drama, action-adventure and universe-making stuff that proves it's truly special.",
    "I binged the first season and was so excited for season 2. Every episode I watched in season 2 I have FORCED myself to watch. The plot has been boring, felt like I was watching more of a tween drama like Riverdale.",
    "Season 1 was a solid 9/10 for me. Great character work, great stories and genuinely interesting. Then after waiting forever for season 2, it's like they sacked all the writers. What a bad turn.",
    "Invincible is my favorite adult animated series of all time. From the very first episode, it grabs you with jaw dropping twists, intense action, and emotional storytelling that only gets stronger as the series goes on.",
    "Spectacular voice acting, animation, and writing, the amount of effort and care put into this series is evident. Each character is so complex and well rounded. I implore everyone to give the show a watch.",
    "Invincible may resemble your typical animated superhero production on the surface, but its subversive and violent approach is exciting and refreshing. You are never given what you are used to.",
    "Such a great subversion of expectations. Almost all the characters are amazing and so emotional, which is highlighted by the amazing voice acting. This is possibly one of the best animated shows I've ever seen.",
    "Invincible is an adaptation from the Image comics that tells the coming of age story of Mark Grayson in a world full of super people. Despite the colourful cartoons and the juvenile setting, Invincible is pretty much an adult show.",
    "Heard about this series from a friend. This was a very gory and violent superhero series that created a huge mystery. As much as I loved the gore, I appreciated that the gore served a purpose. Highly recommend.",
]

add_show("Invincible", invincible_reviews)

✅ Invincible predicted as: Balanced
✅ Added 40 reviews for Invincible


In [82]:
naruto_reviews = [
    "Naruto made me fat. Everyday after school I went to the store and bought a bottle of soda and some snacks, and watched Naruto when i came home. Some months later I was borderline obese. 10/10 would recommend.",
    "I've seen a lot of anime throughout the years. I've seen what people call the 'good stuff', I've seen the 'bad stuff'. In time, I realized and accepted that not everyone is going to like something, or dislike it. When I first started Naruto, I saw the potential in it. From the first episode, they presented a distinct setting with distinct characters. Sure, archetypes were being followed (the boy dreaming of becoming big, the old grandfatherly figure, the teacher, the rival). The first episode showed emotion and depth to its main character and what he will struggle with, along with the dynamics of his interaction with other characters and their quirks. Having seen all of the series up to the current episodes, I can say at least for myself that at its best Naruto is a deeper anime than some give credit to, and can be quite addicting.",
    "Great moments, but lots of filler scattered throughout. You're probably thinking: 'well I could just skip the fillers by searching filler episode up.' That's fine but that's not the only issue with the show. It's that even in nonfiller episodes it feels like they are buying a lot of time. If the show was compressed to maybe 90/100 episodes then it would be amazing. The hype moments really are hype and the ninja stuff is cool.",
    "It's an amazing anime. If u r looking for a good anime to watch then watch naruto. I have watched 7 anime in my life and Naruto is still my favorite. It's an amazing story with the best character development I have ever seen. If u haven't already watched it then watch fast!",
    "God of anime. For me there is nothing above naruto. I just can't get my mind out of this anime. Once you entre the naruto world, there's no coming back. I love every bit of the show (leaving some fillers). If you didn't watched naruto you're missing paradise of earth. 10 rating is too less for naruto.",
    "Last year my roommate got me hooked on watching Naruto. I had never really been a big Anime fan. The story is about 3 ninjas as they progress from genin level onward to some rather high lvl battles and their hardships along the way. Each of these characters is well described with a rich background. And each character possesses unique abilities, none of which I have found boring or pointless so far. It is a truly excellent series. The different jutsus are exciting to watch and well thought out. The quality of the actual animation which can really be seen during the fight scenes is unbelievable. Lastly, the music is top notch.",
    "One Of The Best... This Anime Is Perfect Even For Grown-ups.",
    "Okay, let me start out by saying this show should be Rated TV-MA. When i first started watching this show, i was pretty damn sure it was gonna be another one piece or dragon ball Z knockoff but to my surprise this show has depth, drama, comedy, Horror and Suspense all rolled into one. The lead Villain really makes me mad, he's such a sick freak. Naruto is my number 2 favorite anime, with One Piece being my number 1 and Dragon ball Z being my 9th favorite.",
    "Naruto is...how to say? Magnificent, glorious and entertaining. Its almost flawless in the way it presents itself. The overall story is new and refreshing. The characters are inspirational and beautifully portrayed. The fight scenes are overall breathtaking and neat. The drama in here is very emotional and moving. The jokes are well executed but sometimes overdone. Pure perfection. 10/10",
    "When I first started watching this I had no idea what it was. At first it seemed a little silly and childish but I soon realized the detail that went into the stories, backstories, the world, the ninja way, chakra, their abilities is all fantastically explained in great detail. Once you get past the silly moments the show has some of the best writing I have ever seen. A top notch show. Top notch writers. I highly recommend Naruto and Shippuden. It's one of the best stories I have watched in an anime.",
    "Naruto seems to be one of the more popular anime nowadays. It has an interesting story and a good plot line. The action is exciting, the comedy is great, the characters are likable (except Sakura perhaps). But then again there's also the wonderful world of fillers. From about the 120th episodes to the end of the series there's nothing but filler nonsense. If they had just made the series about 100 episodes with less fillers, this would've been much better.",
    "Naruto: A Ramen-Fueled Ninja Odyssey.",
    "I would rate this anime a solid 8 if not for the tons of flashbacks. I mean enough already, we remember, no need to repeat the same flash back over and over. It's really frustrating.",
    "It's important to note that this review is in 2023. Over 20 years after the initial release, the hype has grown so much in this franchise and noted as an all-time great. It's been said that this show walked so others could run and it feels more like this show crawled while others have sprinted past. You can have a serious show about ninjas, espionage, assassination, etc. Just don't put cringe dialogue and overdone emotions with sprinkled in fart/pervy jokes. This show takes forever to tell a story and could have been great if condensed to ~3-4 seasons.",
    "A friend of mine introduced me to the Naruto series, and I have liked it from the first moment I saw Gaara of the Sand walk across the screen! I am now impressed beyond measure by its storyline, which manages to combine action, adventure, humor, science fiction/fantasy, and coming-of-age elements all in one. All the characters are very unique, no cookie-cutter fill-ins here!, and the bad guys are especially well developed. The only drawback I can think of are the excessively long battle scenes, which can take up several episodes.",
    "Very good anime with action and humor, but really what sets it apart are the characters. The characters are what I love most about it as we follow Naruto on his quest from being a goofball with seemingly little talent to someone who could defend his village if need be. The show is very entertaining. The humor works and the action is great. The show also has very good music as well.",
    "Naruto is the best anime that every human being should watch. This anime taught me many lessons rather than my life did.",
    "OK, so I've watched 103 episodes so far of this series, and I have to say it, Naruto is original. Naruto has a lot of things that DBZ was missing. First off, when characters die, THEY STAY DEAD! Second, the music in Naruto is 100X better than anything heard in DBZ. Third, character development: The characters develop feelings for each other, and you will find yourself developing feelings for the characters. Fourth, fights: There is nothing wrong with the length of the fights in Naruto. They're not too fast and they're not too unbearably long. I give this a 9/10.",
    "I am not a die hard anime fan and i found this series most enjoyable. The characters are well developed and ALL have a reason for doing what they do. After you watch, say, 10 episodes you will start to like them and feel bad when they get hurt, or die. The storyline progresses not too fast, and not too slow providing you with just enough time to swallow what you just heard. The only bad things are that the English voice acting plainly sucks and there are too many flashbacks.",
    "Naruto has to be the greatest anime series in the whole world, it has it all: comedy, drama and jaw dropping action fight scenes. The best fight has to be the fight between naruto and sasuke in the valley of the end. Overall the best anime ever, its worth watching. The characters are all SO original, the most original has to be gaara of the desert. After watching an episode you just have to go onto the next one. I got through the whole zabuza saga in one day, I just couldn't stop watching it.",
    "I can write pages and pages about all the things I liked, to be frank there is nothing I did not like. The characters and plots and mentors and story lines and villains and emotions all top notch. Watch this on my guarantee if you haven't already.",
    "Naruto, popular anime series, started off with a blast with variety of well characterized characters and intriguing story in ninja world and on top of all it is very addicting. Although Naruto began to lose its charm as more and more episodes flew by. The most disappointing part of Naruto are the fillers that are pretty bad, unnecessary and just ruining the show. I only hope that this will change in future.",
    "When someone says that something is overrated, they probably don't actually think that the thing they're talking about is overrated. The story is just your classic underdog story but with ninjas. Every episode keeps you on your toes and makes you want to see more, but the show has so many episodes that it gets tiring to binge. The characters are a mix bag, some are good, some are great but some are annoying. Naruto is not a bad anime, I would even say that it's good. But if I ever were to recommend an anime to someone who wants to get into anime, it wouldn't be my first choice.",
    "In all my experience of liking anime, I have not seen a show quite like this. Very creative, touching, and of course amazing. I love the storyline to it, the characters are very good and creative. The animation is very good and stunning. The fights are very well thought-out. The only thing that is bad about the Naruto series is that it drags on for a long long time. A definite 10 out of 10.",
    "The positives - some of the characters and their back story. Music is also very good, the dialogue and speech can be great too. But damn the negatives which can be the same for this and Shippuden - the fillers and the repeated scenes OVER and OVER again. I literally could skip 4 or 5 minutes off certain episodes which were not classed as fillers. I really don't get how people can rate this show a 9 or 10.",
    "Plot: Good. Characters: Excellent. Music: Great. Animation: Great. Dubbing: Good. Naruto Freakin' Rocks! The action, animation and everything are stunning! The characters are so likable. The animation rocks! Flaws: The anime uses filler arcs so the anime won't overlap the manga. 9/10. If you want hardcore action, crushes galore and lots of humor, Naruto is for you...Believe It!",
    "One of the greatest Animes.",
    "Naruto rules in my opinion! The characters are amazing, every detail, every one of it. Older anime watchers prefer other animes, but this one, along with Pokemon, is a great start for you newbie anime watchers. They have a great plot. It's still a great anime nevertheless.",
    "As someone who grew up with this first iteration of Naruto's journey, I realize how much it means to me. This is a series that helped me learn things and is a generational footnote for me. From Rock Lee fighting Gaara to Sasuke leaving, it's all handled thematically in a way that resonates with me still as an adult. I will never forget Naruto.",
    "Naruto is a pretty good show. At the start of the show it was strong, very few filler episodes and a good story. But nearing the end of season 6 we start to see the filler episodes add up, making the last 3 seasons all filler. Overall the show is good, not including the filler episodes though.",
    "Well many comparisons have been made between Naruto and Dragonball. After watching a few episodes I realized this show is much better than dragon ball. Not only does it have interesting characters and villains but it also has a plot that is ongoing. The main character is interesting - he makes a lot of mistakes. If you have the time check out this excellent show.",
    "In the Naruto series, you feel the actual existence of your life, why you were born, why your life is like this. Because in the Naruto series you know the value of a family, a true friend, and true love. My life is changed and I know the value of life and having a family around you is really important. I highly recommended Naruto Series, If you want to change your life and want to mature early.",
    "The whole series is just absolutely amazing. From the storytelling to the choreography in the fight scenes. It is 100% worth your time and a 10/10 must recommend! You can skip fillers if u want. The canon episodes are amazing. This show is the perfect underdog story. Not only is the main character good but the side characters are exceptional. So overall I would recommend this 11/10 times, and it's a great first anime as well!",
    "Naruto is a very good anime as far as anime goes, but it really is just a reason for you to watch Shippuden which is a much better show in every way. The characters are very likable though there are some problems - Naruto's intelligence seems to change for the sake of comedy. The combat in this show is satisfying when it happens, but sadly the combat is very sluggish. Some battles will last several episodes. Naruto is a good show, and it is enjoyable except for its crazy long side arcs.",
    "The anime which taught me the way to live. No matter what, this anime is one of the best in my life. If you're feeling alone then you should watch this anime - once you definitely get connected. The villains of the anime are broken and well written, the side characters are also well developed. I highly recommend it.",
    "This was the first anime I have seen and recently having finished it, I can say that it was an awesome first experience. Despite having a slow start, it does get much better with the 2nd season and the chunin exams has been my favorite arc in the series by far! Characters like Kakashi and Orochimaru are great. The main character, although quite annoying at first, is very likeable towards the end. Only complaint is there's a lot of filler but you can skip all that. Overall, 8/10 definitely recommend.",
    "It starts off pretty slow and gets slower. The hardest thing to sit through is the constant flashbacks, not flashbacks to the past but to what happened less than a minute ago. Each episode advances the story by a couple of minutes. The main character starts annoying and hovers in that area throughout, he is very hard to like. The fight scenes can last episodes and yet without the repetition would last mere minutes. For everyone else there is much better out there.",
    "This is an amazing story based on the Japanese manga. The characterizations are rich and the plot is excellently developed. Though it may be slightly predictable, the fights and music tracks are quite awesome. This is the true story of an underdog - naruto - recognized by no one and hated by everyone. How he battles with his sheer will and determination against insurmountable odds is poignantly portrayed in the series. It tells about friendship, loyalty, courage and unselfishness. I rate it 8.7/10.",
    "A Hilarious and Action-packed Anime!",
    "Naruto series summary - a brilliant series from start to finish.",
    "It's great and interesting: great beginning, childhoods, characters, themes and universe.",
    "This is one of the greatest anime's of all time. Naruto has a message of friendship, will and hardship. Naruto also has super funny moments, and it has amazing characters like Kakashi, Sasuke, Lee, Gaara and of course Naruto. It really shows that hard work pays off. Naruto is a legendary anime and u must watch it.",
]

add_show("Naruto", naruto_reviews)

✅ Naruto predicted as: Balanced
✅ Added 42 reviews for Naruto


In [83]:
one_piece_reviews = [
    "One Piece is not too different from other Anime series at first glance. It has hilarious humor and so much action it makes your head spin. But the thing that really gets you is when the story really starts to kick in. It has such heart and the characters are so real that you find yourself completely invested.",
    "I'm totally in love with One Piece. Years ago my friend told me that the top three best anime were Bleach, Naruto and One Piece. After watching all three I can say that One Piece is far superior. The world building, the characters, the emotional moments - nothing comes close.",
    "I've followed this series for going on 15 years and it never fails to amaze me. From the characters to the world building, the details are everything here. The story rewards long time viewers in ways that are genuinely shocking and emotional.",
    "One piece holds a Guinness world record for its Manga with 100 million copy sales across the world. The anime adaptation is a faithful and entertaining journey that has captivated audiences for over 20 years.",
    "This is pure greatness, a MUST watch for all Anime fans. Story is perfection. Comedy is breathtaking. Action is marvelous. Drama is absolutely amazing. The length is worth every episode.",
    "One Piece is an excellent show. It's exceedingly funny and I guarantee that you will laugh. The main characters are so weird and insane that you immediately love them. The adventures they go on are truly epic in scale.",
    "The English dub is horrible and all the scenes of the so called violence are cut out. Watch it subbed. The original Japanese version is a completely different and far superior experience.",
    "One piece is a special story with special characters and a fantastic adventure. The anime has painfully slow pacing at times but the story itself is one of the greatest ever told.",
    "One of my favorite mangas ever. The anime on the other hand has painfully slow pacing. It's probably more enjoyable if you haven't read the manga. The filler episodes are particularly frustrating.",
    "One Piece is the best selling manga of all time for a reason! Oda is a genius who created over a thousand characters and many of them are so memorable with amazing world building and brilliant art.",
    "One Piece has become a part of my life. It is the most beautiful thing mankind has ever created in terms of storytelling. The emotional payoffs after hundreds of episodes are unlike anything else in fiction.",
    "Is One Piece too long? Is it really worth it to watch? YES! YES! YES! One piece is not too long, and you should really watch it! The length is justified by the incredible story being told.",
    "Those were the words that started the greatest story of all time. One Piece has remained consistently brilliant across over 1000 chapters and episodes. The world building alone is unmatched in all of fiction.",
    "It's official, I'm fully caught up on One Piece! I started slightly over a year ago. The emotional moments hit harder and harder as you get deeper into the story. Nothing prepares you for the later arcs.",
    "This is simply the best anime TV show or comic out there. In my opinion it is also better than all non-anime TV shows. The depth of storytelling and character development is unmatched.",
    "It may have 1000+ episodes. Don't let that stop you. It's incredibly addictive. All the characters are well developed and Oda makes sure every character gets justified. Unlike any other long running series the story never loses its way.",
    "I started watching this anime with pretty high expectations after having already watched Death Note and Naruto. Now that I have finally caught up I feel like One Piece is the greatest story ever told in any medium.",
    "Pre time skip is great but post time skip is bad in the anime, just read the manga instead. One piece is an excellent series filled with great characters, funny comedy, great action scenes and great drama.",
    "The first time I heard of One Piece I assumed anime could not get better than what I had already seen. Boy was I wrong. One Piece transcends the medium entirely.",
    "One Piece has become something of a colossal cult phenomenon around the world. The series continues to evolve and surprise after all these years with Oda's intricate long form storytelling.",
    "I have watched a lot of anime, however One Piece is one of a kind. After watching all the episodes I'm finding it really difficult to enjoy any other anime. Nothing else compares.",
    "If you're considering getting into One Piece, read the manga instead. One Piece is an amazing series but the obsurd amount of filler and poor pacing in the anime drags it down significantly.",
    "I have been watching the show for a long time, and it just goes on and on. It has a format that repeats itself. The protagonist finds an island, goes to the island, understands the problem, defeats the antagonist and the crew moves on.",
    "One Piece is The Greatest Shonen Anime of all time. It features a grand adventure full of colourful and developed characters, each with their own unique goals and dreams. The narrative is extraordinary.",
    "At first I used to think it's too long, definitely not worth it. Let me just say it's one of the best shows I've ever seen easily in my top 3. The emotional depth and character development are extraordinary.",
    "There are two responses I get every time I try to get someone new to watch this show. I don't like the art style, or I've seen the dub - NO THANKS. Get past both of these and you will find one of the greatest stories ever told.",
    "One of the extraordinary shows of the world. If you think the show is only about pirates, then you are wrong. It is about friendship, dreams, sacrifice and the human spirit.",
    "You know you're watching great anime when the credits roll and you find yourself thinking this makes everything better. One Piece is the perfect recipe. It's just a lovely, fun, emotional adventure.",
    "Dropped the show after 20 years due to poor pacing. One Piece was one of my favorite anime of all time until around episode 500. The various arcs with their stories and characters were highly enjoyable but the pacing became unbearable.",
    "More than just Anime. Absolute Masterclass of fictional medium. An incredible story telling with twists and turns, a fleshed out fictional yet somehow real world, deep character motivations. One Piece is simply the best.",
    "One Piece is simply the best anime ever made. It has an incredible plot, carefully constructed characters, amazing action scenes, drama and comedy. Entertainment with so many layers that it rewards rewatching.",
    "One Piece is the base of anime. If you are an anime fan then you should watch One Piece. It defines the medium and everything that came after owes something to it.",
    "One Piece is epic, epic fun. Everything is big, loud, and exciting, but not without the finer details. The world of One Piece is truly a unique one and we get to discover just how amazing it is with Luffy.",
    "One piece is an anime which shows friendship, action, feelings and adventure. It is the best anime show I've watched and I don't regret the 5 months I spent watching it.",
    "I tried to watch it like 3 times but I couldn't get into it. It has a basic storyline and really predictable action. A lot of adventures, always happy endings. Not for everyone.",
]

add_show("One Piece", one_piece_reviews)

✅ One Piece predicted as: Balanced
✅ Added 35 reviews for One Piece


In [84]:
pokemon_reviews = [
    "A 4 something on IMDb? It was better than that. Pokemon was easily my first and most loved anime because it outclassed everything else when it first came out. The characters, the adventures, the world - it captured something magical.",
    "This show is great. It's on par with the first Pokemon show Indigo League. I enjoy the storyline, having Team Rocket back is great and Meowth is entertaining. Misty, Brock, and Ash star the show and the chemistry between them is perfect.",
    "Whenever I think of Pokemon my heart sinks, but in a good way! It reminds me of my younger life. This was easily my first and most loved anime because it outclassed everything else on TV at the time.",
    "Great character development, actions and adventures. Very engaging. One of the best seasons for me. Better improvement in animation compared to previous seasons.",
    "Pokemon is entertaining and fun. The XY and earlier episodes were the best. All of it is interesting and you get invested in the characters although it has quite a few problems like resetting the series every new region.",
    "For anyone who grew up watching Pokemon - is there really any of us who doesn't suddenly feel nostalgic at the sight of Pikachu? This new series brings back all those feelings while delivering something fresh.",
    "I highly recommend it to teenagers specifically 13-16 years old to watch this amazing series. It has more to offer than usual Pokemon battles. It has personal development and emotional storytelling.",
    "If you don't want to watch the entire anime but want to give it a try, this is the series. It truly is the best Pokemon series and a very good anime series. There are fleshed-out characters and genuine emotional stakes.",
    "The series is really just the best. The story, characters, and the goal is really exciting. It has a good balance and makes Ash look really cool. The animation is crisp and the battles are thrilling.",
    "It was a good show and it's surprisingly still running. Inevitably it got stale later, but the first few seasons were fresh and good.",
    "The concept of traveling to every region in one show has a LOT of potential but this series is pretty dull for the most part. The earlier seasons remain far superior.",
    "I'm a 37 year old dad and was never into Pokemon as a kid. What I enjoy most about the series is the one to one interaction of newer fans discovering the world for the first time alongside their parents.",
    "A Fresh and Exciting New Chapter in the Pokemon Journey! Change can be tough in long running franchises, especially when beloved characters are absent. But Pokemon XY proves that change can be a great thing.",
    "Starting from season 20 it seems like non-sense. And the idea of Ash not growing up after so many experiences is unbelievable.",
    "It's obviously declined in quality very slowly. The first 10 years of this show were great then I started watching after and the new ones make me cringe.",
    "One of the best series of Pokemon anime where we visited all the places and met all the gym leaders. Every fan of the Pokemon franchise should follow this anime.",
    "Ash gets bullied and they win. Ash loses a lot and is just nice. Why can't he be the one winning championships? And why don't his Pokemon never evolve! Despite all this the show is one of the best on the air.",
    "How do you even address a franchise as massive as Pokemon? The weirdest thing about this show is the nostalgia factor. Nearly everyone I know has a strong emotional connection to this series from their childhood.",
    "The new Pokemon series brings a new and exciting story with visually appealing art and graphics. However the producer's failure to develop the characters properly holds it back from being truly great.",
    "Don't get me wrong I love Pokemon and the anime but sometimes I wish it would end. The manga and games are better obviously but the ongoing anime really has stretched itself too thin.",
    "Was a brilliant show in the day, now not so much. Pokemon has always been a great franchise when it was released, and the anime I thought was amazing. But it has gone on far too long and lost its spark.",
    "The show and movies are fresh after the end of the millennium. Ash is a young trainer that are out to collect Pokemon. From Viridian City to the beaches and forests, the world of Pokemon is genuinely magical.",
    "Pokemon is one of the most iconic anime ever made. The early seasons especially are a masterclass in world building and character development that created a cultural phenomenon unlike anything else.",
    "I can't stand it at all. Especially the fact that Goh wants to catch every Pokemon, even the legendary ones. This is so annoying and completely removes any challenge or tension from the show.",
    "The original series of 151 Pokemon were by far the best. The nostalgia factor is enormous and nothing since has quite captured the magic of those early episodes with Ash, Misty and Brock.",
    "I still remember watching episodes from gen I through IV when I was younger on TV. Even now it holds a special place in my heart as one of the defining shows of my childhood.",
]

add_show("Pokemon", pokemon_reviews)

✅ Pokemon predicted as: Balanced
✅ Added 26 reviews for Pokemon


In [85]:
breaking_bad_reviews = [
    "It's ok I guess Re-Watched it 7 times and counting. I guess I liked it.",
    "Really Great I have never watched a show that is as consistently genuine and engaging as Breaking Bad. This is undoubtedly one of the greatest shows ever, and it consistently improves as it progresses. The Journeys of Walter White and Jesse Pinkman are unforgettable. These are some of the best-written characters to ever come from a pen-hitting paper. My praises for the acting and cinematography are unending. Some of the shots are intricate works of art, and I was rarely distracted by the acting. The performances are excellent to the extent that it feels improper to refer to them as performances. Overall, Breaking Bad consistently maintains a level of engagement and technical quality seen in only the best of movies, and in terms of tone, every intense moment is executed with excellence and always achieves the impact it reaches for. I feel like the show's plot in the early seasons lacks a certain level of complexity due to it not having a vast amount of plot threads, and the start is a bit slow-paced, but Breaking Bad is an absolute must-watch. If you have mixed feelings towards Season 1, trust me, it is only uphill from there. If there was ever a series you could call perfect, I think this might be it.",
    "99.1% pure One of the greatest shows ever, the pacing is excellent. The characters are well developed and entertaining. The show ties everything together very neatly. It's honestly a show that get's better each time you view it. It's cathartic to see Walter break bad and how the story unravels is the best way it could have. Very good storytelling, well done to Vince Gilligan.",
    "The Best I cannot stress enough how good this show is. I've watched a lot of TV in my life and this show still remains the best show I've ever seen.",
    "Damn near perfect! Breaking Bad is absolutely, without a doubt, one of the greatest tv shows ever created...it's damn near perfect! I know that people say that about a ton of different shows but Breaking Bad really is universally thought of as one of the greatest shows of all-time. All you have to do is go google \"greatest tv shows of all-time\" and I guarantee you that it's at or near the top of every list you'll find and there's a reason for that...because it is! Every season is just as brilliant as the others, which everyone knows is almost impossible for a show to do but one which Breaking Bad absolutely pulls off!! If you're one of the few people who still haven't seen this amazing show then do yourself a favor and go watch this immediately...you will not be disappointed!",
    "Those days ain't gonna come back.. When you finish the show you'll never be the same..I guarantee you",
    "Breaking Bad is the GOAT! For me it is the greatest TV series of all time. The show is better written than any other show I have ever seen. The entire storyline is a masterpiece and Walter White's classic character arc is the cherry on top of it. The story seems like it was well crafted from the very beginning, like every action has a thrilling and/or heartbreaking consequence. Breaking Bad is one of the few TV shows that delivered a satisfying conclusion. Iconic characters like Walter White, Jesse, Saul, Mike, Hank and Gus were well written.",
    "Best TV show ever made. I wanna delete my brain and watch it again like I never knew it.",
    "ABSOLUTE PERFECTION!!!! Direction- PERFECT!! Screenplay-PERFECT!! Writing-PERFECT!! Score-PERFECT!! Cinematography-PERFECT!! Casting-PERFECT!! Beginning-PERFECT!! Climax-PERFECT!! Acting-BEYOND PERFECT!! Overall-(as I said) ABSOLUTE PERFECTION.",
    "Absolutely Brilliant!! Breaking Bad is every bit as good as everyone says it is. It's easily one of my favorites shows of all-time and one of the best shows in the history of television. The writing and acting is what makes this show so special. I love this show so much that I've already watched it all the way through twice already. There's nothing more I can say about this incredible show that hasn't already been said so just go watch it. It's addicting once you start!",
    "A Masterpiece for sure. I usually don't comment on here but this show is the best I've seen in my life. The setting, character development and the actors... everything is pitch perfect. I'm on my fourth run through this show and I still love it. The dynamic between Walter and Jesse is awesome. I will watch this show once a year and I never get bored. This is the one and only you should watch right now.",
    "Among the best and most addictive shows there is 'Breaking Bad' is one of those rarities where every season has either been very positively received or near-universally acclaimed. Very few shows in recent memory had me so hooked from the very start that before the week was over the whole show had been watched. Visually, Breaking Bad is one of those shows that is both stylish and beautiful, with photography and editing that are cinematic quality. The writing is a fine example of how to have a lot of style but also a lot of substance. The dialogue throughout is thought-provoking and tense, while also having a darkly wicked sense of humour. Bryan Cranston is phenomenal as one of the most fascinating anti-heroes in either film or television.",
    "If you mix Scarface, Robin Hood and maybe Tyler Durden with enough meth - you'll get a mean cocktail called 'Heisenberg'. Believe the hype, it really is THAT good. From an artistic point of view - performances, writing, direction, camera, music - this show is every bit as good as The Wire and Generation Kill. For pure entertainment value, this is simply the best show I've ever seen. Every single one of the main characters has already reached the status of a screen icon.",
    "By far the greatest show I've ever watched Breaking Bad feels like a fever dream. A really REALLY good fever dream. The directing is absolutely perfect. The characters bounce off each other in a really well done way. I can feel the greed, lust, money hunger in each of these characters. There are no true heroes in this story.",
    "Once in a life time series I have never watched any series which depicts emotions so exactly through camera angles, dialogues, screenplay and perfect story line. There are scenes which will stay with you forever after this series and it will change the way you watch TV. Perfection!",
    "Since GOT is over, this is officially the Greatest show ever made. Writers need to take notes and be more like Breaking Bad. They had the ending planned out from the beginning. They stuck with their plan and created the greatest show we may ever see.",
    "in a category all on its own. Nothing compares. Nothing comes close. I may currently be on my 20th dose of this show and it's still as addictive and beautiful as the first time.",
    "The 10/10 show. If there is any show that deserves a 10/10, it's Breaking Bad. Simply the greatest.",
    "Cinematic Masterpiece This show was easily a cinematic masterpiece. From the cinematography to the songs used it's the best. It starts slow but if you give it a chance it speeds up and gets more action-packed. Almost every episode is great and it is easily one of the best shows of all time beside Better Call Saul.",
    "Stunning Comedy-Drama I probably haven't been hooked to a TV show like I am to Breaking Bad before. This beautiful piece of art is incredibly well written and directed, furthermore the actors are doing a tremendous job! Because this way you can entirely fall in love with the show, the characters and every tiny detail of the story and the best part of it, it is unbelievably addictive and makes you starve for more week after week!",
    "Addicted: Meth or Math. I stayed in bed for almost 10 days sick and then it happened. I saw the first episode and I was immediately and I mean immediately, hooked. I saw the entire series in 9 days. Bryan Cranston made it all real. His performance, the creation of Walter White will be studied in acting classes of the future. I developed a visceral need to see Jesse find a way out.",
    "Mind Blowing BREAKING BAD Season One explodes like a sucker punch to the gut, and is nothing short of mind-blowing. The pilot for this series is a definite Must See. Bryon Cranston's idiosyncratic performance is a joy to behold. He embodies a man who is against a rock and an even harder place.",
    "If I could rate it 20/10, I would. The writing is near perfect, the actors and characters are all amazing, the over-arching story just gets only better and better. This show is just a bullet train that goes and refuses to stop. It pulls no punches, everything that happens has all the impact. There isn't a single bad episode. This show is perfect.",
    "Fantastic show that killed my spirit. The first few seasons were incredible. But I found that Breaking Bad exhausted me. Five seasons of disgusting corruption and deplorable human behaviour took its toll. If it had been shorter I would have enjoyed it more. Does a show that is fantastic but sucks the soul and heart out of you make for as good TV as everyone believes?",
    "The Most Over-Rated Show I Have Ever Seen. To be honest, this show is OK, but it is nowhere near worth the hype. I watched the first three seasons and was not compelled to watch more. The depth of this show is ankle deep. The characters are not quite believable, and sometimes downright grating.",
    "Truly Amazing, Wonderful, 99.1 Pure Perfect. This series is unique. Even if you're going to watch to find the flaws, I bet you can't find real major flaws. Vince Gilligan is a true genius. It is the best crime drama TV series possible. And it ended with the best way possible.",
    "ONE OF THE BEST TV SHOWS EVER. Walter White is objectively one of the most charismatic and well-researched characters in the history of cinema. Gustavo Fring is one of the most memorable antagonists. This series is one of the few examples where the further you go the better it gets, and not the other way around.",
    "Incredible Show Finally something that punched me in the face like this show did. This is the best television series I've ever seen. This show is beautifully written with plot twists coming in refreshing ways, at just the right times. The acting is brilliant, and the themes are presented in thought provoking ways that leave me pondering philosophical ideas long after the show is over.",
    "Brilliant Character Work In Pitch Black Comedy-Drama Bryan Cranston has that rare gift of generating sympathy and manic energy at the same time. Breaking Bad is a new kind of monster. It touches on the very same themes of living realistically as a middle class in the United States. The Pilot was about as perfect a Pilot as I've ever seen.",
    "Masterpiece I am utterly shocked by how good this show was. It really dives deep into Walter White's character and his development throughout the show. The cinematography the shots and camera angles placed by Vince Gilligan are gorgeous and beautiful. The acting for this show is absolutely amazing and gut wrenching.",
    "Slightly overrated but definitely good. While it isn't quite the holy grail it's made out to be, Breaking Bad is a solid series. It is well worth watching. It absolutely IS a good series. Just allow for a steady stream of implausible moments throughout. Enjoy the good, there is plenty.",
    "The best series I have seen in my whole life, even better than The Wire.",
    "As close as one can come to view perfection. Everything from the subtle details, to the unmatched writing and the perfect cast, to the very beautifully shot scenes. It may very well be the best TV show out there. The story is well grounded, believable and feels very real. Breaking Bad is a timeless piece.",
    "Breaking Bad truly one of the best TV shows in history, masterfully blends suspense and drama. With Vince Gilligan's creative genius, you find yourself breathless as you watch an ordinary chemistry teacher unexpectedly descend into a life of crime. Bryan Cranston's unforgettable performance and the show's compelling storytelling make Breaking Bad not just a series, but a masterpiece.",
    "Please stay on the air. Absolutely one of the most ground breaking shows on the screen. Definitely not for everyone but finally something that feels real. Nothing is glamorized, and it gives an insight to the darker side of many streets - how the denial and desperation that can occur to the common working citizen, the pillar of society that suddenly changes his moral standing in a society driven by the almighty dollar.",
    "Arguably the greatest dramatic series ever? Over the course of 62 episodes, Vince Gilligan and his team crafted a piece of neo-noir-western American tragedy that almost never lost its footing. There's conflict, conflict, conflict going on in this show, always - from the familial to the genre-leaning to the dark comedy. Everything just fits. Breaking Bad is perfection.",
    "Best Series For A Long Time. I couldn't stop so we watched it over a few days. Brilliant acting. Fantastic assortment of comedy, frustration, infuriatingly suspenseful moments and times of incredible sadness. The storyline just gets better and better. The change in Walt will leave you shocked. This series is well worth the time to watch.",
    "Too dark for my taste. The plot line is great and the acting is good but for me it was just so dark and depressing I can't call it a favorite.",
    "One of the best, if not THE best show of all time. Being able to finish the entire show in a week and a half is a testament to how tightly written and engaging it is. Some of the most fascinating character studies you'll ever see. The storytelling is impeccable and the dialogue is dynamic and engaging. At times very intense, other times funny. The acting is some of the best you'll ever see. An iconic and unforgettable show.",
    "Without a doubt the best TV show ever made. Everything about it was just phenomenal. The acting, writing, cinematography, all of it. It is truly a masterpiece. It really affected me so much I was depressed and feeling empty for 3 weeks after I finished watching it. It has the perfect ending and the seasons get better as they go on which is rare.",
    "Hands Down the Greatest Television Drama of All-Time. Then came Breaking Bad. I couldn't stop watching and felt like I was addicted to it. Just so intriguing, beautiful, real, amazing, and shocking. Then came late season 4 and season 5, and I was speechless. It was hands down some of the greatest filmmaking I had ever seen. All I can say is this is the greatest show of all time.",
    "The Greatest TV Show of All Time. I wish I'd never seen it so I could see it for the first time again. Even at face value, this plot is compelling. Yet the deeper level may be what takes Breaking Bad to the highest pinnacle of art. The series is marked by unparalleled intricacy in its characters. A frank and mature exploration on the meaning of love, life, family, loyalty, and so much more. Not to be missed.",
    "Hello, Mr. Cranston - Meet Ms. Emmy. Bryan Cranston puts on one of the most chilling performances in TV history as a terminally ill teacher who turns to manufacturing methamphetamine. The show is about moral ambiguity and how actions have consequences. Outstanding show. It deserved every award that it won.",
    "Too much hype. I liked the series, although it is much more flawed than I expected. Excellent cast and chemistry between the lead actors and a refreshing take on the drug crime genre. But too many lengthy parts, too many clichés, too much repetition, and weak character development.",
    "Breaking Bad is an absolute masterpiece, each season is better than the last. This series includes the best episode of all television and the greatest season of all television. Overall this series is an absolute masterpiece and it ends in the best way possible.",
    "THIS is a cautionary tale!!! Everything at every moment could have been decided better if truth had been told! Good acting when you feel like strangling them! Don't smoke. Tell the truth. THINK before you act.",
    "Do.Not.Watch.This. This is perhaps the single greatest series ever made. Problem is, everything else pales into insignificance before it; nothing will ever compare to it. It has singlehandedly ruined my past, present and future series experiences.",
    "Dissecting Walter White's Character as Mr. White and Heisenberg. Walter White has been central among the characters depicted in Breaking Bad. Vince Gilligan devised the perfect tool to show the Jekyll and Hyde character of Walter White, an ordinary high school teacher who becomes extraordinary. The weighted yet intriguing relationships of Mr. White with different individuals adds spice as viewers see how his dual personality complicates every social situation. As a whole the serial is a family drama with an interesting mix of crime and violence affecting social virtues of a man and his family.",
    "An Addictive Everlasting Ride. A cult-classic, and for a reason. This show delivers on absolutely every facet. The main character manages to take you on a transformative 5-season ride, allowing you to both love and loathe what Walter White ultimately becomes. The dialogue is grounded yet packed with detail and foreshadowing. The chemical symbolism throughout the show thematically enhances the work to greater depth.",
    "Peak Television. This show is without a doubt the greatest show ever created. The tension, the drama, the writing and even the humour is incredible. Walter White is one of the greatest characters ever written. The acting is unbelievable. Bryan Cranston and Aaron Paul's performances are breathtaking.",
    "Everything You Hear Is True. Often hailed as one of the greatest TV shows ever made, Breaking Bad lives up to the hype. You will be hard pressed to watch a show that is as well shot, directed, acted, and written. Every minute of this series is masterfully crafted. Instantly becoming a classic must-watch show.",
    "An apology. Looking through Netflix I came across this series again so thought I'd give it another go. Over the last three weeks I have binge watched the entire series. The writing is superb, not to mention the acting from Bryan Cranston and Aaron Paul. By the end of the last episode I felt my heart was broken. It's been an emotional rollercoaster of a ride but I enjoyed every minute of it.",
    "The most cinematic television show of all time. Walter White is the most complex and best-developed character in the history of world television. There are dozens and dozens of memorable scenes in Breaking Bad. The series holds up great over five seasons. Breaking Bad remains one of the best series of all time or perhaps the best.",
    "Never been so moved by television. This is arguably one of the best things created by the human race. Without a doubt the best series of all time. Pure perfection. I gave it a chance, and it won my heart.",
    "Was the best then, still the best now. I finished watching Breaking Bad 8 years ago and was blown away by how brilliant it was. Original story, great character building, comedy, drama, edge of your seat moments. After all these years I've started rewatching on Netflix and I'm every bit as obsessed with it now as I was back then. It is absolutely unrivalled television.",
    "Everything you already heard about this show is probably an understatement. Breaking Bad is possibly the best TV drama ever made. There isn't a wrong step, bad episode or miscast character throughout the whole series run. The show is the proverbial roller coaster, taking you to the low points and peaks right along with your favorite characters.",
    "A masterpiece that will last forever. I've never seen a show that made me feel the same emotions as this show does. Amazing show I'd recommend to anyone who wants to see a timeless classic.",
    "A good show but not an iconic show. Breaking Bad is a good show that is worthy of praise however it is not worthy of iconic status. While it started quite poorly and dragged on in its initial seasons the show certainly improved in the final seasons. The series finale has to be the best series finale I have ever seen. Overall a decent show but it definitely saved its best and very memorable acts for the end.",
    "Vastly overrated. I finished the very last episode today and felt a sort of disappointment. At no point was I invested in anything going on. Never did the show manage to lure me in or make me care about anyone. Breaking bad is simply too pointless, boring and devoid of any sort of charm to be considered a good show.",
    "Top TV Shows of All Time. This show is epic in its entirety. The writing, characters, plot, cinematography and the incredible transformation push this show into a league of its own. Walter White turns into a genius mad-man who you just love to hate. Vince Gilligan knew how to build a story without making you feel underwhelmed.",
    "The best action series of all time. It's really good, it doesn't have a boring chapter and the drama is amazing. Great performances, well-done scripts, amazing direction and well-done characters. The writer of this series is a genius!!",
    "Breaking Bad is a Once In a Lifetime Experience. I could not get out of it, I did not come out of my room for 3 days and completed 5 seasons. It intensifies season by season. There are no words to describe how great this show is so stop reading this review and start watching it.",
    "chemistry in a cynical world. Bryan Cranston puts on one of the most chilling performances in TV history as a terminally ill teacher who turns to manufacturing crystallized methamphetamine to support his family. The show is about moral ambiguity and how actions have consequences. Outstanding show. It deserved every award that it won.",
    "99.44% Brilliance. BB has the best writing I have ever had the pleasure of reading. The acting is top notch by Cranston. The dialogue is absolutely brilliant for the most part. The story is brutal and comedic at the same time.",
    "A classic mix of writing, acting and filmmaking. Breaking Bad is one of the best serialised television shows and ranks amongst my favourites like The Wire, The Sopranos and Deadwood. The writing consistently impresses throughout the seasons. What sets Breaking Bad apart is the visual storytelling. Several episodes have distinctive cold openings or flash-forwards that we spend entire seasons intrigued about.",
    "Extraordinary piece of TV. Breaking Bad is an extraordinary piece of television that transcends the boundaries of its medium. From the very first episode, the show grabs hold of the viewer's attention and refuses to let go. The writing is nothing short of brilliant. Each episode is meticulously crafted with tension, dark humor, and shocking twists. In conclusion, Breaking Bad is more than just a television show; it's a masterpiece that leaves a lasting impact.",
    "Say My Name! - Heisenberg. Hats off to Vince Gilligan. What a series he built - the direction, the cinematography, the plot, the suspense. The scene where Hank died, he is my favorite character and I literally cried. The series backbone is its strong villains and only one word portrays Walt - his love for family. Once in a lifetime experience for this thriller.",
    "Don't read reviews, just start watching! Breaking Bad is the greatest show ever! It changes you. Every scene, every line is crafted with purpose. The tension is so thick that it reaches out and grabs you by the throat. Bryan Cranston's portrayal of Walter White is nothing short of a masterclass in acting. Aaron Paul as Jesse Pinkman will break your heart. Their chemistry is electrifying. This is more than TV; it's an experience.",
    "Best - No review here! I'm just here to say that go for it! Literally just go for it!! But just keep in mind don't hurry for thrill in first 2 seasons. There's more after that. After you have successfully completed 2 seasons with patience, you are ready to jump into thrill!",
    "Caveat. The caveat here is that I only made it through 1 season and a couple episodes of season 2 then had to quit. I really tried to like this show but my god is it boring. I've heard the show gets really good in the later seasons but you shouldn't have to watch 10-15 hours to get to the good stuff.",
    "Overrated crime drama. I seriously don't understand why this gets 9.5 stars. The show is above average at best. Sure, the acting is solid, the story has plot twists and Esposito is great again but 9.5? I don't get it.",
]

add_show("Breaking Bad", breaking_bad_reviews)

✅ Breaking Bad predicted as: Balanced
✅ Added 71 reviews for Breaking Bad


In [86]:
jujutsu_kaisen_reviews = [
    "Surpassed the manga I love it when an anime goes out of its way to surpass the manga in terms of quality, making it the definitive way to experience the story. Mangas are such a special form of art that achieving this is difficult, yet Jujutsu nailed it perfectly. I adore the manga already and this anime totally lived up to and surpassed the hype, just amazing.",
    "blew my freaking mind Definitely going to become a hit, like demon slayer or something. The first and second episodes were phenomenal. Literally so good that this anime compelled me to create my first IMDB review.",
    "Breathtaking anime! What an amazing anime Jujutsu Kaisen is. Studio Mappa with another magnificent job. The animation is awesome, especially the fights look breathtaking. So many great, loveable and fun characters with a very interesting story.",
    "Exciting As HELL It's just so exciting. You just need to watch episode 1 to get pumped up over this. Everything is nearly perfect; the plot, characters, action scenes, EVERYTHING. It's not an anime that will make you regret it.",
    "A SHONEN FULFILLING EVERY DEPARTMENT. Great Characters, one of the greatest OP and Ending themes. Amazing plot with great animation. Great music with great action sequences. Gojou Satoru. That's it. Overall, worth watching every single episode.",
    "An absolute blast The animation, the fight scenes, the humor, both opening and closing songs, everything about it is just so so good. It had me hooked from the first episode then sold me completely by the 2nd. Highly recommend!",
    "Shounen done right Characters, including the MC, are well written and occasional drop of motivation and backstory gives them decent depth. Power system has a lot of imagination in it and the tension that any character could die is good for the story telling. MAPPA have made a great job putting the story into motion.",
    "My Jujutsu Kaisen Review! Jujutsu Kaisen is an absolutely thrilling show with some of the best action sequences I've ever seen in animation, plus some really cool and badass characters. Season 1 is amazing but Season 2 is even better, and the series is overall incredibly entertaining. Just like Attack On Titan I believe this is another anime series non-anime fans could enjoy!",
    "Its not that good Honestly I cant rate this anime a 1 because the art and the fight scenes can be quite cool. But the plot and character development was VERY lackluster. I heard so many good things about this anime but every time I turned it on I was just disappointed.",
    "A Good Shonen Take-off Jujutsu Kaisen shows potential and good promise from its initial episodes. Good execution of the story has kept me going through the episodes. Along with instances of amazing action-animation sequences and decent characters, the first opening and ending themes have a nice taste of funk.",
    "This is so overrated This show is so overrated. There is basically no plot and things move really fast. Characters come out of impossible situations and appear to have unlimited abilities. The only things good are the animation and action scenes.",
    "The epitome of mediocrity Have you ever seen an anime where everything about it seems just fine but you very quickly realize that what you're watching has absolutely no soul whatsoever? All characters are cliche archetypes you've seen a thousand times before. There is a main character whose only quality is being strong for no reason. They fight. There is a mentor who is cool. There is a magic system which makes no sense.",
    "Easily one of the best anime of 2020 if not the best. There are some anime which are good too like Haikyuu and AOT but in shonen this started very well. Worth watching.",
    "If it continues like this, in my top 3 list. On episode 9 so far I have nothing to criticise. The protagonist is great, and all other characters are extremely likeable and interesting. The story so far has been amazing and is well-written. The animation is top-notch, typical MAPPA! Give it a go, you won't be disappointed!",
    "Highly Recommended!!! For everyone!! This is the best anime I've ever watched. The animation is flawless and the plot is quite interesting. It just keeps getting better. I binged all the episodes in 1 day. I would recommend this anime to almost anyone who's interested in anime as it includes a little bit of everything.",
    "Jujutsu Kaisen review A fun and entertaining show. The animation is gorgeous. The fighting is top notch. The cast of characters has so many standouts. Overall it's a really great show. Coming back from the second season - everything is ramped up and it was perfect. The fights and the fights and the other fights, this is top tier anime right here.",
    "Overrated copy of Naruto.",
    "Overrated Only thing I can say is overrated and boring! Plot is mediocre, animation good, fighting scenes okayish, some characters are boring some are good but not memorable.",
    "Overrated I will be honest about this. I tried my best to like this anime but I was bored to death. I don't know how it's rated 8.7. When you watch an anime it should connect you with feelings, excitement and emotions but none of that came. The characters were likable, that's all.",
    "On par with demon slayer This show is just as good as Vinland Saga and Demon Slayer. Tons of action, comedy and horror.",
    "Really strong action with a gripping plot! The start of the show is really good, it picks up straight away. The characters are really good and quite likeable, each with their own backstories. The action is fast, fluid and filled with unique spins and ideas. The wacky cast of curses allows an even wider range of moves. To conclude the show is excellent and does deserve the praise it is getting.",
    "A Great Contender for Shonen's BIG 3 Jujutsu Kaisen is the year's best anime. The screenplay keeps you engaged in each and every episode. Pacing, execution and direction is pitch perfect. A dark anime with the theme of curses that also makes you laugh and get intense at the same time.",
    "Wasted Potential I assume most high ratings come from people who watched the show for great animation and fighting scenes, because other than that it's quite mediocre. The plot is very unfocused and most plot points are irrelevant two episodes after they are introduced. There is literally no character development besides getting stronger.",
    "Mediocre The first 6 episodes were top notch but as the story developed it slowly got boring. There were like 4 or 5 episodes dedicated to the fight between the two schools that felt overwhelming and unnecessarily long. Lots and lots of talking, fighting, stop to talk, fighting, stop to talk.",
    "Not bad but nothing new either Season one is a solid 7. Season two is probably a 6. It's an ok anime but repetitive and brings nothing unique or new to the screen. You have another protagonist with an evil creature living inside him in yet another coming of age story set in some sort of training school. No originality going on here.",
    "PEAK SHONEN.",
    "its good but not great Good show to watch but it's nothing new to the anime genre.",
    "Iconic Out of all the new gen animes this is by far the best I've ever seen. Incredible animation, writing, and characters. Season 2 has me mind blown. The Shibuya Incident arc surpassed my expectations. Please give this show a try! If season one doesn't convince you, season 2 will.",
    "Not a masterpiece...YET Its an all-rounded show with action, drama, comedy and mystery with good character development. Nice music and excellent animation. The show is well directed and the fight scenes are top tier. Despite all that I really couldn't relate to the main characters of the show.",
    "All style, no substance. Honestly I think it's a pretty childish story disguising itself to be more than it actually is with more mature themes. If you like mindless fighting you will enjoy this show. There just isn't anything of substance in terms of meaningful narrative, dialogue or relationships. It's all flash and style.",
    "Excellent! This series captivated me from the first episode onward. The second season is more fast paced and action oriented than the first season. I definitely recommend this for anyone who looks for a good anime to watch.",
    "One of the BEST Anime Series Ever! This anime is a rare gem. The characters are charming and delightful! The storyline keeps you engaged from start to finish. Gojo is a fun, reliable and charismatic character. This series has the best ensemble of characters compared to other anime.",
    "Not an anime fan, but wow I don't ever watch animes but I accidentally stumbled upon this one and it was the best mistake I could've made. The plot is gripping, the characters are lovable and humorous and the attention to detail is incredible. This show will always have you on the edge of your seat.",
    "The best shonen of the generation JJK prevails in both art style and animations. Fights here are very intense but also have strategy and tactic implemented into them. The plot has some great unexpected moments. Events on the screen are always entertaining, I wasn't bored even for a second across 16 episodes of season 2.",
    "Obsessed!! Violence, action full on and comedy on point!! I love all the characters! The animations were on point and Gojo sensei stole my heart!! Fighting scenes were so damn good!! I binged it so hard. Every genre lover is gonna love this anime!!!!",
    "A Shonen that stands out above all demographics A shonen where main characters die? Where Itadori has a development that no one envies because it's based on deaths. That has epic fights both in story and animation. Where the lore is expanded and it blows your mind. This anime and its movie are gems that you have to enjoy.",
    "GOOD JOB MAPPA The animation quality in Jujutsu Kaisen is exceptional. The fight sequences are fluid and intense, with dynamic choreography that highlights the series' unique blend of supernatural abilities and martial arts. The season balances its darker themes with moments of humor and humanity, maintaining an engaging pace throughout. The underlying themes of death, fear, and personal growth are explored thoughtfully.",
    "If chaos was an anime The characters are indeed interesting, the struggle of the main character Yuji, Gojo being a really cool guy, Todo having a great dynamic with Yuji. The humor aspect is also good. However the second season intensifies all the issues - the world building is messy, power levels feel random, and some arcs go on far too long. Visually it's peak, no doubt, but the narrative remains chaotic.",
]

add_show("Jujutsu Kaisen", jujutsu_kaisen_reviews)

✅ Jujutsu Kaisen predicted as: Balanced
✅ Added 38 reviews for Jujutsu Kaisen


In [87]:
peaky_blinders_reviews = [
    "One of the finest shows to have hit TV. Peaky Blinders is one of those shows that grabs you and doesn't let go. The combination of gritty drama, incredible performances, and stunning cinematography makes it unforgettable. Cillian Murphy is phenomenal as Tommy Shelby - intense, calculated, and completely magnetic. The period setting is brought to life vividly. The tense confrontations, the twists, the quiet moments of reflection, this show nails it all.",
    "Peaky Blinders is without a doubt one of my favorite tv shows of all-time. The acting and writing is what makes this show so special but the attention to detail they put into every aspect of this show is amazing. This series will hook you from the very first episode and hold on throughout the entire series. I can not recommend this show enough.",
    "After the show, you cannot stop thinking about Thomas Shelby. I've never seen any movie characters that stay in your head like that.",
    "Peaky Blinders is definitely one of the best series I've ever watched. From the very first episode, it grabs you with its unique atmosphere and distinctive style. Every episode is a trip back to 1920s England, packed with intrigue, drama, and action. The characters are simply fantastic. Every actor nailed their role perfectly. When the series ended I was left with a mix of satisfaction and longing. Honestly, I wanted more.",
    "Peaky Blinders is not only one of the most underrated TV shows of all-time but it's also one of the greatest shows of all-time. It's one of the rare shows where every season is just as great as the others. If you haven't seen this amazing show yet then do yourself a favor and go watch it immediately.",
    "Peaky Blinders is a gripping crime drama that transports viewers to post-World War I Birmingham, where the Shelby family builds a ruthless empire. Cillian Murphy delivers an outstanding performance as Tommy Shelby, portraying him as both a fierce leader and a haunted soul. The plot is tightly woven, full of tension, betrayals, and power struggles that keep viewers on edge.",
    "Peaky Blinders is without a doubt one of the best TV series I've ever watched. It's a cinematic masterpiece that seamlessly blends gripping storytelling with impeccable performances. Cillian Murphy's portrayal of Tommy Shelby is nothing short of mesmerizing. The cinematography is breathtaking, capturing the gritty beauty of post-WWI Birmingham. From its razor-sharp dialogues to its intricate plotlines, everything is executed with precision.",
    "Peaky Blinders is a captivating crime drama that immerses you in the gritty underworld of post-war Birmingham. Cillian Murphy delivers a tour-de-force performance as Thomas Shelby. The series masterfully blends historical drama with a modern sensibility. The plotlines are intricately woven, exploring themes of family loyalty, ambition, and the struggle for power. A must-watch for fans of crime dramas and historical fiction.",
    "Superb show for me, the best tv show ever created. The characters, the plot, the storytelling and of course Thomas Shelby the GOAT of Television. Every season every episode is great. I watched the show two years ago and now I am rewatching and it's never boring.",
    "Peaky Blinders is the rare diamond of a TV show that has great writing, great casting, great cinematography and a brilliant use of music. For the first 3 seasons, that is. Already at season 4 it begins to go down. However season 5 loses everything that made this show great. Season 6 is simply unwatchable. Dark and gloomy, zero style, bad weak dialogue. The remaining actors are still brilliant but what can even Cillian Murphy do with such weak material?",
    "Peaky Blinders earns a well-deserved 10/10 for its masterful blend of storytelling, character development, historical context, and production quality. Cillian Murphy's portrayal of Tommy Shelby is nothing short of mesmerizing. The writing is razor-sharp with dialogues that are both thought-provoking and laced with a dark poetic edge. The historical setting adds another layer of depth and authenticity.",
    "I wanted to like this show given how many people praise it, but having seen series as good as Breaking Bad, Chernobyl and The Sopranos, this show is not among those actually great shows. Characters are solid and the acting is mostly good but some characters are very inconsistent. Everything is a little too dreamy plotwise. It's good enough but not quite among the best.",
    "Season 5 just seems to have a complete lack of direction and content. All the characters seem to have lost their substance. Everyone is so overdramatic doing stupid things that make no sense. Almost every scene is so dragged on to a point they become boring. The modern music is so overused it's difficult to stay immersed. It all feels so diluted and cheesy.",
    "Peaky Blinders is a fascinating journey through time that takes viewers into the criminal world of Birmingham in the 1920s. This British masterpiece is not just a story about gangsters - it is a deep dive into the lives of people who survived the horrors of the First World War trying to find their place in the world. It has everything: drama, crime, love, betrayal and family.",
    "Peaky Blinders is all style and no substance. Endless slow-motion shots to dramatic music of characters walking, smoking cigarettes, or squinting into the distance. The priority is clearly to make everyone look cool. The characters are props - lifeless mannequins. The actors are all good but they have nothing to do but pose.",
    "Peaky Blinders is one of the most unique British dramas ever made. There is nothing glamorous about being a gangster in this world. The attention to detail, the costumes, the sets have all been created to the highest standard. The quality of acting is undeniably as good as you'll find in any prime-time British drama.",
    "My husband and I can never agree on a show to watch together, and this was a win from both of us. The show as a whole was FANTASTIC. Cillian Murphy absolutely killed this role and very quickly became one of my favorite actors with his diversity.",
    "Superb drama created and written by Steven Knight. Gritty, realistic, intriguing and highly entertaining. Some very clever storylines and plot developments. Great work by the main performers. Great soundtrack too. Though set in the early 20th century, the music is mostly rock and it works. The show went out on a high while it was still quality viewing.",
    "Incredibly well made show top to bottom. Every character has their own complex personalities and layers. Thomas Shelby has to be one of if not the best protagonist I have ever seen for a TV show. Cillian Murphy truly puts on a performance of a lifetime. The cinematography and set pieces are by far the best I have ever seen in any show. Peaky Blinders should be considered a top 5 show of all time.",
    "Peaky Blinders is a tale that is extremely loosely based on an actual English gang. It's a story about anti-heroes who you'd despise in real life but who also generate feelings of sympathy. Such a well made show, very engaging, and there's just something so special about it that makes it irresistible. Cillian Murphy just kills it.",
    "Peaky Blinders is an exemplary series where the writing, directing, production design, and acting shine brilliantly. This show stands out as the best ever made primarily due to Tommy Shelby's character. He embodies every man's dream: fearless, fair, smart. The series captivates with its sharp dialogue, intense scenes, and a rich immersive setting.",
    "This series has everything a successful series needs. Great writing, interesting characters, intelligent dialogue and AMAZING direction and cinematography. The actors are excellent with some of the best performances you'll ever see on TV. Peaky Blinders is one of the most unique productions out there. It's one of those shows you will watch and never forget. This is not the classic gangster show you may have in mind. This is a poetic masterpiece.",
    "This Epic Show will steal your hearing, eyes and all your senses. The best TV show you will ever watch and probably the best you ever will. The acting is in its finest, the soundtrack was incredible. The plot always twists and twists again and holds mystery. Every character in the show acted very well and played an important role. That's what makes it a Masterpiece.",
    "Amazing to see some calling this a masterpiece. The acting is great but it's essentially a higher than average budget soap. Certainly watchable but could have been brilliant. A series that boasted talent most feature films would envy and were let down by the BBC writers.",
    "Early on I loved this show. Eventually it is like watching the same episode over and over again. There is no depth of characters like we see in shows like Boardwalk Empire. The slow motion shots should go out the window at this point - it's just getting plain ridiculous.",
    "Peaky Blinders is nothing short of a masterpiece. The show's storytelling is both intricate and engaging, with each season building on the last. The period details are meticulously crafted creating a vivid and immersive experience. Cillian Murphy's portrayal of Thomas Shelby is one of television's most memorable. The modern soundtrack featuring a blend of contemporary and classic music adds an unexpected but highly effective edge.",
    "Peaky Blinders is a brilliant exploration of post-WWI Birmingham's criminal underworld. Cillian Murphy's portrayal of Tommy Shelby is both intense and captivating, driving the show's complex narrative. Every episode is a masterclass in tension and character development, making Peaky Blinders a must-watch.",
    "Peaky Blinders is nothing short of a television masterpiece. From the very first episode, this series grabs you by the collar and drags you deep into the gritty world of post-World War I Birmingham. The writing is razor-sharp, weaving together themes of power, loyalty, and family with historical context. Visually it is a feast for the eyes with stunning cinematography and meticulous costume design.",
    "Cillian Murphy's portrayal of Tommy Shelby is worth watching for alone. Simply phenomenal, a masterpiece.",
    "Peaky Blinders is a riveting British crime drama that immerses viewers in the gritty world of post-World War I Birmingham. Cillian Murphy's performance is both mesmerizing and multifaceted, bringing depth to a character that is equal parts ruthless and vulnerable. The show's writing is sharp and sophisticated, weaving intricate plots with rich character development.",
    "I very much enjoyed the first few series of this. An interesting and entertaining plot, well produced and superbly directed. Sadly, this is a series which is rapidly going downhill. Due to its popularity they are clearly stretching this out for all its worth.",
    "Don't get me wrong, the initial appeal is fantastic. A bunch of gangsters post World War causing havoc and standing up against the man - sounds great. Unfortunately the story has no direction. The show becomes monotonous and the characters predictable. Each season has a focal point but in essence it's all the same.",
    "First few seasons are great fun, a bit silly in places but still has an air of cool about it. Tom Hardy adds a lot. Season 5 is cringey and boring.",
    "Certainly had potential, started very interesting, but it quickly became repetitive and boring. Too bad, because the cinematography is good and the actors are not bad. It is hard to take this show seriously especially with such unlikable characters. The show is interesting at times but it is too pretentious and the dialogues are too modern.",
    "Peaky Blinders is more than just a gripping tale of crime and power; it is a meditation on ambition, trauma, and the human condition. Cillian Murphy's extraordinary performance as Tommy Shelby is at the core of this experience. The series changed me because it forced me to confront the darker sides of ambition and power.",
    "If you like Victorian crime shows like Ripper Street, Whitechapel and Sherlock Holmes then you will love this. Peaky Blinders is told from the criminals point of view. You get a feeling of belonging to the Peaky Blinders and despite all the terrible acts they carry out there is a fundamental decency somewhere deep down in Thomas Shelby.",
    "Peaky Blinders is a must-watch for fans of historical drama with its richly detailed setting and complex characters. Set in the aftermath of World War I, the series follows the Shelby crime family led by the enigmatic and ambitious Tommy Shelby brilliantly portrayed by Cillian Murphy. The writing is another high point with sharp impactful dialogue that often borders on poetic.",
    "It's a great series, I loved it. Everything about it was so great, the acting, the writing, the directing. It drew me in from the first few episodes. The series gives you an insight on England in the 1920s and 30s and a peek at some of the social and political conditions at the time. I was sad when it ended.",
    "The show is great for the first 2 seasons. Season 3 is watchable. Season 4 is disappointing and season 5 is unbearable. Acting, costumes and filmography is still great but the script is weak. Full of cliche, every episode is obvious, repeating the same story differently.",
    "Peaky Blinders is a masterpiece, from beginning to end. It has a very well structured plot, characters played to perfection and the more it progresses the more promising it becomes. Every character in the show evolves greatly and the final message they send to the viewers is powerful.",
    "I have never seen such a beautiful series in my entire life. The actors are amazing, the scenario is outstanding, the guest actors like Tom Hardy are amazing. Cillian Murphy and Paul Anderson belong to my heart. Must watch series.",
    "What to say about Peaky Blinders. The series is just perfect. I was never for a moment bored while watching it. The writing and music is incredible but the direction and cinematography is the best you will see in any series. Cillian Murphy is one of the best actors of all time as is Paul Anderson and Tom Hardy.",
    "Cillian Murphy and Sam Neill are amazing. As I watched the series I found myself extremely impressed with Cillian Murphy's acting. The entire cast was great. The writing, character development, the acting, the soundtrack, the camera work, the grittiness of it all. This show started spectacularly and rips right through the gate!",
    "I was not expecting it to be this good. The story is interesting, the acting is brilliant and the cinematography is just beautiful. When I compare Peaky Blinders to other popular TV shows that use sex, brutality and violence to shock the audiences, this sincere work is like needlework - fine, classy and detailed.",
    "Peaky Blinders leaves me entertained though morally perplexed. Despite its style and good acting, I feel unrewarded. It certainly distracted me from everyday life though ultimately I'm no happier and am scarred with vivid graphic images of murder and death. The lead characters are so torturously greyscale that it's hard to justify the ultraviolent immoral lifestyle they choose.",
    "The best TV show I have ever watched. The way it started, the way it ended, the way the events and the story progress, the dialogues that take it to another level, and the charisma of the characters. I loved it because of the dialogues, relationships, strength, and charisma of Thomas Shelby.",
    "I saw it when it was released and it is a masterpiece. Acting is perfect on all characters, storyline is on point. I keep rewatching it every year. If there is a series you need to see it's this one. It's so amazing how a show can affect you and this one did. Plenty of gore and violence but most importantly it also includes family loyalty.",
    "Set in the aftermath of World War I, the series follows the Shelby crime family led by the enigmatic Tommy Shelby brilliantly portrayed by Cillian Murphy. The show does a remarkable job of capturing the post-war atmosphere of Birmingham complete with its social and political turmoil. The writing is sharp with impactful dialogue that often borders on poetic.",
    "After watching the last episode I paused for thought. Whether the first episode where Thomas Shelby rides in with his black horse or the very last where the white horse rides out, it reeks of passion, anger, pride, angst, fear and fight. The set is atmospheric with magnificent shades of grey. Music magnificent enhancing every moment. A masterpiece.",
    "This must be the best show that the BBC ever made. I absolutely love everything about this show. The constant battle between Cillian Murphy and Sam Neill is an absolute delight to watch. The storyline with the dark industrial atmosphere of Birmingham post World War One all adds up to be one of the shows you can't dislike.",
]

add_show("Peaky Blinders", peaky_blinders_reviews)

✅ Peaky Blinders predicted as: Balanced
✅ Added 50 reviews for Peaky Blinders


In [88]:
simpsons_reviews = [
    "No one could've imagined that The Simpsons would become as big as it did. The first three seasons showed them as if they were an actual family. By Season four the show took a turn for the best, becoming more of a satire with clever witty and sophisticated humor. Seasons 4 to about 10 are often said to be the Golden Age. However as the year 2000 came the humor became more lurid and toilet like. Despite this the show continues to remain popular even today.",
    "The golden age of The Simpsons seasons 3-8 is flawless television. It was awe inspiring how they managed to structure joke upon joke upon joke. The Simpsons was intelligent, heartfelt, structured and most of all FUNNY. However all good things must come to an end. From season 11 things started to drastically change. Now on the 31st season the show is just completely unwatchable. The golden age is some of the best television ever constructed and cannot be ignored.",
    "When I was 10 I adored the Simpsons. Me and my siblings watched it every day after school and got a bagful of laughs. After seeing the Simpsons Movie I decided to give the show another chance and I am glad I did because it is smart, creative, funny and original. It is true that the show has been declining in written quality but the more recent seasons are still watchable thanks to the animation and the endearing characters. The voice acting is exceptional bringing the dysfunctional family to life.",
    "Brilliant television series that could probably be best described as The Flintstones gone stark raving mad. The Simpsons everyone knows them. Love it or hate it, it is near impossible to criticize the intelligence and creativity of this series. The show works because of great comedy but also great lessons that can be taken from most episodes. The people within the program may be animated but they are just as complicated and vulnerable as the people watching them.",
    "What can I say about the Simpsons? From seasons 1-13 comedy appropriate to the society was cleverly exploited causing the Simpsons to become a global phenomenon. From seasons 14-19 Simpsons episodes were mediocre. From seasons 20-26 storylines seemed to consist of depression in the Simpson household and offered little to no comedic value. The writers have quite literally lost the plot and the show really should be cancelled. I rate The Simpsons a 10 simply because of its earlier brilliance.",
    "The show caused serious controversy at the time, was totally unique and changed the television landscape forever. South Park and Family Guy piggy-backed off the Simpsons; it was the precedent and they simply pushed the envelope harder. Still the greatest animated TV show irrespective of what's happened recently.",
    "Seasons 1-9 of the Simpsons were absolutely fantastic, easily some of the best television I have ever seen. The writing was sharp and witty, the characters unpredictable and to some degree you could relate to them. The Simpsons is probably the most famous TV show ever. The newer episodes are over the top, pointless and predictable. They should just end the show now whilst it's still decent.",
    "The simpsons were the best for the first 12 seasons. By season 13 to today they are terrible. The plots do not make sense. Homer is stupid but he was hilarious, now he is a dimwit that is just plain stupid. I really think this show will not last long due to the boring episodes.",
    "This is a classic show but only the first 20 seasons are worth watching. After they were bought by Disney they went downhill. It's great and funny the first 20 seasons and then it just gets tiring and the humor is more for kids and family.",
    "No cartoon and even dare I say tv show has had such a profound impact on the zeitgeist. Pretty much everyone knows The Simpsons regardless of their demographic. The characters are all likable and unique. The storylines are original and interesting. The writing is incisive, snappy and witty. There is a noticeable decline in quality as the series progresses, but it manages to hang on to its brilliance for many many years.",
    "Yet again Disney ruined something which was once brilliant. I used to watch The Simpsons and always had laughs at the gags and character exploits - not anymore. Why do they have so many musical numbers and woke storylines? Dull and uninspiring writing now.",
    "I surprisingly had a very difficult time rating this show. Although the series does still make me chuckle here and there the quality has completely fallen off in the show's later years. When The Simpsons first went on the air for at least 8 seasons it was one of the very best comedy shows ever produced. The characters were great, the writing was slick, it seemed like every single joke hit its mark. I laughed out loud all the time.",
    "What more can I possibly say about a TV show that has already been praised to death? It's a very clever and intelligent show - they never dumb anything down. In the early days The Simpsons derived a large part of their popularity from the everyday down-to-earth unglamorous average-blue-collar-slob aspect of the Simpson family. This show is extremely densely packed with jokes - everything from cerebral witticisms and sly satire to Homer falling down and going Doh!",
    "Family Guy seems to be the in thing right now but The Simpsons is still the hierarchy of animated television. This show can be enjoyed by all races, religions, and ethnicities alike. Things do tend to wear over time and people's likes and dislikes tend to change. Along with Seinfeld and Married With Children, The Simpsons is and always will be a most noteworthy sitcom in TV history.",
    "The Simpsons is an animated American sitcom created by Matt Groening. It is a satirical parody of the Middle American lifestyle. The show is set in the fictional town of Springfield and lampoons many aspects of the human condition as well as American culture, society as a whole, and television itself. Time magazine's December 31 1999 issue named it the 20th century's best television series.",
    "The Simpsons is truly a one of a kind show. On one hand it can be educational - there isn't a reference to someone or something that kids would not know. On the other hand you have your comedy. The Simpsons is very random and that's what sets it apart drastically from the others such as Futurama and King of the Hill.",
    "The Simpson's is without a doubt the greatest comedy to come out of America. It has covered everything from everyday life, endless special guests, to parodies of famous movies. But now I think the creators are running out of ideas, the jokes from the new episodes are going from classic to desperate. I think they should finally close the book of the greatest piece of animated comedy.",
    "This show may have gone downhill significantly over the years but we have to remember how amazing it once was. The Simpsons has several notable features including lots of brilliant humour, excellent writing, social commentary, a lot of voice talent, and impressive animation. It holds the record for the longest-running cartoon show. Unfortunately it hasn't been the same in recent years gradually declining in quality.",
    "The Simpsons is a world treasure but I feel the series has gone on for far too long. Gone are the days of universal innocent jokes that happened in the earlier seasons. The latter episodes seem focused on current events making them less relatable to a wider audience. Overall this is a series that will remain unforgettable and the long-running work of the writers, voice actors, directors and animators is to be commended.",
    "Such a classic funny family with every episode being a whole new story. The characters are well developed and that is why the show has worked well for so many years. I love this show so much! It makes me feel comforted and I really enjoy the fact that it doesn't require a lot of thought. If I've had a busy day I can just put an episode on and watch it with ease.",
    "I have watched every episode of this show and will continue to watch. I would be lying if I said I didn't enjoy myself for a portion of it. Though I'd also be lying if I wasn't very critical of this show and didn't hate a good amount of its episodes. This 6 is carried by how good the good is.",
    "To me The Simpsons from season one right up until season ten was one of the funniest television shows ever to air. Creator Matt Groening together with Sam Simon and James L. Brooks should be very proud of the work they achieved. The jokes range from silly slapstick all the way through to sharply satirical and everything in between. Once we move past the end of the tenth season there is a definite drop in quality. Once we reach the fifteenth season the standard has truly dived.",
    "The Simpsons is the longest running animated TV series since The Flintstones. We have Homer Simpson one of the most beloved TV characters of all time with his famous quote Doh! He's an overweight lazy and not the brightest bulb but so incredibly lovable. The show has been ripped off so many times but nothing has come close to the brilliance that the Simpsons writers have brought us.",
    "The Simpsons is one of the best TV shows of all times. You could watch each episode 20 times and never get tired. Some watchers argue that the show went downhill rapidly after about the 10th season. The Simpsons was an amazing show until after season 6 and then was still good until around season 12.",
    "The Simpsons is without a doubt one of the best shows of all time. I watched them from the very beginning, back when they were shorts played at commercial time on the Tracy Ullman Show. If you were to count all the laughs over the last 20 years you would find that The Simpsons are funnier than Seinfeld. The characters are well established and the city of Springfield seems to have everything you could want.",
    "To me it's hard to picture life without The Simpsons. All the sub-personalities that compose my dark side get permission to live and breathe when I watch that show. Homer ends up taking responsibility by accident which makes my inner Hallmark self happy. I find most of the older episodes better than the newer ones but it's still the Mad Magazine of the new millennium.",
    "The first 12 seasons were great, they were funny, entertaining, and I binge watched twelve seasons in just a few weeks. After that it got boring and started feeling like a broken record. It felt tedious to watch and almost like a chore. It was basically just background noise. Honestly it isn't even worth watching 12 plus seasons.",
    "The Simpsons was my go-to comedy from 1992 until the movie came out. From 1992 until 2007 I used to record save and constantly rewatch the Simpsons. The writing was top notch, the parody of real life things was great, and voice acting was flat-out superb. Then the movie came out in 2007 and it had a few moments but felt lackluster. Eventually the writing went from meh to flat out lazy and about as funny as a terminal disease.",
    "The Simpsons is a show that has sustained ten years of constant humor. The stories have gradually become better and the second fiddle characters were getting more screen time which translates into a much more realized show. The pop culture references abound and delight those who can pick them out. The guest stars aren't there as a special appearance - they actually work into the storyline.",
    "Not only is The Simpsons the best TV show ever made it is also one of American history's most important political forums. Groening and his co-creators have never been backstroking anyone and certainly never kissed any ass. They show us a society beyond salvation and we are laughing our brains out.",
    "The Simpsons was once a brilliant parody and satire of American pop culture and Western society. It was designed for families - the kids could relate with Bart and Lisa and there was plenty of moral and tongue in cheek jokes about society and culture with a host of guest stars. It's a shame what's happened to this show. The free-wheeling gags it used to deliver with such ease are now weighted down.",
    "The Simpsons which first broadcast as a half hour TV show over twenty years ago was at first a TV show everyone loved. Almost all of the jokes were 100% original and it was the funniest show on air. But the newer ones are just painful to watch. Groening is rapidly running out of ideas and now it's showing.",
    "It was once one of the funniest shows on TV. In the earlier seasons when the characters were more like real humans were the highpoint of the show. Now it's just one outlandishly ridiculous sight gag stacked on top of another with no real substance or plot behind it.",
    "This was once one of the funniest shows on TV. What was once a brilliant parody and satire of American pop culture has now become something unworthy of being satirized. It's a shame really, tragic and almost a parody of itself that it is still on the air. The Simpsons is an inevitability.",
    "The Simpsons has become the longest running show on TV at the cost of its core integrity. In its prime it was the best thing to grace the small screen - a funny ground-breaking animated comedy with lightning-quick wit and insightful social and brilliantly integrated parody. It created its own universe with an entire town of original characters. The Simpsons is now a pale shadow of its former greatness.",
    "If not The Simpsons there surely wouldn't be such great cartoons as Family Guy, South Park and American Dad. After so many years I still can't get enough of this family. It's also amazing that creators still have new ideas with so many episodes already aired and with very big competition from other shows.",
    "I have been an avid fan of The Simpsons since it started when I was only four years old. The show has been almost always uniformly perfect from the start. My whole childhood and adolescence would be unthinkable without them. The Simpsons inspired Family Guy and American Dad. No they are not as good as The Simpsons. They are good on their own terms but The Simpsons started it all.",
    "I can't describe how much this show affected my life. The show could probably have 3 to 4 different ratings depending on the era. For me the first 10-11 seasons would easily be a 10. Season 3 is when it became great. It slowly goes down hill after that before the bottom drops out. The show needs to end, it's well past its sell by date.",
    "What can I say, The Simpsons is in my opinion the greatest show of all time. The skits and jokes work on so many levels. Every episode revolves around a different character which makes it even more exciting. The characters are classics. This show is not just for children and teens. It is meant for every age group. Overall, this show is gold.",
    "The Simpsons is the best TV series of all time. The skits and jokes work on so many levels. Even the so called bad episodes at least have a few great gags in them. I can relate to it because I have been watching it for over half my life. Sometimes I have this thing called the Simpsons drought where for some reason I don't watch it at all for maybe six months. The most obvious aspect is its unfortunate decline but this show is gold.",
    "It is no longer possible for me to imagine a world without The Simpsons in it. I was ten years old when I saw my first episode but I was hooked on the show ever since. Sure there are a few lesser episodes especially in the last two series but that only encourages me to watch the old ones over and over again.",
    "This was once one of the funniest shows on TV. Now it's just crap. Now a days the show is just one outlandishly ridiculous sight gag stacked on top of another with no real substance or plot behind it. The earlier seasons when the characters were more like real humans were the highpoint of the show.",
    "The Simpsons is a show that rewards paying attention. There are always enough obscure pop culture references or subtle background gags to ensure that the second third or tenth viewing of an episode will find you noticing something you hadn't before. It's satiric, witty and funny. The Simpsons and South Park are probably the only American comedy series that are actually funny and satiric.",
    "It's very hard for me to try to put a rating on this show. At one time it was the best comedy series on TV and possibly even the best one ever. Unfortunately it went into a decline after the first ten years or so and then a steep decline to the point where it is less than a shadow of what it once was. They really should have quit several years ago.",
    "The Simpsons is a show that has been there for me. It started in 1989 which was the year I was born. It can be touching occasionally and more often the viewers are treated to an unequalled cavalcade of obscure references, surreal sight gags, wacky adventures and self-mocking irony. I can admit the show isn't as perfect now as it was years ago but it still constantly makes me laugh.",
    "The Simpsons is the best television program ever. If you were to count all the laughs over the many years you would find they are funnier than Seinfeld. The first standalone episode was funny and the last one they made was funny. The laughs just never end. I can watch reruns all day long too. It just never gets old.",
]

add_show("The Simpsons", simpsons_reviews)

✅ The Simpsons predicted as: Balanced
✅ Added 46 reviews for The Simpsons


In [89]:
money_heist_reviews = [
    "La casa de papel started as a promising bold show. The first and second season had amusing and strong plot. Well defined characters and a storyline that could certainly catch your attention. The duel and chess between the protagonist and antagonist was surprising and full of ups and downs. But then came the third fourth and fifth seasons. The strong amazing plot turned into a cheesy foolish joke. Unnecessary flood of new characters whom you cannot relate to at all. The plot which was filled with surprises became totally predictable and utterly boring.",
    "I've heard about how good Money Heist is since it started back in 2017 but for some reason I kept putting off watching it. I finally gave in and started watching it recently and now I can't stop. It's every bit as good as everyone says it is. The writing and acting are both excellent, there are so many twists and surprises that it will keep you on the edge of your seat and guessing throughout the entire series. Just a little warning - once you start watching it it's hard to stop.",
    "This show deserves every possible award it could get. The original storyline, the masterplan behind every move and character is simply out of this world. This show is one of the few that kept me on the edge of my seat for every season. This series ended the same as Friends did: at its peak. The story was told, there were no unnecessary storylines added so all in all you're in for one hell of a ride.",
    "Season 1 and 2 were great and amazing. But then Netflix saw a chance to make money and prolonged this show into unnecessary seasons that completely ruined it. This last season was a disaster filled with unrealistic stuff.",
    "I am a biggest fan of this particular series. Each episode carries the suspense and makes us more eager to watch another episode. I personally love the character of The Professor, meanwhile I admire the intelligence of Berlin and Palermo. The director characterized these characters very well and each situation grabs the pulse.",
    "All my friends were talking about La Casa De Papel as if it was the best series ever. Once I started I binge watched the first four seasons. Yes it is addictive, you can't help wanting to know what will happen in the next episode. The plot itself is sometimes very far fetched but the cast is really good. Everyone will have their favorite character, some will die and some will survive, that's what makes this show really entertaining.",
    "Totally surprised to see the amount of negative reviews. Personally this is one of the BEST series ever created. The story line, the characters, the dialogues, the story narration by Tokyo, the emotions, the passion and the action, everything is just magical and very captivating. Be prepared to handle loads of exciting cliffhangers. This series is pure work of art.",
    "If they could have stopped in season 2 this series would have been the best heist series ever made. But they didn't and season 3 and 4 are meaningless, boring, not convincing and illogical.",
    "Money Heist is a Spanish crime drama series that has gained a massive following worldwide. The characters are well developed with each of them having their own backstory and motivations. The show's cinematography, soundtrack, and attention to detail contribute to its high production value. However it can sometimes feel like the plot relies too heavily on certain conveniences and coincidences.",
    "What started off as a superbly intelligent and gripping drama series has slowly but surely fallen into the Hollywood trap. If new to this, savour Seasons 1 and 2. By season 5 plot is entirely replaced by countless hours of shootouts with thousands of mostly non-lethal bullets fired. Difficult to rate overall - 9 to the first two, and 2 for the last.",
    "Well the writers and producers outdone themselves. They started with a good idea, intriguing characters and an elaborate plot. By the end of season 2 it became apparent they could not drag this anymore. Season 3 was just a remake of season 1. Season 4 is an awful soap opera with no actual plot and no actual action.",
    "I found this Spanish TV series by chance and I really recommend you watch it. It's one of the best series I have seen lately. A chess game between the police and the robbers, you will be surprised by its development. You cannot foresee anything.",
    "I watched this series and there is nothing comparable with it. Intense drama makes you on the edge of your seat experience. I still want these types of shows but can't get that intense heart pumping and blood pressure increasing experience elsewhere.",
    "Watched 8 episodes yet but took only 8 hours. Couldn't stop watching with all twisted episode ends. The intro video and music are amazing. Story keeps getting you every time with every new piece of information. Impossible to get bored. A well designed heist plan extended through the season.",
    "I usually pick up any TV series with thorough analysis after reading many reviews. But this one I picked randomly. I binge watched it for almost a day because this thing is so fast paced and keeps you awake as if you're on a rollercoaster ride. The plot twists, the anticipation, the direction - you'll get addicted and start admiring some characters while hating others.",
    "Character development, story development, script writing, acting. Most series don't have even one of these. This one has it all. And without any CGI or technical tricks they batted it out of the park every episode. No wonder it won so many awards.",
    "Every episode had its gripping and human slow moments, such a wonderful balance. Couldn't wait to see the next episode. Each one had surprises, no standard script which is so enjoyable. The actors played so well, better than many of the better known Hollywood names.",
    "Could not stop watching this series. What an excellent set of actors fully into their roles in a natural way. Refreshing storyline unlike most other movies or series about bank robberies.",
    "With some engaging characters and a money heist script with several twists this started off SO WELL. Unfortunately someone realized they had an audience they wanted to keep around for a while and from that moment on it went downhill. It developed so many glaring plot holes and the telling of the story grows more and more formulaic by the episode.",
    "This has been a real treat! An amazing series, great acting, direction and such a suspenseful story. I love heist movies and I literally couldn't stop watching through the night. The characters are simply amazing! Don't miss this!",
    "Bid good bye to common sense. The more you watch it the dumber it gets. The idea of the plot is good but the direction is just like a soap opera.",
    "I watched Part 1 without having much expectations. At the end I just had to admit it was a brilliant series and everything that happened throughout actually makes sense. Sure it has its flaws. Sometimes it's a little too over dramatic. But the overall suspense, twists and action more than make up for it. It was just straight up smart and unique. Highly recommended.",
    "Seasons 1 and 2 were great and well worth a watch. So fast paced and full of tension, but most of all they were original and fun. Since then someone at Netflix has unfortunately cashed in and completely destroyed this show. It's become a nonsensical soap opera. I would absolutely recommend you watch the first two seasons and forget they ever made anymore.",
    "One of the many great series on Netflix depicts a group of people breaking into Spain's national mint. Rapid-fire dialogue, clever editing, and all sorts of twists and turns make this one of the best shows out there.",
    "It is a great non-Hollywood European series. It is well paced, well directed, actors are selling their characters well mostly. It is a pleasant romantic fiction adventure with a superficial revolutionary message. Generally well done in most aspects.",
    "Fast paced. Action. Great acting. Ingenious plot. This show is perfect.",
    "This must be one of the best series I've seen lately. It is so thrilling and unpredictable in every single episode, that you have no idea how it will end.",
    "Season 1 and season 2 were 9/10 for me. However season 3 was meh and season 4 pretty meh except the two last episodes. Unfortunately another show Netflix destroyed because of its success.",
    "I don't get the hype. With a 100% positive critics rating on Rotten Tomatoes and with many of my friends going crazy for it, my opinion might not be very popular. But I find Money Heist to be utterly ridiculous, overly melodramatic and just plain stupid.",
    "I love shows like this. The professor is one of the best characters I've seen in a series like this. If you can get past how annoying some characters are this show is well worth the watch.",
    "I'm completely in love with this tv show. I just finished the 3rd season in one day, I am just addicted. Extraordinary actors, story, message, everything. This season made me so emotional, you fall in love with every single character. I can just say MASTERPIECE!",
    "Money Heist took the world by storm with its unique heist story and captivating characters. The Professor a genius strategist with a Robin Hood complex is as intriguing as he is ruthless. The series excels in building tension. Cliffhangers and shocking plot twists keep you glued to the screen. The iconic red jumpsuits and Salvador Dali masks become symbols of rebellion.",
    "Seasons 1 and 2 were great and well worth a watch. Since then someone at Netflix has unfortunately cashed in and completely destroyed this show. It's become a non-sensical soap opera and how some of the characters are still allowed to make decisions is beyond me.",
    "Money Heist is awesome! Whenever a new season drops I dive right in and binge-watch it in a day. This show takes you on a wild ride of emotions. It's exhilarating and you become so invested in the characters feeling like you're part of the heist in some way.",
    "Words can't describe the awesomeness of this TV show. It is so terrific on a level comparable to Breaking Bad with every actor delivering outstanding performances making all characters both likeable, loveable and remarkable.",
    "A band of robbers led by a man known simply as the Professor infiltrate and rob the Spanish National Mint. Entertaining initially and could have been a masterpiece. The initial plot was very clever and intriguing. Unfortunately instead of being tight the plot diverged into silly escapades. The further into the series one went the worse it got. By the end I just wanted it to be over so my intelligence would no longer be insulted.",
    "I love this series. I've watched all the seasons and have never been disappointed. For a long time there was no such cool embodiment of the idea in life. You truly care about each character and don't know what to expect.",
    "The show will keep you on the edge of your seat and you really get attached to the characters. The best parts were the love stories and the redemption of certain characters. The acting of Pedro Alonso as Berlin and Alvaro Morte as the Professor was excellent. I've recommended this show to everyone I know.",
    "This series is just brilliant. I have to admit that it took me a few episodes to really get into it but once you are into it you can't stop watching. It is really thrilling like a cat and mouse game between the police and the bank robbers fighting over who has the better plan.",
    "You will laugh and you will cry. It really is that good! Perfect score, perfect script, great character development, amazing actors, thrilling plot twists, great photography. Everything is there! You will feel the show in your gut and love it! Watch it in Spanish and not dubbed.",
    "Seasons 3-5 are sort of a 2.0 version of seasons 1 and 2. Totally unnecessary. It looks like a repeat made with a bigger budget. Regrettable how entertainment companies work.",
    "This is the best series I have ever watched. Extraordinary actors, story, message, everything. The professor always the best. Berlin who always lives life on principles. Loyalty, bravery and leadership with skills. Emotions made this series more special.",
    "My god this show is amazing. It's thrilling, emotive, dark, funny, shocking, surprising and very very clever. The best thing I have watched in a long time. It's a must watch for anyone.",
    "I heard all of my friends say that I needed to see La Casa de Papel. I liked the show in general but it is so unrealistic at times. If you love over the top action movies or just another crime series then this one is for you. But definitely do not expect the best series ever. It's a decent series. Entertaining for once.",
]

add_show("Money Heist", money_heist_reviews)

✅ Money Heist predicted as: Balanced
✅ Added 44 reviews for Money Heist


In [90]:
vikings_reviews = [
    "Seasons 1-3 and the 2nd half of season 4 are the best this show has to offer. Ragnar was what made this show great. Vikings is brutal and has some very dark and complex moments. First 3 seasons 10/10, season 4 7/10. Ragnar deserved better.",
    "The first 3 seasons are excellent. I used to recommend this series to all my friends. Few episodes, good story, unique characters. Then the fame came and brought greed. History Channel decided to double the earnings by doubling the number of episodes in season 4. Without enough content to fill the episodes the show became slow and boring.",
    "Vikings is one of the rare shows where it is never dull and full of entertainment. It's a show with plenty of violence, drama and mystery. The character development is what keeps the show fresh and interesting. It takes you into this world of Vikings and keeps you intrigued throughout the series. I know everyone keeps talking about how the show dropped in quality when Travis Fimmel left but it was still a pretty good show after he left.",
    "The first few episodes don't really convey the scale of the ones that would eventually follow. Ragnar is not a good man or father and the danger of the character is always present in Fimmel's wild eyes but you still understand why his men fought for him. The scale of the show gets bigger and bigger and the battle scenes more and more expansive.",
    "First four seasons are 9/10 and last two are 2/10.",
    "I've been waiting for someone to make a good series about the Vikings. It really looks authentic. The acting is mostly top notch. The story itself is really exciting and I just want to watch more. I highly recommend this show.",
    "Travis Fimmel as Ragnar is just incredible. Think of him like a Viking Jack Sparrow but with so much more going on under the surface. Every little head tilt smirk and hand gesture tells a story. Gustaf Skarsgaard's Floki is brilliant chaos in human form. What makes Vikings special isn't just the epic battles. It's how deep they dive into everything - the spiritual journeys, the family drama, the politics.",
    "I'm not a fan of historical and quasi-historical films and series so I had no high expectations for Vikings. But the first season bought me right away. It's completely different from anything I've seen so far. The acting and characterization are excellent and the action scenes are incredibly believable and realistic. It gets more and more complicated and tense. In the later seasons the story branches out in too many directions and gets overly complicated.",
    "I really enjoyed this first episode. Writing is strong. Acting is good and more importantly I am really fascinated by the characters. The most impressive thing was the look of the show - photography, art direction and the whole look is so lush and rich it looks like a feature film. More raw and intense than Game of Thrones but still with a nice magical feel to it.",
    "I've watched the two first episodes and I can say that I'm POSITIVELY surprised. It's well written, exciting and charming. The show is very enjoyable especially if you're interested in Scandinavian history. I love Norse Mythology and I am very happy that this show aired.",
    "After last season clearly you can see a few young actors are trying to be good enough like Travis Fimmel. The storyline and role-playing are really weak in the later seasons.",
    "Something a bit different and very enjoyable. This is a fantastic and exciting look at the relatively unexplored Viking world. The show has a quality feel to it. The sets are impressive and believable and the direction and framing of each scene is obviously set by a skilled hand. The characters are all interesting and likable and the acting is top notch.",
    "First seasons was so great. I was telling everyone that this show is way better than Game of Thrones. Not sure whether the producers decided to make it as provocative and shocking as GoT or they were out of ideas. The last season is such a waste of time with boring emotionless banal dialogues with characters you have no emotional bond with. The first 3 seasons are excellent and worth watching.",
    "They should have just stopped after season 3. Season 5 was so bad that I couldn't finish the last 2 episodes without forcing myself just to confirm how bad it was.",
    "Maybe it isn't real 100 percent history. Some liberties were taken but for the most part it seems to ring true. At times it almost seems like a contemporary crime drama. The storyline is fairly literate and not the usual mindless adventure. The characters are convincingly drawn and likely motivated. The music is good and the full size replicas of the ships are very accurate. Never dull with plenty of interest to engage the viewer.",
    "A fantastic and exciting look at the relatively unexplored Viking world in the same vein as other fictional historical dramas such as Spartacus or Rome. The plot centres around a single character and their family. The show has a quality feel to it. The characters are all interesting and likable and the acting is top notch.",
    "First things first - the characters brought to life by the brilliant performances are spectacular. The writers, producers, directors and most of all the actors were all so inspired with so much passion for the product that every scene evokes its own life. Though the show is super serious it doesn't always take itself too seriously which is rare and very welcome.",
    "I watched Vikings in 2020 and it is so worth it. The first 3 seasons are 10/10, just amazing really and season 4 is very good as well. You can save time though and skip the last 2 seasons unless you are really hooked on the story.",
    "The series starts well and gripping until the third series after which it gets chewy like bubble gum. Producers get greedy and trashing out episodes. Thousands of people die but not many of the protagonists. After watching a few episodes in series 4 I regret the hours wasted watching this unending saga.",
    "The first five seasons of this series are almost perfect. The characters and their arcs are so good, the actors and actresses topnotch. No the show wasn't all that historically accurate but oh my gosh was it entertaining!",
    "I only watched this show up til Season 5. This show is amazing. The development in characters is spot on. Following Ragnar, a very ambitious man going from a farmer to a king. The character development with Ragnar, Floki, Athelstan and Ecbert is superb.",
    "Season 1 to 3 are awesome. 4 to 5 are meh not the best but solid. Season 6 ruined a good show.",
    "The seasons of the show all would be ranked differently. Overall the show was good and I enjoyed it. It was entertaining enough for me to rewatch it a couple of times. Seasons 1-3 are the best with good character development and action. Seasons 4-5 start to slide down the slope. Season 6 definitely struggles until everything starts coming full circle.",
    "'Vikings' continually looks wonderful. The sets and scenery take the breath away and are very atmospheric. The photography is both sumptuous and haunting. The music stirs up the emotions. For the first three seasons and most of the fourth the writing is intelligent and thought provoking with an expert mix of amusing humour, truly touching dramatic moments and suspenseful tension. Unfortunately Seasons 5 and 6 are nowhere near as good and felt like a different and inferior Vikings show.",
    "Vikings is a brilliant masterpiece with exceptional acting performances by all the lead actors. The legend of Ragnar Lothbrok and the saga of his sons was beautifully scripted. I enjoyed watching the savage battles, barbaric raids, cruel lifestyle of the Vikings, beautiful Scandinavian lands and the very intriguing character arcs. An impressive, pleasurable watch with haunting music and shocking surprises.",
    "Travis Fimmel as Ragnar Lothbrok is unforgettable, embodying the cunning and charisma of a legendary Norse hero. The series masterfully balances brutal battles, intricate character arcs and the exploration of Viking traditions versus Christian values. Though later seasons shift focus to Ragnar's sons the drama and action remain compelling. A captivating saga that's both brutal and beautiful.",
    "One of the best historical TV shows of all times. So well written, directed and acted. The production design is astonishingly accurate and the enactment of the ancient Vikings' ritual is beautifully made.",
    "Exiting story in the start. Excellent performance of most actors and production. But then after Ragnar's death the story went in different directions to fill the schedule. Lower quality and some obvious script errors. Overall it is a good production but my rating is for the first 3 seasons, the ending seasons do not reach 7.",
    "At first I did not expect that this series is this beautiful. After I watched it was a shock - I watched one of the best series I have seen in my life. My integration with the legendary character of Ragnar Lothbrok was tremendous. The best four seasons. Unfortunately after that I started to feel a little bored.",
    "This is one of those shows where they force the drama too often. It's almost like an ultra-violent soap opera that feels like it might be dragged on a bit too much. But you can't help yourself from pursuing the narrative, nor can you keep from occasionally cheering, gasping or crying. Travis Fimmel KILLS it as Ragnar. It's an emotional ride that is both satisfying and frustrating.",
    "The Viking era is an exciting one. The series itself has a wealth of legend to draw from and fill several seasons. The first episode is a good introduction. Since it is the work of the History Channel we get some authenticity in this series and not just entertainment.",
    "It used to be great and fun to watch but now the petty dramas are tiresome, there's no passion and excitement. I don't like any of the characters anymore. Ivar sucks, Floki sucks, there's no hero to root for anymore. Time to end it.",
    "The first 3 or 4 seasons of this show are brilliant. It is one of my favourite shows and Ragnar is one of my favourite characters of all time. I still enjoyed the rest of the seasons and would recommend them too but they fail to compare to the initial seasons. The last few episodes of the entire show made absolutely no sense.",
    "Not without faults but a show worthy of Valhalla. Travis Fimmel kills it as Ragnar. His character is fascinating. It's an emotional ride that is both satisfying and frustrating. The show adapts and evolves and despite its flaws it's a fantastic experience.",
    "Season 1-3: 9/10 one of the best historical series. Season 4: 7.5/10, should have ended it right there. Season 5: 6/10 boring and stupid but has its moments. Season 6: 3/10 complete disaster more like fantasy than historical fiction. Watch The Last Kingdom instead - it's Vikings just without disappointments.",
    "The twilight of the Gods is upon us as the age of Viking heroes comes to an end. It has been an 8 year journey filled with great characters, multilayered narratives and countless emotional moments of both joy and heartbreak. There have been ups and downs throughout the seasons but I loved every moment and I am satisfied with the way everything has ended.",
    "Vikings is an amazing series with incredible battle scenes. In the first seasons we almost can't point out negative aspects, all the episodes make us want to watch more. In the final seasons some episodes began to be a little uninteresting which made me think that maybe the series should have finished earlier to be perfect.",
    "I had given this show 9/10 earlier and the first few seasons without a doubt deserved it. However things became increasingly worse as the show progressed. In the later seasons unrealism reached a frustrating stage with character decisions that made no sense.",
    "After nearly thirty days of watching it part by part, finally completed the series. Vikings is a brilliant masterpiece with exceptional acting performances. The legend of Ragnar Lothbrok and the saga of his sons was beautifully scripted. I enjoyed watching the savage battles, barbaric raids and the very intriguing character arcs. A pleasurable watch with impressive cinematography, haunting music and a perfect conclusion.",
    "Having just finished Vikings I'm left in awe of its powerful storytelling, richly developed characters and breathtaking visuals. This series doesn't merely depict Norse mythology - it plunges you into the grit and glory of the Viking Age. Travis Fimmel as Ragnar Lothbrok is unforgettable. The series masterfully balances brutal battles, intricate character arcs and the exploration of Viking traditions versus Christian values.",
]

add_show("Vikings", vikings_reviews)

✅ Vikings predicted as: Balanced
✅ Added 40 reviews for Vikings


In [91]:
chernobyl_reviews = [
    "I was born in Pripyat. I was four years old when the accident happened. Watching it is more horrifying than living through it. We didn't know what we were dealing with. This mini series is a masterpiece, perfect in every way. Both of my parents worked at Chernobyl plant. I think this show is the best depiction of the Chernobyl disaster and the stories of its victims. This show is to remind all of us of the cost of lies.",
    "A Belarusian here, born in 1983. Chernobyl is never forgotten in Belarus and all the details of the tragedy are widely known. Yet the series managed to depict the horrible events in a way never before seen. I had to literally pause a couple of times to comprehend what had just been shown. Goosebumps and tears, what a masterpiece. The tragedy will live forever because of this haunting masterpiece.",
    "I'm from Kiev, Ukraine. I was born in 1983 and I was 2 and a half years when the Chernobyl catastrophe happened. The authors of this film made a GREAT job to show every detail of what the world looked like in the times of the Soviet Union. The most important thing that this film shows is that the Soviet authorities lied to people about this catastrophe all the time. I highly recommend to watch this film. This is a tribute to all the heroes who lost their lives in a radioactive flame.",
    "I'm Ukrainian, born in 1988 and still live here. I want to give the authors of this show a big thumbs up for the whole set they have made. Every little detail of the buildings, flats, uniforms, clothes, cars - almost everything is 99% identical to the real things of the time. I am shocked at the level of production of this show. No one outside of Ukraine has ever made a good TV series about Chernobyl before.",
    "My husband grew up near Kyiv and his father drove one of the buses that evacuated the civilians from Pripyat. We watch this together and he is amazed at its authenticity. The set detail, the way the Soviet regime hierarchy functioned, the denial and secrecy surrounding the disaster. He tried to find errors or inconsistencies but has not been able to. The acting is impeccable by all. This story has been waiting to be told and there is hands down no one better than HBO to do it.",
    "Chernobyl is scarier than most horror movies in that it is a dramatization of actual real-life horror experienced by thousands of people on that fateful April 1986 morning and the years that followed. This disaster has haunted the nation, Europe, and the rest of mankind more than three decades later. And that creeping dread permeates the whole show. It's difficult to watch. But it certainly makes it a must-watch.",
    "I'm Russian. Amazing work! Never, you hear this, never ever before has Western cinematography made such an authentic film. I speak about details: cars, kitchens, clothes. Thank you for this.",
    "The cinematic quality is unbelievable! I found myself looking at many scenes with disbelief, a complete state of shock. Every actor did an amazing job at portraying each character and their feelings. I felt like I was contaminated by radiation after watching. That just shows how incredibly effective the show is at displaying the horrific events that took place in Chernobyl.",
    "I'm Russian, my father went to Chernobyl as a physicist quite a while after the explosion. I can't find words to describe how emotionally deep this film is. The details of the Soviet life shown in this TV series are incredible. The filmmaking and the music are absolutely unbelievable. This is one of the best things I've ever seen in my life.",
    "On 30th of April 1986, a cloudless sunny day in southern Poland, I was 13 years old. Teachers started running, closing all doors and windows, telling us to stay inside, that the air was radioactive. One hour later we were given Lugol's iodine. I've never written a movie review before. I've never seen such an accurate depiction of how a communist country operates. How it looks. I was literally time-traveled over 30 years back. Incredible. 100/10.",
    "In terms of series expression it moves in a style we can call documentary-drama. The gloomy atmosphere of the Soviets is very well reflected. The background music supports this. You feel the catastrophe up to your buttonhole. Acting certainly takes the business to a much higher level. Chernobyl is now at the top of the Top Rated TV Shows list on IMDb. This is a significant improvement. It really deserves a round of applause.",
    "Grim is an understatement. This is one of the very few shows that I'd bless with a ten star. It is subtle considering the subject but at the same time does not flinch in any way from the harsh and immediate reality. I'll never love this show but I will forever have had my eyes opened by it.",
    "I was born in Ukraine, 1971. This mini series exceeded my expectations. Authors put quite an effort to correctly depict intricate details of this catastrophe. People, behavior, political relations all very accurately depicted. I am thankful to the authors and actors for their job. My deep bow to those who lost their lives and health, saving millions of lives. EVERYONE should watch this show.",
    "Chernobyl is about the brave men and women who had to navigate the ridiculously scary nuclear accident at the Chernobyl Nuclear Power Plant in April of 1986. It is heartbreaking to watch as this shocking disaster unfolds. This show will grab you from the very first episode and not let go. The attention to detail is just awesome. I can't recommend this show enough.",
    "I work as an apprentice within the nuclear industry. This was the most horrifying thing I have watched in a long time. This isn't due to the fact that the show is actually scary per se but because the whole thing is true. The acting is excellent, you almost forget you are watching a show as the reactions and emotions displayed feel incredibly genuine and believable. The science included is spot on.",
    "This will go down in history. This will be shown in schools. Truly shocking and extremely realistic. And that's coming from an ex-Soviet citizen. I was amazed how detailed everything was, even the representation of the evil Soviet regime. Totally on point.",
    "I grew up in Soviet Russia, born in 1977 and this is incredibly accurate. Starting with the explosion, the incident itself, the way Soviet politicians and the media handled and covered it, and ending with all the little tiny details. This thing keeps you on the edge of your seat and coming back to the real world after watching an episode is like waking up. Thank you. I also wish some of the younger Russians see this show and stop romanticizing the Soviet era.",
    "My parents were located 120 miles north of the accident. Before any word got out they told me they remember seeing a film of dust in the sky. It's crazy to see the accuracy of this show. I watched it with my parents and their eyes started to water because to this day they cannot believe this happened. It's the only show that leaves me wanting more. HBO has created a masterpiece.",
    "I can not overstate just how good Chernobyl really is! It's one of the best mini-series ever made, right up there with Band of Brothers! It absolutely lives up to all the hype and love it's gotten since it first came out! It's one of the top rated shows ever for a reason because it's fantastic!",
    "An infamous world event captured in a riveting series. Chernobyl provides so much depth and wider information into the disaster. Deeply unsettling and painful to watch at times, the effects of radiation poisoning are explicitly shown in all its horrors. Everything in the series is visually stunning, staying truthful to history whilst always creating the correct atmosphere on screen. It is historically important and an absolute must-watch.",
    "I am Russian. I was 16 when this happened. My father was among people who were called to Chernobyl. A few years ago he had cancer. Now in remission. God bless him and all others for saving lives of millions, sacrificing their own. I don't bother with the British accent like some critics. They speak to my heart on the language of my heart.",
    "They say reality is far more horrifying than fiction. This proves it.",
    "I am from Minsk, Belarus. The series left me in tears. It all looks even more frightening than we could imagine. The course of events is being told quite clearly and accurately. The actors look pretty Russian and Belarusian and Ukrainian to me, and all the locations look very authentic and detectable.",
    "I watched this miniseries and I'm totally hooked. Hard to believe that these things actually took place and only 35 years ago. It's really quite frightening. The political and non-fiction elements give it a leg up. The acting from Jared Harris and Stellan Skarsgaard is excellent.",
    "Captivating from the first moment. The craft on display here is of the first order - the cast is perfectly chosen, full of gravitas and portent, the mood is suitably dark but recognizably human, and the play of events nicely paced. Chernobyl is great TV and an exposition of human frailty and sacrifice.",
    "It is hard to pin down any one element that distinguishes the series as A-class, but it has that somewhat indescribable character of screenplay, acting and theme that mix together to leave me with a sense of intrigue and wonder. Think Shawshank Redemption. When you watch Shawshank you know you're watching a masterpiece. Same here.",
    "This miniseries will go down as one of the best shows in television history. The casting was great and the overall creepy and dreadful atmosphere kept me hooked through all 5 episodes. None of the episodes dragged or were boring at all and the tension builds to a grand finale that should be viewed by everyone at least once.",
    "I liked the show but don't quite understand the hype. This is an interesting mini series and there is some good acting. After the first episode it really is quite slow with a lot of talking. There are parts that drag and the variety of accents is a bit odd. Overall a good show as long as your expectations aren't out of the world.",
    "I was born in Belarus and being a little girl I remember this horrifying time. Our history teacher came to class that day and tried to explain what happened crying without stop. It was the first time in my life I saw a man crying. I'd like to give big thanks to the entire production team for such an incredible authentic work. Everything from the set design to showing how the Soviet regime functioned.",
    "I've seen all the documentaries so I knew most of the history of the event. Nothing prepared me for the visual horror, the totally immersive details. Ep3 had me sobbing. This is a must watch. If you can, view it at night with no lighting and a good sound system - the music is absolutely haunting.",
    "The story of Chernobyl is well known all over the world however I don't think everyone knows the magnitude of what happened and how close they came to an even greater catastrophe. Visually it's immaculate, aurally it's brilliant. We feel the real terror and tension. We have incredible performances from Jared Harris and Stellan Skarsgaard. This series could have focused on sensationalism but it didn't - it treated each story with respect.",
    "This HBO mini-series is about the 1986 nuclear power plant disaster. The most incredible aspect is the roar of the exposed core. It breathes fire like a dragon. The thesis of lies and hubris and scientific truths and real heroism is the story's beating heart. The final episode has a tough job of explaining the accident and puts the exposition into the courtroom. This is prestige television at its best.",
    "I'm Polish, I was 8 years old when that happened. Me and all my classmates were forced to take Lugol's iodine before fallout hit Poland. This miniseries is absolutely outstanding. It's almost a semi-documentary. Please also check Chernobyl Prayer: A Chronicle of the Future by Svetlana Alexievich.",
    "After reading the overwhelming amount of positive reviews I had to give this miniseries a shot and god damn am I glad I did. This miniseries will go down as one of the best shows in television history. The casting was great, the overall creepy and dreadful atmosphere kept me hooked through all 5 episodes. A tragic lesson in human error.",
    "Terrifying, outstanding and heartbreaking all at once.",
    "Acting is superb especially Jared Harris. The story is so important and the writers did a bang on job as well as the director. This should be a mandatory show for high school students. This real life story is more scary than any fiction ever written.",
    "This show however takes the cake in atmosphere, soundtrack, setting, pacing, acting and cinematography. There has rarely been a show that has filled me with a simultaneous sense of sorrow, horror and menace throughout its duration. The end of episode 3 left me in a state of dread. This series is a slow burning existential nightmare and well worth watching.",
    "Easily the best miniseries I have seen. Beautifully shot and written. The accuracy is absolutely impeccable. The series makes you feel like you are there. At one point I could even feel my face warming watching some of the brave people trek through the reactor building.",
    "This TV show is a life lesson. You are not watching a TV show based on an event. You're just experiencing it with its causes, problems and all. The Art Direction and Set Decoration is just way beyond perfect. Makeup is beyond perfect. Acting performances are great. This is one of the best series of all time.",
    "If there is anything wrong with this I'm unaware of it. This has a large cast and everyone in large and small roles is absolutely perfect. The cinematography and production design are beyond criticism as are the writing and direction. This is very important television, something that cannot be said a lot of the time.",
    "I thought I became a victim of the low attention span generation, I struggle to get past 2 episodes of most box sets. Chernobyl is glorious! I cannot pull myself away from the screen. There are only a handful of good shows out there true to the word entertainment and this is apart of that squad.",
    "I have watched all the TV shows you can think of. This is mind-blowing.",
    "Been at the site, read all there was to read, watched all the documentaries out there. The aspect of this mini series and its execution is flawless and worth of my first review. An absolute must see.",
    "Informative, shocking and well acted. It is a good series. It fell short of the hype for me and was drawn out and slow in several places but I would recommend a watch if someone was struggling to find a new series to binge.",
    "I would suggest adding Horror to the genre of this TV series because I felt very scared and uncomfortable watching it.",
    "This mini series is a must-watch. You really get how dangerous and complicated nuclear power is. The direction, script and cinematography are awesome. It balances the science with the human story really well so it feels real and informative.",
    "In April 1986 two events happened that I will never forget: my daughter was born and USSR Chernobyl nuclear power plant exploded. This excellent HBO mini-series seems to disclose what really happened that fateful night. The performances are outstanding. Despite the running time the viewer does not feel tired or bored. This is one of the best mini-series ever made.",
    "I watched a lot of material on this matter and living all my life in post-Soviet Ukraine I was struck by how well they recreated the atmosphere of the Soviet town and the attention given to details. Not to mention excellent actor performance. It is dark and almost hard to watch but so was the real story. Highly recommend.",
    "Sometimes you watch something and it lives with you. Chernobyl follows the likes of Threads, a drama so realistic and a story so well told that people will remember it for many years to come. The effects and acting are truly first rate, as is the superb makeup. However it's the way that this story is told that's the best element. A story full of hardship, lies, cover ups, pride and of course love. Superb from start to end.",
]

add_show("Chernobyl", chernobyl_reviews)

✅ Chernobyl predicted as: Balanced
✅ Added 49 reviews for Chernobyl


In [92]:
rick_and_morty_reviews = [
    "This show is honestly great. It is easily the best animation show I have seen in a while on any network. The second the show starts you can tell that the characters fit perfectly with each other. A couple episodes in and I could tell that this was a unique show. With Dan Harmon writing and directing at the helm of it all, the magic that this show brings is a rare gem in television that people need to watch.",
    "Rick and Morty is a hilarious new show by the genius behind Community. The episodes have always seemed fresh and the writing is hilarious and creative. The hilarious mixture of wittiness, slapstick and action all add to making this show one of the best cartoons I have ever seen. The show easily parodies different movies and topics. I highly recommend this show to anyone.",
    "It's hilarious and original! My favorite animated series are Family Guy, Archer, Bojack Horsemen and this is right there with them. It's definitely weird but in a good way. You really become invested in its world and characters like it's a thrilling drama not an adult animated comedy. Each episode is funny, smart and weird. It's exactly what a science fiction comedy should be.",
    "This is an extremely solid show. It is dark but there are certainly moments that approach the limits of any sort of television style format. The strength of the show is just how amoral Rick may be. The entire show is just chock full of nuggets and neither preaches nor deigns to condescend to the audience. Pacing is always even and feels right. This might well be the best thing on television right now.",
    "This show is incredibly original with its writing, characters, style and imagination. Every episode AND joke is completely different from the last which keeps this show fresh and exciting to watch. The directing is great and the art style is really really good. This is truly a breath of fresh air when it comes to comedy.",
    "Rick and Morty might seem like some stupid knock off of Back to the Future but it's not. Behind all the jokes each episode surprisingly has a strong message. The absolutely clever writing and well designed character personalities are basically what this show floats upon. Unlike many other cartoons the sub plots in this show are actually relevant and memorable. This show is near perfect.",
    "The show was amazing in the first 2 seasons. Honestly at the time I think it was one of if not the best programs on TV. Mainly because of the fantastic writing, unique plots, trippy lo-fi feeling animation and it was funny. But now the show has lost sight of its characters, the jokes feel like they're rehashing ideas, and the whole atmosphere of the show has shifted. It doesn't feel like a show made out of love anymore, it feels like a corporate product.",
    "Rick and Morty is equally one of the most influential pieces of entertainment in the 2010s decade while also creating one of the most insufferable fan bases. The premise is one of the most distilled versions of Postmodern ideology. At its best this show goes off the rails with references to other pieces of culture. At its worst it's patronizing and self-referential.",
    "Rick and Morty is a very funny show. The humor is crazy and is hurled at the audience with breathless speed. What makes it a truly great series is the intelligence of the crazy sci-fi ideas that fuel the stories. Every idea is nonsensical but every idea is ingenious and fully explored. This is not just good comedy, it's good sci-fi.",
    "This may be the best Cartoon Network show period. The characters and stories in this show are just so freaking fun. Rick of course steals the show. What sets this apart from lesser shows like Family Guy is that this show works so much harder to make you laugh and doesn't try to just gross you out. It really does focus on character development.",
    "If you look at the timeline for the writers you can see all the writers from season 1-3 are gone. These new writers are very formulaic. Originality is gone. First three seasons are wonderful and you can watch them over and over. Season 4 and 5 are tolerable enough to watch once. I would strongly suggest they get back the old writers.",
    "I just finished watching the first three episodes and feel instantly hooked. I have the same feeling I had when I watched Family Guy for the first time. In this show things that will make you laugh are dark and awkward at times. I haven't found a single bad moment so far.",
    "Seasons 1-3 are fantastic: intelligent, funny, weird and deep. Brimming with original ideas. You'd wonder how Rick and Morty would save the day and be impressed by how things turned out. There's even strong moments of drama and pathos and a promising mythology build-up. Seasons 4-5 are a pale shadow - it's lost the intelligence, the wit and the depth. It's ditched the mythology and recycles ideas.",
    "Too much of the Adult Swim programming is mean-spirited and ugly but Rick and Morty balances its cynicism with moments of genuine familial and societal warmth. The characters are well-developed and evolve over the course of the season. There isn't a lot of TV I can watch with my son but this is just about the best of it.",
    "Currently this show seems to be very insecure about itself. It insists upon itself and thinks it's smarter than it actually is. The show seems to always complain about having to do serialized episodes and always takes a stab at the fans for caring about it. It's like it's afraid of being cliche and sincere and has to keep trying to prove how cool and smart it is.",
    "I find the Rick and Morty series good to watch in the background and you can jump in at any moment. I enjoy the creativity of the show as the premise follows Rick, a scientific genius who developed portal travel, and because of this anything can happen and the stories can run wild. One thing seen underlying in the show which is done very well is the character development and their interaction with one another.",
    "If I was to rate this show after the first 3 seasons alone then it would be somewhere between a 9 and a 10. Unfortunately everything started to go downhill after season 3. The thing I liked about the earlier seasons was the originality and the fact you actually cared about the characters. Unfortunately in the new seasons most episodes are filler. A lot of the jokes are lazy, repetitive and unfunny.",
    "My eleven year old son pushed me to watch this show with him. From the first minutes of the show I've been laughing my ass off. This show is brilliant. It is so funny but not just that! Every episode either has some kind of beautiful structured chaos or representation of some kind of ethical or philosophical theory and dilemma.",
    "One of the most unique animation shows of the last 10 plus years. This is absolutely one of those out-of-the-box quick-witted clever hilarious sometimes very weird and sometimes very dark sci-fi animation adult comedy shows that has graced late night screens in recent years. Like Family Guy or South Park this is absolute brilliant animation comedy that is meant to not only beg your attention but also because it has a deeper underlying message.",
    "I remember the days where me and my friends would all get together every single night a new episode was released. Filled with excitement we would see how the show would blow our minds and made us laugh until you get tears in your eyes. The later seasons are really not even worth watching anymore. The writers are exploiting the endless possibility multiverse too much.",
    "Rick and Morty is the most creative show I've ever seen. Each episode brings in new concepts which are explored in a humorous way or in a deep way most of the time it can even be both. With likable characters that you're invested in Rick and Morty seems to be a joyride of a TV show.",
    "Look, the first two seasons are some of the best television ever created seriously. And season 3 was pretty good albeit inconsistent. Then it just plummets off a cliff and dies on impact.",
    "This TV show is like the duo from Back to the Future mixed with the zaniness of Futurama mixed with the sheer insanity of Heavy Metal. There is so much going on in every episode and it is all very thought out.",
    "I don't watch adult cartoons, I used to in my early 20s but haven't in years. I picked this up in my early 30s and I love this show. The writing is unbelievably funny and the characters are equally hilarious. Do yourself a favor and watch.",
    "Rick and Morty is a wild ride that will have you hooked from start to finish. The writing is sharp and often takes unexpected turns keeping you on your toes and engaged throughout each episode. The dynamic between the two main characters, the eccentric scientist Rick and his good-hearted but easily influenced grandson Morty, is what really makes the show shine.",
    "I have watched seasons 1-4 of Rick and Morty countless times. It's got the perfect mix of dark comedy, movie references and emotional scenes whilst still leaving a lot unanswered. Until season 5, the new writers just fail to encapsulate what made Rick and Morty, Rick and Morty. All the jokes are stale and it really felt like they just ran out of ideas.",
    "Shows such as Futurama, The Simpsons and Archer were presumed to have set the bar for animated series in the modern era. Rick and Morty walked in with elements from all these great shows and simply decimated the competition. Every episode is a roller coaster and leaves you with laughs and thought provoking concepts.",
    "There is more intelligence in one episode of the unlikely duo than in many wannabe legitimate sci-fi shows or movies. This is the show that reconciled me with my geeky side. The best thing about the show is that it manages to reassemble pieces of deja vu plot and combine them with scientific concepts and make fresh material out of familiar tropes.",
    "Sadly Season 7 is a clear indicator that the team has run out of ideas. The show seems to be trying too hard but cannot hide the loss in creativity. Rick and Morty was once a masterpiece in animated television but this once nuanced and engaging show is now a cartoon for children apparently written by them too.",
    "The first 3 seasons are great. They're funny, irreverent and have a beautiful undercurrent of tragedy to make the comedy poignant. Rick's the smartest guy in the universe who loves science but is undercut by his emotional immaturity. Seasons 4-5 while having some great highs are very self-aware. The characters behave like they're in a TV show.",
    "Rick and Morty is a brilliant masterpiece that seamlessly blends sci-fi with comedy in a way that feels both groundbreaking and incredibly entertaining. The characters are superbly written and voiced. Rick Sanchez's mad genius and Morty Smith's endearing naivety create a dynamic that is both hilarious and heartwarming. Each episode is an unpredictable adventure filled with clever twists.",
    "Seasons 1-3 deserve any ounce of the praise they get as they make for some of the best animated television in recent years. The genuinely hilarious and clever humour is timed impeccably and balanced beautifully with poignant emotional elements. Seasons 4 and 5 were really disappointing with a lot of predictability and fatigued rehashing and the energy is noticeably slower.",
    "An amazing show! Probably one of my favorite animated shows. Very good story with hilarious atmosphere and funny cultural tropes. It's surprisingly quiet thematic at times for a comedy and has wonderful and creative ideas that are both wacky and genuine. The characters are well written and somewhat deep.",
    "This is one of the best and most iconic series of our times. The main cast is interesting and well portrayed. You will not find any pattern regarding the plot of each episode - each and every one of them are unique, creative and mind blowing science fiction wise. The Evil Morty episodes are some of my favorites capturing the silent rise of what appears to be an evil intelligent commander.",
    "This show has no limits. It's creative, fun and imaginative. All the characters are likable and relatable. The animation is really good and the show can have its emotional moments. Thanks to all the crazy ideas this show never feels repetitive. The fact that it can show tons of insane concepts and seem realistic really shows how genius the writers are.",
]

add_show("Rick and Morty", rick_and_morty_reviews)

✅ Rick and Morty predicted as: Balanced
✅ Added 35 reviews for Rick and Morty


In [93]:
bleach_tybw_reviews = [
    "Words aren't enough to describe this show. You get everything - the visuals, fight scenes, story arc - everything is on an insane level and the CGI effects are next to impossible. You will become a true fan after this. So far it is the best anime of 2022 in terms of story and everything.",
    "Been a long time Bleach fan, first saw it on Adult Swim years ago. It was the very first Japanese animation I watched and eventually fell in love with. So much detail in the animation, top of the line. The CGI hollows were great. Had to rewatch the first episode 3 times in a row to take it all in.",
    "Man, they said Bleach was mid, they said Bleach was never coming back, but now I bet most people are eating their own words because this show is incredible. Not only did this episode have godlike animation but the music and voice acting was beautiful. I was not expecting it to be this good. This is looking like it's going to be one of the best animes of the year.",
    "Amazing. One of the best anime ever. Like I have never seen such peak fiction in my eyes and heart. It's one of the best shonen ever made with cool designs, powers and plot. The war arc is insane! The comeback episode is the best. I had to rewatch it and will be watching it again. I cannot wait until the next episode airs because this right here is peak fiction.",
    "The art style, the music, the animation is fantastic. You can't ask for anything better than this. The first episode has completely blown my expectations. So gorgeous I almost cried. There is a reason Bleach is in the big 3. The animation and sound design are movie scale. I recommend watching Bleach from the beginning to catch up with Thousand Year Blood War - it is worth it.",
    "The first two episodes have been amazing - the animation has improved, the storyline is gripping, and the music has more depth. Tite Kubo is a master storyteller, thank goodness he was involved in the making of the anime. For those of you who know next to nothing about Bleach, watch the previous seasons quickly so you can sooner enjoy the TYBW arc!",
    "I started watching Bleach only a year and a half ago and it was the last in the big 3 for me to watch. The first episode was incredible. When Ichigo's song came on I jumped out of my seat I got so excited. I can't imagine waiting over a decade for this show but if I had watched it back then I would have waited a decade.",
    "Finally we get the rewards of our wait and patience. The animation is top notch and unreal. The plot is fire, it is one of the best plots and arcs among all the anime. You will clearly see that Kubo was directly involved as promised. The anime is uncensored. The voice actors really did a good job. Animation is movie quality.",
    "It's basically everything you hoped for. But do they really have to overdo the I have one hidden power left to the extreme? You knew one of them would win but they dragged it out to the extreme. One fight is basically between 6 to 10 you thought you won but I have this extra hidden power moments and it goes like that over and over.",
    "Despite impressive visuals and the long-awaited return of beloved characters, Bleach's new arc suffers from serious narrative flaws. Unjustified power escalation - antagonists showcase increasingly powerful abilities without any logical explanation. Fights follow a monotonous pattern. Positive aspects are the stunning animation, long-anticipated Bankai reveals, and high-quality visual design. The series clearly prioritizes spectacle and fan service while sacrificing narrative logic.",
    "I don't post reviews often but I felt I had to for this one. Bleach was what got me into anime. When Ichigo finally shows up in the episode the emotion I felt surprised me. I felt happiness, excitement, joy and nostalgia. I am excited for what's to come. Bleach fans deserve an ending and I can tell from this episode it's going to be as epic as I always thought it would be.",
    "I really liked how the studio was able to develop from drawing and animation to another level that serves the amazing story. The intro and outro music is fantastic, I thought I was watching a movie not a series. I watched the episode more than once because of my great admiration for this masterpiece.",
    "This is what I have been waiting for. I previously watched the original Bleach series and as a comparison I am loving this show a lot more. Visuals are amazing, combat is great, voice acting is still S tier and the story is compelling. The only negative I have is the requirement to watch the original series if you're a new viewer.",
    "I haven't been this excited for a Bleach episode since I was a kid. The joy I felt seeing the first episode was palpable, the animation was gorgeous, the music nostalgic but slightly different. The end song for the first episode brought back so many memories. I now want to go and rewatch everything from before.",
    "This is the greatest anime arc in history. Every character develops so well. Animation is awesome. Fight scenes legendary. New main antagonist is damn powerful. He is not only the strongest villain in Bleach but one of the strongest villains in anime history. This anime arc is my personal favourite and is a must watch.",
    "For many years Bleach was slandered and hated blindly by many people. While Bleach has its faults it's a genuinely good story. Now those same people are saying Bleach has relied on animation. Regardless it's a magnificent return for old and new Bleach fans. The story, the character details, the animation, the music and the obvious care put into the show makes me love the TYBW show.",
    "The story started off really good, so good that it secured a top spot in many sites. But then the story progression kind of stops when the fights started. The fights feel extremely dragged out. Story progression is almost absent and fights are extremely predictable. I hope the upcoming episodes will have some emphasis given to story as well.",
    "Very poor direction. Doesn't even feel like Bleach anymore. Season 1 was ok, season 2 was not that great, but now that they've fully gone the original direction with season 3, the drop in quality is drastic. They're moving the plot way too fast and not letting the scenes breathe. The greatest flaw is the direction. This feels like any of the generic cookie cutter anime that have flooded the industry these days.",
    "Thousand-Year Blood War is a stunning adaptation that improves on nearly every aspect of the manga. The animation is top-tier with fluid combat and vibrant color palettes. Scenes that felt rushed in the manga are now expanded with added context, emotion and atmosphere. The soundtrack is phenomenal. Character arcs especially Ichigo's are given more depth. TYBW doesn't just adapt the source material, it refines it into something that feels definitive.",
    "This arc in particular saved my life. From a suicidal guy with no hope I was able to find myself and have the courage to fight despite my fears. The anime adaptation is simply the best. It is clear to see the staff loves Bleach. The love in the animation, movements and storytelling is very visible. I have never seen an anime feel so alive.",
    "Bleach TYBW gives an insanely amazing comeback to a long ended shonen anime. Starting from the characters to the animation, Bleach TYBW improves a lot. The final arc is a serious and bloody one with a lot of fights. The characters became more bold and aggressive and the action sequences were totally heavenly. It feels nostalgic and amazing to see an action battle shonen anime back with its most intense arc.",
    "After years of waiting, Bleach's Thousand-Year Blood War arc was supposed to be the grand return that finally tied up all the loose ends. Instead it feels like more of the same problems. The pacing is inconsistent with drawn-out battles that lack emotional weight. Despite the high stakes there's a lack of real tension. The animation has improved but flashy visuals can't cover up the weak storytelling.",
    "Big three is running strong now that Bleach is back. There will be many heartfelt moments and blood shed all around. Growing up with Bleach was one of the best experiences ever. Now that I'm an adult I'm proud to say the magic is still alive as I watch it with my daughter.",
    "My initial opinion of season 1 was to award an 8 rating based on the potential story growth. However after viewing season 2 I had no choice but to bring the rating down. There was no story growth, no individual character growth. There was nothing but solid fight scenes for all 13 of the second season episodes with nothing furthering any part of the story. You could have dropped watching every single episode of season 2 and missed nothing.",
    "Bleach: Thousand-Year Blood War is a monumental achievement that captivates and surpasses all expectations. This adaptation flawlessly brings Tite Kubo's visionary masterpiece to life. With flawless animation, impeccable storytelling and intense action, it unquestionably earns a perfect score. The storytelling dives headfirst into the final arc of Bleach masterfully capturing the essence of its complex narrative. The pacing is spot-on ensuring each episode leaves you eagerly craving more.",
    "The Thousand-Year Blood War arc raises the stakes significantly by introducing the Quincy, a powerful and long-lost enemy group. The arc delves deep into character development, providing significant growth for both the main protagonist Ichigo Kurosaki and several other key characters. The arc is filled with intense and epic battles. It ties up many loose ends, reveals hidden secrets and provides closure for various character arcs.",
    "After 10 years of sabbatical the highly anticipated conclusion of the Bleach anime has finally begun. Seeing Bleach in such modern animation style and direction is such a breath of fresh air. All in all a 10/10 first episode. Most animes would kill to have it and nobody deserves it more than the faithful Bleach fans who kept their belief that their beloved series would return.",
    "I write this after just watching the last episode of Cour 2. The reveals in this season have been amazing to say the very least with stellar voice acting performances from all in it and an astounding set of visuals in nearly every episode.",
    "Each episode of Bleach Thousand Year Blood War just keeps getting better and better. You can see from the story side how they represent each character's background without fillers. The masterpiece of the soundtrack and music just fits each moment and character. The producers and studio have really learned from the original Bleach filler mistakes.",
    "I am extremely excited and happy because all the episodes are meeting my expectations. From the first episode the series has made it clear that this time it will not be as soft as before and will be as brutal as possible. The color palettes chosen are very well suited to the ruthlessness and theme of the series.",
    "Ohh boy was I hyped for the return of the anime. The first episode got me on the edge of my seat with the animation which was high quality and we didn't expect that level of quality. It was everything a Bleach fan could ask for. Throughout the first part the quality of animation kept getting better. One of the greatest plot twists in fiction was animated and it was done justice.",
    "Anime doesn't get better than this. The show is SO good I finally created an IMDB account just to rate it. For anyone that thinks it's anything less I'd love to hear their opinion. The show looks like it could peak right from the start. Get your popcorn, turn off your lights, sit back and enjoy the most masterful creation of anime out today.",
]

add_show("Bleach: Thousand-Year Blood War", bleach_tybw_reviews)

✅ Bleach: Thousand-Year Blood War predicted as: Balanced
✅ Added 32 reviews for Bleach: Thousand-Year Blood War


In [94]:
tmnt_2012_reviews = [
    "Voice acting is great, animation is beautiful, story line is well thought out and interesting. The turtles personalities are true to the source, they still have the brotherly bonding, it's funny enough for children. Just overall great. I love this show and I'm 25.",
    "It was indeed a surprise for me when I watched the season premiere. I was a little skeptical and scared to watch as one of my favorite childhood memories could be ruined again. I was so wrong. This proved not only to be a worthy watch but added some more elements to the individuality of the characters. This is indeed a worthy and more than expected successor to the 80s super-hit show.",
    "TMNT being my favorite childhood series I didn't think I could ever see that magic again. Nickelodeon has surpassed that with this new series. The personalities with each turtle go much deeper than the simple catch phrases and slapstick personas we grew up with, yet maintains it with a fresh new approach. The humor in this show is undoubtedly to be enjoyed by both children and adults.",
    "I've held off watching this new show for a long time afraid it'd damage the overall feeling I have for TMNT. What a wonderful surprise and adventure this show has become. The new designs took a place in my heart after just one episode and the action and animation does not disappoint at all. It's the writing of this show that has sealed the deal for me. Each turtle's personality comes across flawlessly.",
    "Wow what a shock this show is. I was surprised at how well the characters are written out and how they interact with each other. When the turtles had their first fight they had problems fighting alongside each other and it was refreshing to see that these turtles are very flawed but organic and have a lot of room for growth.",
    "The 2012 iteration of Teenage Mutant Ninja Turtles revitalizes the beloved franchise with a winning combination of action, humor and nostalgia. The character design, animation quality and voice performances breathe life into Leonardo, Michelangelo, Donatello and Raphael, ensuring that each turtle's distinct personality shines through. The writing smartly infuses humor that appeals to both younger viewers and longtime fans.",
    "This form of the TMNT is great. Nick has taken the ideas and applied them for a new generation to enjoy. Deep down the core of the story is still the same 4 brothers being a team and kicking Shredder's butt. But it is more than that - it is also about family and applying the things they learn from Splinter in everyday life.",
    "I love everything about this show. It was my childhood and to this day I still think it's one of the best shows I have ever seen. The finale of every season is absolutely beautiful especially the Season 2 and 4 finales. Both the Splinter and Shredder of this series are so cool and badass and the action of this show is absolutely phenomenal.",
    "This show is easily one of the best reincarnations of the turtles. It takes a good amount from the 80s series while also bringing in some elements from the 03 series. The show is enjoyable for all ages having its comedic moments for the younger viewers while also getting serious and providing an interesting plot that keeps you watching.",
    "Having grown up with both previous cartoons and the three live action movies I've gotta say this is so far the best thing that bears the novelty of the Ninja Turtles. All the classic characters are represented very well. For the first time the character of April O'Neill is actually believable.",
    "This is one of the best TMNT incarnations since the comics! The plot, the script, the characters, everything is great. The way they are different between each other, story has humor and seriousness at the same time. Every character is well written and the story develops really well.",
    "My dad and I watched this show together when I was younger. He ended up passing away before the show ended and I didn't have the strength to finish it for a few years. Then my little brother decided he wanted to watch it with me and I finally finished it with him. This show is fantastic! The character development is phenomenal and the story telling is very fun. For a kids show it gets very deep and emotional at times.",
    "I was never into the TMNT when I was little but when I saw previews of this show it looked funny and interesting to me so I gave it a shot. It was SO worth it. Love the story line and frequent humor and you really get to understand each character. There are a couple episodes that make me laugh for a good 10 minutes every single time I watch them.",
    "I am a huge fan of the TMNT franchise and I have three brothers of my own and the four of us exemplify the Turtles' personalities. The fantastic creative team behind this computer animated series has established what is in my opinion the best rendering of the Turtles to date. The first real nemesis becomes the Kraang and it's through their battle with the Kraang that the Turtles encounter April O'Neil.",
    "This is a love letter to everyone who grew up with the Turtles. There are references and homages to previous incarnations in every episode. It brings back fan favorite characters and also characters people forgot about. It's not afraid to bring in some Japanese culture. As for the returning characters they are more developed than ever.",
    "Season 1 is filled with some pretty fun episodes to watch. Season 2 has more character development as well as some new faces. I have rewatched this show many times. There are some aspects of the show that just feel a bit lifeless but I do really like the turtles designs and I love how the intro changes slightly with each season. The VAs did a wonderful job and matched their characters perfectly.",
    "I was wary of watching initially but I think there is something here for both older and newer fans. It's a decent modernisation that contains elements of the early cartoons and the graphic novels while also being its own thing. The series offers good humour and action. The series can be surprisingly dark and violent for a kids show.",
    "I remember waking up early just to watch this show. I was a fan of the very old show but along the way the show lost its stand totally. When I heard of this upcoming series I thought it was going to be a failure like the recent ones but I was wrong. The art style has certainly improved making it much more attractive and I really like the plot of the origin of the turtles.",
    "My 4 year old son got me watching this and after the first episode I was a TMNT fan! Very well written, better than any cartoon I have ever watched. The Network obviously put a lot of time and energy into the characters and story lines. Each episode is like a short movie.",
    "I'll admit it, I never watched the 1980s Ninja Turtles show. When I heard of this new show I face palmed. But the season 2 finale came and I watched it and took everything I said back. This isn't just a hip and modern remake, this is a love letter to everyone who grew up with the Turtles. It's not afraid to bring in some Japanese culture. The new characters are also really enjoyable.",
    "I am a huge fan of animation and when my kids first started watching this show I sat off to the side and huffed that this was not the Turtles I first loved. But once they got me into it I found myself enjoying the Turtles more than ever before. The creators have shown themselves to be excellent story-tellers.",
    "Freaking love this show. Everything is top notch. I've been around since the Turtle comics first came out. This show is a tribute to the original Turtles story line. The animation is beautiful. The storyline and writing is excellent. It is oftentimes freaking hilarious.",
    "This is the incarnation of the TMNT that I grew up with. I loved nearly every second of this show. Leonardo struggles to find his voice as a leader and goes through a lot living up to what people expect from him. Raphael has a lot more to explore beyond his temper. Donatello shows how even when you're a genius he doesn't know everything. Michelangelo is still the silliest member of the team but shows you can't stereotype him.",
    "Nearly everything about this show is just perfect. The way Leonardo starts off as a nerd who overtime learns to be a heroic leader rather than starting off as one is great. The fight scenes are fantastic and are some of the best I have ever seen in any TV show. Shredder in this series has to be in my top 10 villains of all time. Every time he is on screen he radiates intimidation.",
    "I absolutely loved it! A definite recommendation to people that loved the old show. I loved Splinter's character, he says many inspirational things. Raph is a hot head but he's very sensitive. Leo has good leadership skills. Donnie is a total genius. Mikey is fun and great at naming stuff. It's the best version of the turtles.",
    "I seriously love this show so much. Season 1 was a great start to the series showing the potential it had. Season 2 is where the show gets really good for me, the plots are really engaging and the characters start to really shine. The dynamics between Leo, Raph, Mikey and Donnie started to feel really great. I could write a whole other review to say how much I love Mikey in this series.",
    "This was the show that introduced me to the franchise. Most of the characters are pretty likable and most of the voice actors fit the characters really well. The writing is pretty good and consistent. While I love this incarnation there are a couple of things that bug me about it but this version alongside Rise of the TMNT deserved more love while they were still airing.",
    "Ciro Nieli and everyone involved with this show gets everything right about TMNT. I'm a long-time Ninja Turtles fan and can safely say that this is the greatest incarnation by far! Everything from the exhilarating action scenes to the comedy is spot-on and the interactions between the Turtles is just fantastic.",
    "Best TMNT show ever made. Perfect mix of serious storytelling, fun stories, character development and perfect voice casting. Hoon Lee brings such a well-roundedness to Splinter and the right amount of gravitas. The turtles go through happy and tough times and learn and grow.",
    "Awesome reboot from an original kids show. It even packs crossover episodes with the old show. This show also knows how to mix comedy with drama into a perfect blend making it a show with a little bit for everyone. Great show I can watch it for hours.",
]

add_show("Teenage Mutant Ninja Turtles (2012)", tmnt_2012_reviews)

✅ Teenage Mutant Ninja Turtles (2012) predicted as: Balanced
✅ Added 30 reviews for Teenage Mutant Ninja Turtles (2012)


In [95]:
power_reviews = [
    "The second they started focusing on Tariq the show took a nosedive. Why make the spoiled son the focus of the show I'll never understand. I wish I had stopped watching after season 3.",
    "I absolutely loved Power. Every season just kept stepping up their game. The best part of the show is the relationship between Ghost and Tommy. Tommy steals every scene he's in. The show follows Ghost, a man living a double life of leading a drug empire and owning one of the most successful night clubs in the city. Some of the writing in this show is unbelievable but that doesn't take away from the fact that the show is fantastic.",
    "Power is a great edge of your seat thriller that will keep you entertained throughout the entire series. The acting and the writing were so good that the show just sucked you into their world. The character development is as good as any show on television. Even though these are criminals you're watching you really start to care for them and root for them. This show really takes off on season two.",
    "People can say what they want about 50 cent but he did a great job with this show. Power looks genuine and the story is very appealing. The cast in the show are marvelous especially Omari Hardwick. The soundtrack is good too. There are too many bad series on TV and this show would fit TV anywhere.",
    "The rise in Starz ratings began with Power. Ghost was a driven individual who desired to be more than a drug lord and it seemed that everyone he encountered stood in the way of his desire to be legitimate. My favorite was Tommy. He was funny, straightforward, family-oriented and loyal. I'm relieved they're doing a spin-off about him since I didn't want his story to end.",
    "Power Series 1-3 is impressive! James St. Patrick is interestingly, undeniably a confident nightclub owner with so much on his weight and there's so much more to go on - family, love affairs, friendship betrayals, it's all here! Trust is a dangerous weapon in the world of Power.",
    "Plot and story line fine but it is a bit useless to have the sex scene every 15 minutes. Yes people have sex but do not ruin the show with all the overrated sex scenes.",
    "This is an incredible creation of a show. The show follows the life of a rich drug dealer desperately trying to get out of the game. But without the support of his friends or family he has no way to do so.",
    "Really enjoyed the show. Great story arcs, acting and characters. Really makes you binge watch it. But the finale was a total disappointment. I understand they did what they did to create spinoffs but I'm sure these creative minds could have come up with a better way.",
    "Starts off great, I really enjoyed watching series 1, 2 and 3, however the story soon becomes monotonous and predictable and simply more of the same. Had massive potential but poor writing has killed it.",
    "This show was genuinely excellent in its opening season, dark, gritty, relatively credible and low-key. It was stretched a bit in the follow up seasons but remained a great watch with good drama and some excellent characters. Sometime around season 4 they seemed to have fired the writing team and it just turned into utter drivel.",
    "Power is a great show that seems to invite the viewer to watch more and more of it. I couldn't just stop at one episode, I had to watch them one after another. 50 Cent has done an excellent job in making this show not just good but great. I'm surprised this show doesn't have a higher rating because I believe it deserves it.",
    "This show presented Ghost as a criminal mastermind who had aspirations to become legit. At every turn Ghost would think 10 moves ahead, from season 1 till season 6 episode 10 everything was written perfectly. After episode 10 it felt like I was watching a parody of Power and nothing made sense.",
    "I have enjoyed Power since day one. It's been a thrilling ride with a lot of intrigues, drama and action. The storyline has been exciting and fun the whole time, always leaving the audience wanting more. The writing is getting lazy but I would recommend this show anytime.",
    "Overall this show was great! But they ruined the ending just as bad as the Game of Thrones ending. The last episode has the lowest rating in the history of the entire show.",
    "Up until 3 weeks ago I hadn't known anything about this series. It was my best decision since Breaking Bad. Both the acting and the script are superior, the story takes place in NYC which I love, so I totally got hooked. I binge watched 6 seasons in 3 weeks and I wish there was more.",
    "Power is a gripping dark crime drama and thriller that centers around James Ghost St. Patrick and his attempt to build a criminal legacy while maintaining a legit facade as one of the most renowned night club owners on the eastern seaboard. Power weaves a complex narrative throughout its seasons and it's one that never fails to keep the viewer entertained or surprised.",
    "If you like drama, sex, drugs and gangsta shows then this one's for you! It has so many twists and turns. The cast is perfect, I love this show.",
    "I started to watch Power because of my boyfriend and I was skeptical. I can't compare this series with any series I have seen before. Drama, love, money, drugs and power. These five nouns explain exactly what the story line is about and all of them are perfectly balanced. It forces you to see the rough world outside and how people just try to manage their life within sophisticated circumstances.",
    "First 3 seasons were very impressive but after that it becomes so boring. The story soon becomes monotonous and predictable.",
    "High action, high drama, well acted, very gripping, keeps you on the edge of your seat, full blown thriller. Definitely the best show I've ever seen on Starz. In the upper echelon of premium television shows.",
    "Not my usual show at all. Really good, exciting story line with quick and unexpected twists! Really enjoying it! Would recommend not to read the description as it didn't do justice at all, just watch open minded and it won't disappoint.",
    "The series caught me right from the start. The characters and the storyline simply fantastic. Episode after episode, exciting season after season until the last minute of the finale which I had absolutely not expected. Tommy and Ghost, a fantastic team.",
    "Amazing show from the first episode right through. Likeable characters, storylines that are top notch and complex, enjoyable right through.",
    "This is a compelling show that will draw you in no matter your age or race. It has all the right elements you want in television to keep you coming back. The cast is phenomenal as well as the acting. I was hooked from episode one.",
    "I enjoyed the show but I really think the writing let it down at the end. The ending was dragged out by different scenarios and the same scenes were different in each episode which was a bit messy. All in all a good show with some good characters and storylines but ultimately defined by a disappointing ending.",
    "After season two it turns into the walking dead in the sense that they rush through plots. As soon as they solve their problem another antagonist that was never introduced is now involved, ruining Ghost's life.",
    "This show is fantastic. There's never a dull moment, edge of the seat stuff all the way through. Full of game changers and heart stopping thrills. Power solidifies all the elements of a perfectly cast, powerfully scripted, great show.",
    "Everything Curtis Jackson touches turns to gold. Between the directing, editing, music, casting and acting this show is consistently riveting and absolutely captivating. One of the best dramas currently on TV or in a very long time. The sub plots and story arcs are always engaging from week to week.",
    "At the beginning Power didn't really catch me up but the progress made me a huge fan of the series! Every season is better than the previous one. The show is still intriguing and thrilling. The over-nudity is annoying though.",
    "Power is a series that tells the tale of James Ghost St. Patrick who had a troubled life and has made it and is now a club owner trying to get out of the illegal ways of making money. The storyline for this picture is very solid and well delivered. The writing is clever and more intricate than you'd expect though sometimes it gets painful and unrealistic.",
    "This show is worth watching and entertaining. It is also very easy to root for the characters even if the characters' decisions are bad. The last 2 to 3 seasons were not as good as the first couple but that happens with most shows.",
    "Season 1 is a bit slow but with every episode it gets better and better.",
    "Good show! Love the characters and how each one makes you love and hate them at the same time. Can be a little unrealistic but all in all a good show.",
    "It's a great show full of action and intrigue. Great acting cast. The show is pretty much realistic as far as the game goes. To a degree some of us can somewhat relate to the lifestyle choices some of the key players have made.",
]

add_show("Power", power_reviews)

✅ Power predicted as: Balanced
✅ Added 35 reviews for Power


In [96]:
black_lightning_reviews = [
    "Good premise and potential, and some good acting and special effects. Conversely, the massive cheese factor in the writing combined with some equally bad acting make it impossible to really get involved in the show. I truly love this show now after they fixed everything that was wrong, but sadly the formula for all these DC shows of cheese, camp and cringe remains a work in progress.",
    "Black Lightning's first episode was super cool and I see the potential for it to get even better. There was nothing over the top to a viewer with a mature and realistic awareness to today's American culture. The cast was perfect. The story will only get better.",
    "Being that only one episode has aired I'm not going to automatically say this show is destined to fail as I have high hopes for Black Lightning. I will say the first episode somewhat screamed victimization as though it was trying too hard. Had this been spread out over a few episodes I think it would have been depicted better.",
    "Black Lightning has kept me watching. It loses points for season 4 which I feel started badly though it picked up in later episodes. Super villain Tobias Whale is a very memorable character throughout the series and his evil chuckling fits well with his psychopathic nature. On the whole Black Lightning is an enjoyable superhero drama.",
    "It's about time that the CW aired a show that relates to the black community. It's refreshing to see a show about black heroism, black family values and black romance. This show is somewhat very true to life in the African American school system. Black Lightning is kinda cheesy but overall good.",
    "This is a slick, well done, unique take on the superhero genre. It deals with modern day issues, speaks in a contemporary language, and shows evil from a perspective very rarely attended to in superhero storytelling. It manages both camp and realism.",
    "This is NOT as bad as some reviewers have made out. It is fairly entertaining and does have some nice plot twists. The acting is generally good. Overall a good series, it IS entertaining even if the writers have made Black Lightning into a wimp without the conviction of his principles. Lightning and Thunder, the two daughters, are much better heroes.",
    "It looks like most of the negative reviews here are all politically motivated. This is a slick well done unique take on the superhero genre. It deals with modern day issues, speaks in a contemporary language, and shows evil from a perspective very rarely attended to in superhero storytelling.",
    "BAD. I am a black female who tried very hard to like this show. Way too many black issues all in the first episode. The acting was borderline community playhouse level. Once Black Lightning put on that tacky costume I was lost. The fight scenes are terribly choreographed.",
    "I had to log in to write a review for the very first time to express how delighted I am in Black Lightning. Full of game changers and heart stopping thrills it solidifies all the elements of a perfectly cast powerfully scripted great show. I hope this show has a very long run.",
    "Black Lightning Episode 1 gave me chills, thrills and a realism that most TV just does not have these days. It is current, topical, political and has all the elements of a great superhero series. Teenage social media and smartphone addiction, gangs of disaffected youth in a society that doesn't care. This show deserves a 9/10.",
    "I'm going to rip this off like a band-aid. The script was lousy, the filming was lousy, the editing was terrible. I loved the cast and this show had such potential. Such potential yet such a mess.",
    "I tried to be loyal because at least I am represented as a black person on screen. But I didn't like the characters. Some were just jarring. The acting often seemed wooden. It aims for grittiness but it comes across as some very disinterested actors earning a quick buck.",
    "First episode in and this is first rate compared to the recent oversaturated superhero show landscape. Not too campy, cartoonish or stereotypical but rather quite relatable. The acting is well done with no cringe worthy excess. They are compelling characters, a well balanced storyline and good actors to boot.",
    "Black Lightning is not just about race. It's about family and coming of age and aging and the gridlock of being powerful but conscious of that power's repercussions. Filmed beautifully, it's visually captivating and so easy to get into. It's entertaining, it's engaging, and it's attempting to tackle a highly charged subject.",
    "I enjoyed the show but it is not the best show out there. Hollywood is fairly new to the whole diversity thing so it's interesting to look at how they take on issues as sensitive as police brutality, racism ingrained in the justice system and gang violence. The story line seems uncertain and predictable.",
    "I watched all of Season 1 and enjoyed it but I literally can't remember any of it. I remember enjoying the show but it has no lasting effect in any way and has a very forgettable story.",
    "The concept of a retired hero putting back on the suit is an interesting one. Despite that very interesting concept the remainder of the show was quite mediocre. The acting was decent and the plot was predictable.",
    "I enjoy the show. Black Lightning is not the main character of the show just in name alone. His journey was in season 1 and now he is just there in a show that should end.",
    "I'm up to season 3 and I kind of like it. If you treat it like a visual and audio representation of a comic book then it's exactly right.",
    "The show could definitely be so much better but as of yet I haven't seen anything very eye-catching and the story line seems uncertain and predictable.",
    "There are a lot of reasons this show stands out. It's story is centered around the daily and life-long experiences of people of colour in America. It provides us with the depiction of a family that loves and cares for each other. The title character is a new kind of superhero, not just because he's a proud black man, but because he's older and world weary.",
    "I'm noticing a lot of negative reviews. This isn't a terrible series but the poor acting and writing and corny costumes can make you sometimes feel you're watching an episode of Power Rangers. I still like it.",
    "I love this series. Black Lightning tells of a black principal superhero who harnesses electricity to stun his foes. The bad guy is intriguing and scary. I love this show and I'm sure any modern superhero flick fan will enjoy it too.",
    "This show is a new favorite of mine. It has the suspense, grit and sense of humor that I love from the DC universe. It also has one of the most realistic portrayals of black family life that I have seen on TV.",
    "The CW brings on a black superhero TV show. It starts with an interesting premise. It's a middle class black family trying to protect an oasis of civility within a tough black community. It still tackles the growing BLM and other racial issues.",
    "I was excited for a new superhero show but the more I watched the show the worse it became until I gave up on it. The action is horrific. The story line is long drawn out and unappealing. The main focus of the show is black issues every chance they get.",
    "This show is a beautiful marriage between DC and CW. It captures the realms of black fatherhood, the struggles of the black community, sexuality and feminism. The actors do a great job of portraying the characters and the writing is good.",
    "The show is not perfect granted but it is not that bad. It is entertaining, the acting gets somewhat better in time. I only dislike some editing choices. It really is not as bad as people say.",
    "The first episodes were amazing but from there it is somewhat losing its steam. I think they are trying to have so many story lines so early to the point where it feels rushed.",
    "I watched this show because it looked interesting and I love a lot of superhero tv shows. The acting feels forced and hard to ignore. A lot of the characters are stereotypical with next to nothing unique about them.",
    "I had hopes this show would get better but it turned out just as bad as the first season. When compared to the ratings from all the other superhero shows it's on the bottom.",
    "I liked the show for the first season and part of the second but the writers are not so good. People who get killed are constantly brought back to life which is typical CW nonsense and the characters constantly make bad decisions.",
    "Bad acting, bad stories, bad everything. I watched the entire first season hoping things would get better but they didn't. It wants to be a drama, it wants to be action, it wants to be a superhero show but it isn't any of that.",
    "This show is entertaining in an uncomfortable way. It's so different from other superhero shows. It's GOOD in unfamiliar ways. This is several cuts above the emotional stupidity of most super stories.",
]

add_show("Black Lightning", black_lightning_reviews)

✅ Black Lightning predicted as: Balanced
✅ Added 35 reviews for Black Lightning


In [97]:
black_lightning_reviews = [
    "Good premise and potential, and some good acting and special effects. Conversely, the massive cheese factor in the writing combined with some equally bad acting make it impossible to really get involved in the show. I truly love this show now after they fixed everything that was wrong, but sadly the formula for all these DC shows of cheese, camp and cringe remains a work in progress.",
    "Black Lightning's first episode was super cool and I see the potential for it to get even better. There was nothing over the top to a viewer with a mature and realistic awareness to today's American culture. The cast was perfect. The story will only get better.",
    "Being that only one episode has aired I'm not going to automatically say this show is destined to fail as I have high hopes for Black Lightning. I will say the first episode somewhat screamed victimization as though it was trying too hard. Had this been spread out over a few episodes I think it would have been depicted better.",
    "Black Lightning has kept me watching. It loses points for season 4 which I feel started badly though it picked up in later episodes. Super villain Tobias Whale is a very memorable character throughout the series and his evil chuckling fits well with his psychopathic nature. On the whole Black Lightning is an enjoyable superhero drama.",
    "It's about time that the CW aired a show that relates to the black community. It's refreshing to see a show about black heroism, black family values and black romance. This show is somewhat very true to life in the African American school system. Black Lightning is kinda cheesy but overall good.",
    "This is a slick, well done, unique take on the superhero genre. It deals with modern day issues, speaks in a contemporary language, and shows evil from a perspective very rarely attended to in superhero storytelling. It manages both camp and realism.",
    "This is NOT as bad as some reviewers have made out. It is fairly entertaining and does have some nice plot twists. The acting is generally good. Overall a good series, it IS entertaining even if the writers have made Black Lightning into a wimp without the conviction of his principles. Lightning and Thunder, the two daughters, are much better heroes.",
    "It looks like most of the negative reviews here are all politically motivated. This is a slick well done unique take on the superhero genre. It deals with modern day issues, speaks in a contemporary language, and shows evil from a perspective very rarely attended to in superhero storytelling.",
    "BAD. I am a black female who tried very hard to like this show. Way too many black issues all in the first episode. The acting was borderline community playhouse level. Once Black Lightning put on that tacky costume I was lost. The fight scenes are terribly choreographed.",
    "I had to log in to write a review for the very first time to express how delighted I am in Black Lightning. Full of game changers and heart stopping thrills it solidifies all the elements of a perfectly cast powerfully scripted great show. I hope this show has a very long run.",
    "Black Lightning Episode 1 gave me chills, thrills and a realism that most TV just does not have these days. It is current, topical, political and has all the elements of a great superhero series. Teenage social media and smartphone addiction, gangs of disaffected youth in a society that doesn't care. This show deserves a 9/10.",
    "I'm going to rip this off like a band-aid. The script was lousy, the filming was lousy, the editing was terrible. I loved the cast and this show had such potential. Such potential yet such a mess.",
    "I tried to be loyal because at least I am represented as a black person on screen. But I didn't like the characters. Some were just jarring. The acting often seemed wooden. It aims for grittiness but it comes across as some very disinterested actors earning a quick buck.",
    "First episode in and this is first rate compared to the recent oversaturated superhero show landscape. Not too campy, cartoonish or stereotypical but rather quite relatable. The acting is well done with no cringe worthy excess. They are compelling characters, a well balanced storyline and good actors to boot.",
    "Black Lightning is not just about race. It's about family and coming of age and aging and the gridlock of being powerful but conscious of that power's repercussions. Filmed beautifully, it's visually captivating and so easy to get into. It's entertaining, it's engaging, and it's attempting to tackle a highly charged subject.",
    "I enjoyed the show but it is not the best show out there. Hollywood is fairly new to the whole diversity thing so it's interesting to look at how they take on issues as sensitive as police brutality, racism ingrained in the justice system and gang violence. The story line seems uncertain and predictable.",
    "I watched all of Season 1 and enjoyed it but I literally can't remember any of it. I remember enjoying the show but it has no lasting effect in any way and has a very forgettable story.",
    "The concept of a retired hero putting back on the suit is an interesting one. Despite that very interesting concept the remainder of the show was quite mediocre. The acting was decent and the plot was predictable.",
    "I enjoy the show. Black Lightning is not the main character of the show just in name alone. His journey was in season 1 and now he is just there in a show that should end.",
    "I'm up to season 3 and I kind of like it. If you treat it like a visual and audio representation of a comic book then it's exactly right.",
    "The show could definitely be so much better but as of yet I haven't seen anything very eye-catching and the story line seems uncertain and predictable.",
    "There are a lot of reasons this show stands out. It's story is centered around the daily and life-long experiences of people of colour in America. It provides us with the depiction of a family that loves and cares for each other. The title character is a new kind of superhero, not just because he's a proud black man, but because he's older and world weary.",
    "I'm noticing a lot of negative reviews. This isn't a terrible series but the poor acting and writing and corny costumes can make you sometimes feel you're watching an episode of Power Rangers. I still like it.",
    "I love this series. Black Lightning tells of a black principal superhero who harnesses electricity to stun his foes. The bad guy is intriguing and scary. I love this show and I'm sure any modern superhero flick fan will enjoy it too.",
    "This show is a new favorite of mine. It has the suspense, grit and sense of humor that I love from the DC universe. It also has one of the most realistic portrayals of black family life that I have seen on TV.",
    "The CW brings on a black superhero TV show. It starts with an interesting premise. It's a middle class black family trying to protect an oasis of civility within a tough black community. It still tackles the growing BLM and other racial issues.",
    "I was excited for a new superhero show but the more I watched the show the worse it became until I gave up on it. The action is horrific. The story line is long drawn out and unappealing. The main focus of the show is black issues every chance they get.",
    "This show is a beautiful marriage between DC and CW. It captures the realms of black fatherhood, the struggles of the black community, sexuality and feminism. The actors do a great job of portraying the characters and the writing is good.",
    "The show is not perfect granted but it is not that bad. It is entertaining, the acting gets somewhat better in time. I only dislike some editing choices. It really is not as bad as people say.",
    "The first episodes were amazing but from there it is somewhat losing its steam. I think they are trying to have so many story lines so early to the point where it feels rushed.",
    "I watched this show because it looked interesting and I love a lot of superhero tv shows. The acting feels forced and hard to ignore. A lot of the characters are stereotypical with next to nothing unique about them.",
    "I had hopes this show would get better but it turned out just as bad as the first season. When compared to the ratings from all the other superhero shows it's on the bottom.",
    "I liked the show for the first season and part of the second but the writers are not so good. People who get killed are constantly brought back to life which is typical CW nonsense and the characters constantly make bad decisions.",
    "Bad acting, bad stories, bad everything. I watched the entire first season hoping things would get better but they didn't. It wants to be a drama, it wants to be action, it wants to be a superhero show but it isn't any of that.",
    "This show is entertaining in an uncomfortable way. It's so different from other superhero shows. It's GOOD in unfamiliar ways. This is several cuts above the emotional stupidity of most super stories.",
]

add_show("Black Lightning", black_lightning_reviews)

✅ Black Lightning predicted as: Balanced
✅ Added 35 reviews for Black Lightning


In [98]:
sam_and_cat_reviews = [
    "This is a nice enjoyable series to watch. Yes there was room for improvement but it was enough to be fun to watch. The cast was great. They really committed to their respective characters and to the storyline of the whole series. This series connected very strongly with the other teenage shows.",
    "It isn't as good as some other TV shows and not every episode is great. Most are though. It is rather comical but when you look at the other two TV shows that kind of brings this one together, Victorious and iCarly, this one isn't as good. It is still funny and for everyone even though it is aimed at younger audiences.",
    "Another low-brow immature offering from Nickelodeon. The show generates a few laughs but continues to rely on slapstick comedy a little too much. Sam Puckett bores me and although Cat Valentine is adorable there is only so much of her you can take. Putting two supporting actors together does not equal a star.",
    "I tuned in to see what Jennette McCurdy was up to and Ariana Grande's character totally shocked me. She is so unique for her age and she's quick and an extremely good actor and comic. Some of the stories and lines give bad examples to kids but some of the stories lines and sight gags are extremely funny. The acting and laughs sold me.",
    "Sam and Cat features Sam Puckett from iCarly and Cat Valentine from Victorious meeting up in Los Angeles and becoming friends. Sam and Cat were just meant to be on-screen together because their dynamic fits so well. The episodes are sometimes a hit or miss but give it a shot if you're an iCarly or Victorious fan.",
    "Ariana Grande's character Cat is written to be so dense it's not even funny. On top of that her voice is horrendously irritating. My son is nearly 6 and I have never once told him not to watch something until now.",
    "I loved Sam and Cat when I was younger but rewatching it there were some things that made the show just ok. Cat is by far the best character. The theme song is catchy but I still grew up on this show and it should have lasted longer.",
    "This is rubbish. I love the music that Ariana Grande puts out but the stupid affectation she adopted for her role as Cat makes me want to rip my eardrums out. Nickelodeon used to have some funny shows on it but those days are long past.",
    "Sam and Cat is exactly what the title says - two characters from two of Dan Schneider's previous shows put together in one environment. There are a few laughs scattered throughout but genuine gut-busting comedy is nowhere to be found. It seems Schneider is simply going through the motions.",
    "Not as good as iCarly or Victorious. It was good at times but overall it was missing the dynamic of a big friend group. I think they were trying too hard to live up to the other shows.",
    "The show can't decide which way it should go, sometimes it falls on the random and brainrot side but other times on the creative and funny side. It just lands between the two main series. The cameos are nice but more like a wasted potential.",
    "Sam and Cat is a fun and entertaining teen comedy that is sure to make you laugh and leave you feeling good. The chemistry between the two lead actresses is undeniable and their comedic timing is spot-on. The show also tackles important themes such as friendship honesty and responsibility.",
    "Sam and Cat is a bit of a disappointment. It looks like every other show running on Nick right now. The Cat character isn't as funny as she was on Victorious. The humor seems skewed for 5-8 year olds. There are no original characters and no original jokes.",
    "For all of the haters you are probably adults rating a TV show which is not fair. The plots have gotten wilder, the stories have gotten funnier and the level of excitement increased. These girls are truly amazing together.",
    "Sam Pucket and Cat Valentine make one crazy duo! That's a good thing because people love crazy! If you like both iCarly and Victorious turn on Sam and Cat.",
    "This is one of those pre-sold spin off shows trying to fuse characters from Nickelodeon classics iCarly and Victorious. The character dichotomy is appealing to viewers who respond positively to the way the two girls complement and complete each other. It's a cheap and unimaginative production though.",
    "I used to watch it over and over when I was a kid. So I thought I would try it again and it still is one of my all time favourites. It has a great amount of episodes that keeps you engrossed at all times throughout the show. I would absolutely recommend this to anybody.",
    "Having one dumb character on a sitcom is already ridiculous but having two completely moronic characters is quite possibly one of the worst ideas ever. The show isn't creative or innovative and it's based on two characters who could never stand on their own.",
    "Sam and Cat is an unnecessary spin off. It's just an excuse for Nickelodeon to drag on the half dead Victorious and iCarly corpses for just a little bit more live comedy. Honestly this show was my childhood and I grew up on it so I can't say I hate it.",
    "Just rides on the coattails of iCarly and Victorious. Cat's voice and character is dialed up to an 11 and not in a good way. Extremely average show that can be entertaining at times but has no substance whatsoever.",
    "The bad acting and seriously flat and horrible voice from Ariana Grande hurts to watch. The usual tropes and unoriginal. I cannot stress enough how terrible this is from Grande's performance.",
    "I feel like this show is from a brief era in American television where the creators think it's funny to be as obnoxious and loud as possible for laughs. This show is the very worst of children and teen comedy. My child loves this programme but unfortunately I've had to ban her from watching it as she starts to mimic the characters' behaviours.",
    "If you can get past the absolutely annoying character that is Cat played by Ariana Grande then you might find this show cute. It's about two friends that are pretty much opposite and they make it work. Her voice and the ignorance her character is supposed to portray just gets under my skin.",
    "The show is not perfect. Sam and cat is nothing special. It is about 2 girls who meet and become roommates who are totally different and that is basically what the show is about. This makes it boring and repetitive. The only reason I gave it higher is because it does have some funny parts.",
    "My daughter watches this show over and over so I guess it is doing something right as long as you are only 9 or 10 years old. Cat's voice drives me insane. The acting is terrible, the plot ludicrous. Avoid if at all possible.",
    "After watching a few episodes with my child I asked her not to watch it anymore. I was disgusted by the low level bullying and discrimination that peppers the show. The way characters are portrayed and treated if they have a lower intelligence than Sam is awful.",
    "This show is honestly one of the most nostalgic shows there is for me. As a huge fan of Ariana Grande I really enjoy seeing her in movies and TV shows. It makes me watch episode after episode.",
    "Sam and Cat is complete trash compared to shows like Drake and Josh, Zoey 101, iCarly and Victorious. Sam and Cat are two friends living together off of babysitting money. Sam does the same stupid tough girl thing and Cat is plain stupid.",
    "I know I am not the target audience but a lot of shows my daughter wants me to watch with her are pretty good. But not this. Annoying characters with nothing new or original. Just annoying people with very contrived personas clearly just rehashes from previous shows.",
    "This show is the most unnecessary show ever. Babysitting is one of the most mundane occupations to revolve a show on. Cat became this obnoxious character that just blurts out at everything. She doesn't even have a brain anymore in this show.",
]

add_show("Sam & Cat", sam_and_cat_reviews)

✅ Sam & Cat predicted as: Balanced
✅ Added 30 reviews for Sam & Cat


In [99]:
bridgerton_reviews = [
    "I absolutely love Bridgerton, but what on earth happened this season? Polin didn't feel like the main characters at all. Also why was the season divided into two parts? It felt unnecessary. The drastic changes they forced on the characters were abysmal. This isn't an adaptation from the books, it's a completely new story.",
    "Fun for those who enjoy the frothy visual thrust, light story and modern casual take on costume drama. It's a trashy soap opera in a dream version of Regency. I do like the diversity in casting and experimentation in costume. Casting is strong. However the soundtrack is the same thing again and again, not creative, boring.",
    "If you are into the accurate retelling of history then skip this. This isn't about the accurate retelling of history. Now if you're into the book series do give this a try. It's like a whimsical romance. Something like those Hallmark Christmas romance movies but with more mature scenes. They really brought to life the family dynamic of the Bridgertons.",
    "As a black dude I actually liked this show. I thought for a second that this was true history but then I began to understand that this is all fiction and a made-up fantasy. I didn't take it seriously. I just turned my brain off and enjoyed the fantasy.",
    "I have to note that I had reservations about this when I first saw the casting. After seeing the characters on screen I must say they are PERFECT. The visuals, directing, and music reminds me of a true historical romance book. This doesn't feel like a made for TV show. I love how they managed to make the colors so beautiful and vibrant.",
    "Knowing nothing about the books or what to expect from the show I have to admit that the diversity threw me in the beginning. By the second episode I became colorblind and grew completely enchanted by it all. Although I loved it, in a way it made me sad. All the diversity made me see how the world could have been and needs to be.",
    "The costumes, settings and music are all beautiful but we've seen them before. This season lacks the substance and intrigue that led so many to binge watch the first. The love triangle between Anthony, Kate and Edwina feels forced with none of the actors particularly charismatic.",
    "I enjoyed the first season, I loved the second season even more, third season was kind of boring. The only thing that kept me interested was the delightfully subdued interactions between Francesca and John. As for the main story between Colin and Pen, BORING. It was like bad fan fiction. There was no character development.",
    "Usually I find myself disappointed by second seasons of shows but I can confidently say it's better than season one. The second season is a must watch with a thrilling slow burn and an incising story. I cannot wait to see how this show plays out.",
    "I first read the books years ago and this adaptation has done it justice. Perfectly cast. The people obsessed with the diversity should know that this isn't a factual historical show but a version of Georgian history that lives in the minds of all of us who love this type of thing.",
    "This is not a quality Pride and Prejudice type series or Downton Abbey. This show is an exaggeration of those kinds of tales and it is pure fantasy using the backdrop of Victorian English high society. Do not take it seriously and enjoy it for what it is.",
    "I really don't understand all the negative reviews for this series. I am not a big fan of historical shows and would have never watched this had it not been recommended. What a pleasant surprise this series has been. Lovely settings, amazing wardrobe, lovely cast, smart and engaging story line.",
    "Ever since hearing this was going to be made into a series I have been anxiously awaiting it. This series has gone beyond my expectations. I love the costumes, the pageantry and above all the cast. I love the addition of a diverse cast. It in no way detracts from the story and makes it a beautiful representation of today's society.",
    "Facing a Christmas Day at home alone this splendid release was perfection. It is gorgeous to look at. It is witty and warm and faithful to the romantic literature it is based on. The hand touch between Simon and Daphne is far more sensual than anything written or portrayed in Fifty Shades. Just luscious.",
    "Is it entertaining? Yes. Do I expect from fantasy to give me accurate descriptions of the world? No. It is fantasy dear viewer. I expected something light and lovely and it was highly entertaining. It is a piece of heaven and hard work. And it shows.",
    "The sets, the costumes and the cast make this regency romance sparkle. A lovely historical fiction series.",
    "This is the TV equivalent of delicious cake topped with swirls of frosting. Sweet and fluffy and beautiful to look at. This isn't historically accurate and isn't trying to be. If you're looking for an absolute visual treat of pretty people and prettier costumes then this is a winner.",
    "You can rely on Netflix to splash the cash when it comes to a series of this kind and Bridgerton doesn't disappoint with beautiful houses, lavish parties and exquisite clothes. The plot is better than average with a whole string of Bridgerton siblings needed to be married off. Very good entertainment.",
    "I haven't read the book and didn't even know there was one so I had no expectations. It's a fantasy show and I don't need it to be accurate, I need it to be entertaining and it absolutely was. Great costumes, humorous storyline, the chemistry and mystery had me enthralled.",
    "I binged the entire season right after it dropped! Like the well-written books, I couldn't put it down.",
    "It's a great series. Not supposed to be the books but a different version inspired by them. I like the diverse cast and the amazing costumes. I've been a Julia Quinn fan since she started writing and I think this series does the Bridgerton books proud.",
    "Wonderful, entertaining and an aesthetic accomplishment. Even the music is exciting. There's also a lot of good humour. The acting is superb, the costumes are amazing and the story is interesting.",
    "After 2 excellent seasons of a well written well produced series I waited with anticipation for series 3 to drop on Netflix. And boy what a disappointment it was. The showrunner had decided to suck out any of the whimsical joy of Seasons 1 and 2 and instead dished up an incoherent mess.",
    "Such a fun and joyful experience to watch. Love the casting, truly some of the dreamiest actors around!",
    "I'm not familiar with the Bridgerton source novels but this is a racy entertaining and often funny update of the Austen and Bronte novels. I loved the casting of all the major characters and the diversity of the whole cast. It is not life as it was but as I wish it were.",
    "A refreshing take on historical drama. The humour is inspired, the costumes are divine and the cast all played their parts perfectly. I was hooked from beginning to end.",
    "Season one was steamy and tense. Season two was just tension that was overly abused. It was a cycle that never ended until the very end. I certainly would recommend season 1 but season two was meh.",
    "I wish people would look beyond colour. It's a fictitious period drama. So what if black people are Dukes and ladies.",
    "The leads of Season 1 had chemistry and overall the ensemble puts in a good performance. The sets and costumes are amazing. Unfortunately the leads in season 2 have no chemistry and their limited acting skills are glaringly bad at times.",
    "I have watched season two and three several times before writing this. The last episodes of both seasons are so chapped and unclear and could have made more to be more fulfilling. I hope that there will be more seasons and more to hear from Shonda.",
]

add_show("Bridgerton", bridgerton_reviews)

✅ Bridgerton predicted as: Balanced
✅ Added 30 reviews for Bridgerton


In [100]:
euphoria_reviews = [
    "I'm loving this show and the issues it explores, however I do not believe the content jibes with a high school age cast of characters. I believe it would have better and more realistically been explored in a college setting. Nevertheless I give the show a 9 because it is very well done and crosses a lot of boundaries in TV making.",
    "In the beginning it seemed like the show was dark just for the sake of being dark. Everything seemed clichéd and there was a vast degree of exaggerated negativity. However during the fourth episode I began to witness the story take a definite direction. I don't think I've ever been moved this much by a TV show.",
    "Sex drugs and more in high school. Although this is a show about high school teenagers it's too much for teens to watch. There are parts when the scenes are much more explicit than necessary for the story. There are some fascinating and bizarre characters that keep things morbidly watchable. Even if you don't agree with the suitability of the content the acting is better than it has to be.",
    "I like it but I don't understand the point of it. Cinematography is beautiful. Actors did a really good job and overall it makes a decent show. But I don't understand if it's trying to spread awareness or trying to encourage young people to do this type of behaviour. I would still recommend it to people.",
    "This stylishly filmed angst-fest follows a variety of morose high-schoolers as they deal with addiction, self-esteem issues and the local neighborhood psychopath. It's also a very glamorized version of teen angst. Zendaya is terrific as the head angster. It captures a lot of the sadness, confusion and isolation of being a teenager.",
    "If today's teenagers are really this vapid shallow self-involved and absurd the world is in deep deep trouble. Obviously these teenagers are incredibly affluent. You know not everything is about sex in high school right? Depressing as hell.",
    "Excellent acting, excellent directing, excellent visuals. The only thing that is really bad is that it's soooo out of reality. They created a fake world of teenagers being always on the edge with too much drugs and too much sex. I hope no actual teens are taking notes from it.",
    "It's way too over dramatic. The lives of each teen seems so unrealistic. I see that they're trying to cover real problems but it just doesn't make much sense for high school students. I'm 16 years old and it felt like I was watching a bunch of teenagers acting way too old for their age.",
    "From a technical perspective Euphoria is untouchable and definitely brings novelty to the genre. I really enjoyed the cinematography style and the editing is out of this world. The visual representation is dreamy, the lights are vivid and colorful. I cannot say that I was really impressed by the plot which I found very unrealistic.",
    "The show is truly addictive and intense. The amazing cinematography and creative storytelling are hard to beat. The impressive cast made it even more addictive. By watching Zendaya laughing and cry could be so entertaining and moving.",
    "After watching the pilot I can already tell this show is going to be something special. It's gorgeously shot, well-acted and above all painfully honest. One of the best pilots I've seen in recent memory.",
    "I honestly wasn't sure that I would stick with this show but I'm happy that I've given it time. I like Rue but she's terrifying. I've known Rue's and it's a story that needs telling. It's a difficult show to watch because you're watching teenagers self-destruct and honey it's graphic. There is a lot of trauma.",
    "This is a show that requires you to not only be a mature viewer but also willing to let the show take you on a journey that will not be pleasant. Where Euphoria truly shines is within both its performances and overall look. Zendaya delivers an engaging central performance. Overall Euphoria is a series that warrants the praise it has been receiving.",
    "Euphoria is a show that's hard to rate let alone review. It's an experience like no other for all age groups especially if you're a teen. Its depiction of issues is graphic, brutal and honest. Its acting performance is mind-blowing. Its cinematography and shooting gorgeous.",
    "After months and months of tease and hype Euphoria is finally here and it delivers. It encapsulates everything that my generation has to deal with and it does it in the rawest most hard-hitting way possible. The characters feel grounded, real and relatable. Dare I say it Euphoria might be a generation defining series.",
    "This is one of the realest shows I've ever watched. This show shows the harsh reality of what actually goes on in some people's lives. Not everyone had it so easy in school. This is by far one of the realest and meaningful shows HBO has ever produced.",
    "I actually can't believe people giving a 10 rank after seeing episodes of season 2. I was a big fan of Euphoria. It wasn't just a show, it hit different. It was unique, it really made you feel things and think. But what is going on with season 2? They just make the characters do stuff. You should have ended it on season 1.",
    "Blunt, provocative, raw showcase of sex, drugs and social media. Everyone knows that the times have changed and today's kids live much differently. The show is an over the top teen drama but overall one blunt outspoken provocative series.",
    "I'm no prude but I've seen less sex in actual pornography than in this show. The fact that the characters are teenagers makes it extra uncomfortable. The casting is great, the cinematography is extremely interesting and I like how it plays out like a mystery.",
    "Shows like these that create false realities are a major reason teen mental health is such an issue. It is unacceptable that pop culture is glorifying this unhealthy lifestyle through softcore porn.",
    "The honesty in this show is amazing. With fascinating characters and stories that feel genuine this show finds a comfortable place in your mind for weeks after you have finished it. It offers a honest portrayal of the modern teen culture in the US. The soundtrack is one of the best and most awesome soundtracks.",
    "I fear for my goddaughters. This is a trip into the extremes of teenage psyches and it is terrifying! Beautifully filmed and acted with a great soundtrack. Be warned this is not for the faint hearted. An excellent series that will open your eyes to the potential disaster that faces youth in their journey to adulthood.",
    "Euphoria is delicate, cold, warm, disruptive, deep, dark, uncomfortable, provocative. It has so many elements and layers of humanity. As a warning it could be triggering for some people.",
    "If I'm rating this solely on filmmaking, acting and story I'd give it a 9. But seriously there are no happy or redeeming qualities of this show. High school isn't even close to what this show shows. This series is made with pure evil all around. Great show but man I wanna go watch some SpongeBob now.",
    "I had heard about this series from female friends and watched it with hesitation. My hesitation was based on a completely false premise and I should have been hesitant based on the extreme discomfort I would experience in watching it. An excellent series that will open your eyes.",
    "Both seasons have stunning visuals and cinematography. Season 1 had a more defined plot and you could feel they were working towards something. Season 2 is really dark and has a lot of unbearable scenes to watch. Someone was talking about how euphoria makes you mortified at the thought of what your children could possibly be up to in school.",
    "I wanted to watch this show for a long time and finally started it. The show has beautiful cinematography, it's visually stunning and the actors did a really good job. It's a teen show but not really meant for teens as it deals with a lot of heavy matters. A binge-worthy show that I would definitely recommend.",
    "Technically the series is very well done. Great camera and directing, good acting and the music fits in great. But Euphoria pushes way beyond anything I've seen produced for television. The nihilism is thicker than the powders smoke and alcohol consumed. It's less a reflection of reality than it's a warning.",
    "I give it six points for Zendaya who is always an interesting actress even in a show that is chock full of shock value for shock's sake. This might have been better if it had focused on high school grads first going to college. Seeing young kids involved in stuff this serious is just disgusting.",
    "This show is a masterpiece. Everything from the characters to the music choices and the brilliant cinematography is a pure joy to watch. It isn't for the uptight however. You will see lots of violence and nudity but it's necessary for the story. Just watch it.",
]

add_show("Euphoria", euphoria_reviews)

✅ Euphoria predicted as: Balanced
✅ Added 30 reviews for Euphoria


In [101]:
something_very_bad_reviews = [
    "I'm a couple of episodes into Something Very Bad Is Going to Happen and it's landed well so far. Watch it in the dark - the low lighting and muted palette aren't stylistic afterthoughts they're doing real work. It's not a conventional horror. It's more interested in sustained unease than quick payoffs. The pacing is deliberate and the tone is controlled. From a craft perspective it's very assured with strong performances and thoughtful production design.",
    "This one started out so good, eerie, uncanny and throughout Lynchian vibes. Almost scary at times. Characters' weirdness felt like it was all part of a puzzle. Then the reveal happens and from there onward it started to fall apart. The first 4 episodes felt like a false twist for the last 4. It was still fun though.",
    "I was extremely hesitant to watch this after the reviews I read. I am so glad I gave in and watched it. Is it perfect, no, is it bad, no. As a horror buff I completely respect it and found it to be worth the watch. I actually enjoyed the fact that everything you think you know or expect after episodes 1 and 2 changes.",
    "The first episode starts off with so much promise. About halfway through the pacing just falls apart. What begins as a tight intriguing story turns into something long winded and unnecessarily dragged out. You can feel the writers stretching a plot that should've wrapped up in a few episodes. This could've been a sharp memorable limited series.",
    "Netflix's Something Very Bad Is Going to Happen is a 2026 horror series built around a simple but effective premise. One of the first things that struck me was how well produced it is. The story itself is very good. It takes familiar horror ideas and presents them in a way that remains engaging. Jennifer Jason Leigh is definitely a standout.",
    "The series is beautifully shot. The directors play well with light, shadow and bursts of red. But the plot drags and most of the scenes could be cut. It feels like a movie's worth of material was split into 6 episodes. The last two episodes explore what actually is wrong but by that time I won't blame anyone for being bored.",
    "This was actually so good. Full of twists, great scares and humor. Plus it's so well made, at times Lynchian, aesthetically so pleasing. The writing is really good, the characters are great and the dialogues were amazing between them. Camila Morrone is an absolute superstar.",
    "I understand that a dark and moody film works in cinema but Netflix is different. Many watch on a computer screen during the day. Why do directors still choose to film everything as dark as possible? When you can barely see anything the fun is gone.",
    "This series wasn't as bad as people are saying. It's slow but the story did intrigue me. Usually I would've stopped watching after the first episode but the mystery kept me going and I wanted to know how it would end. The acting is actually really decent. I don't think it deserves 1 star at all.",
    "I agree with all the other commenters saying that the first 4 episodes are good! Building suspense lots of different things happening and you don't know what's going on. Ultimately I found the big reveal to be really dumb. The show really dropped off in the second half and it was disappointing.",
    "Brand New Cherry Flavor was ahead of its time and this probably is too. Such a vibey entrancing first episode then it never let up. Genuinely funny, stunning gore and practical effects, brilliantly original story, supernatural, witchy, gnarly, unforgiving. Truly one of the best series I've ever watched.",
    "This show is like that office meeting that could have been an email. Almost an 8 hour show that could have been a 2 hour movie. The sound design is jarring and painful to the ears. The lighting is all dark. The editing is intentionally choppy to seem edgy but comes across as a massive interruption to the flow.",
    "If this is a social commentary on the horrors of marriage I'm here for it. The story unfolds like a horror show making this couple face all those feelings the week of their wedding. Characters develop in the ways you least expect and the plot forces them to face the greatest fears of any union. I love how this ends but also hated it.",
    "Another Netflix padded out series about nothing. It's just drawn out scene after scene of absolute nothing. Not great. Bailed after 2 episodes.",
    "Something Very Bad Is Going to Happen is the sort of deliciously baroque endearingly weird atmospherically gothic horror miniseries that sticks out from many in the genre. The suspense and ambiguity of what's going on here is so much fun with some great and hilarious twists wrapped up in spooky wintry atmosphere. Camila Morrone delivers a star making performance.",
    "The first 4 maybe 5 episodes were extremely thrilling, mysterious and interesting. There was a sense of dread like something very bad is going to happen. Unfortunately they gave the entire thing away and went almost comedy, family drama. They could have held the mystery to the end. The acting was great and scenery perfect.",
    "Not exactly scary per se but a constant level of dread and unease is felt throughout. The MVP of the series is Rachel played by Camila Morrone who I am not familiar with but she really carried the show. A very satisfying horror series.",
    "This show started promising but then it becomes so boring. There is no horror at all just bad writing and a super annoying protagonist. It just feels like they dragged a plot for maybe 4 episodes into 8. The show is saved by good performances throughout the cast.",
    "I really enjoyed this series. Get past episode 1 and you're going to keep watching. Well acted especially the lead woman. I did figure out the ending by episode 6 but not exactly as I thought. Fate I'm betting your fate will be glad you watched.",
    "A great refreshing show that brings a breath of fresh air to the TV landscape. New ideas and things that you've rarely if ever seen before. The staging is incredibly well done. The soundtrack really kicks into high gear later and generally the entire cast is so well chosen. The series is simply fun and never feels dragged out.",
    "I went into it with mild curiosity at best. I ended up binging six episodes in one sitting and finishing the final two the next day. The writing around relationships is exceptional, the dialogue carries real weight. It's not just about plot, it's about communication, connection, doubt and what it really means to choose someone.",
    "I lasted the first episode and 20 minutes of the second and then just gave up. Poor acting, ridiculous script and the most stupid characters you can imagine.",
    "Up through episode 4 it was very good. Creepy, weird, Channel Zero level. Then it was like they lost what they were writing about. Episode 5 and 6 things went normal. Nothing makes sense after episode 4. I would not waste my time if I were you.",
    "Starts off strong. At that point you don't really know what's going on and it's a good creepy mystery. As the show went on it got a little less enjoyable but it held my attention well enough to finish the entire series. My one major complaint was that it was so dark I could barely tell what was happening in some scenes.",
    "The show dawdles and the revelation in the later episodes about its premise isn't anything new as the storyline has been recycled many times. However the finale is expertly directed and the ending is very satisfying to watch. Camila Morrone is fantastic in this.",
    "A series that starts with a brilliant scary bang but slowly loses its way. It begins as a terrific road trip full of dark omens and ends as a confusing family drama that spends too much time explaining its own rules. Camila Morrone is the best part of the show.",
    "I loved the first 4 episodes. It was extremely thrilling mysterious and interesting. Unfortunately they gave the entire thing away. I jumped on Reddit horror after episode 2 to recommend this as a very eerie thriller then later had to rescind. I will say the acting was great and scenery perfect.",
    "Started out ok but after 3 episodes I could simply not take anymore. Extremely slow. Not scary, not engaging, no feel to it. Just boring and weird and not weird in a good way.",
    "The first episode was a great setup. After episode two I saw that there was nothing to setup for. Useless dialog, shots that go on endlessly. The background music was great and wasted on non suspenseful imagery.",
    "Gave it three stars only because it held my attention enough hoping it would get better. It did not by a long shot. Eight episodes was tedious. This good cast was wasted. A better choice would be to watch the movie Ready or Not.",
]

add_show("Something Very Bad Is Going to Happen", something_very_bad_reviews)

✅ Something Very Bad Is Going to Happen predicted as: Balanced
✅ Added 30 reviews for Something Very Bad Is Going to Happen


In [102]:
moana_reviews = [
    "Moana is a return to the classic Disney formula, the clichés and characters ripped from a number of other animated films. However, the pure beauty and skill of the production rises the old story into new heights. The quality of the animation helps too: it's clear they have reached the pinnacle of blending realistic textures with stylised designs. Even the music has been perfected here, the annoying catchiness of Frozen's tunes replaced by memorable but effective songs that fit the culture and setting of our adventure. Moana is good, old fashioned Disney magic.",
    "The movie is still one of the best modern Disney movies. Its a real pleasure to watch with the kids. My daughter especially enjoys the Polynesian style and the catchy songs. Worth a watch and a revisit from time to time.",
    "The animation is phenomenal. Disney's best-looking film in a long time and one of their best-ever looking films. The soundtrack has garnered a huge amount of praise, and for good reason. 'How Far I'll Go' is an 'I want' sort of song that's infectious, heartfelt and inspiring. 'You're Welcome' sees Dwayne Johnson showing a quite wide range of emotions through a surprisingly good singing voice. 'Shiny' is deliciously kooky. Moana has now joined the list as one of my favourite female Disney characters, while Maui is a fun, compelling character. Knocks it out of the park in visual beauty and sheer entertainment value. 9/10.",
    "One of the best modern Disney films.",
    "Animation at its best! Now Officially one of my Favorite Movies!",
    "I love everything about this movie. The colors, the setting and the characters, all good to my eyes!",
    "Disney breaks no new ground with Moana's story. The animation is wonderful, the world is gorgeous, but the story is rather dull and shallow. If either Moana or Maui were engaging characters, this movie could have been good. Moana could have been such a good movie, but it suffers from the same failings as most recent Disney animated ventures: wonderfully crafted world without a story worthy of that world.",
    "Why this movie has 7.6? Y'all crazy, this is a great movie, with amazing music. Yes it's not a perfect portrayal of polynesian culture but that's usually never the point with Disney movies.",
    "THE BEST DISNEY MOVIE EVER. The animation, screenplay, sound and story line is outstanding. Kids love the movie, and even adults. Overall 10/10.",
    "Great soundtrack and visually creative but nothing much else to offer.",
    "Instant classic. Easily ranks among the best from the Disney Studio, and a very worthy successor to Mulan 1998. Oddly also seems to be best work ever from Dwayne Johnson. The ultimate themes of redemption, forgiveness and self-discovery at the finale are an absolute joy.",
    "2016's 'Moana' is worth watching! It's a luau of a movie!",
    "The pay-off is not worth the build-up.",
    "Wow, what a great great movie! The visuals are breathtaking! Moana is such a likable, smart, and relatable person. The chemistry between Moana and Maui is so great and so natural. The songs are so charming, catchy and fun! How Far I'll Go is such an emotional song - I dare say I like it a little more than Let It Go! I just looved this movie and it is great for kids and adults.",
    "It's no surprise that I found myself tapping my feet to the songs in this film because Lin-Manuel Miranda shares his talent with this film. The film itself might lack some intricacies but this journey story is entertaining and heartfelt. The vibrant colours of the green islands and the blue sea help propel the film above and beyond. Johnson is stellar as Maui, newcomer Auli'i Carvalho holds her own against a star like Johnson. Disney nods and respects their past while looking to the future with Moana. Moana is a visually beautiful film with great songs and a strong female lead character.",
    "From the opening lyrics to the final shot, Moana was a fun glimpse into Polynesian life. Even if you can predict the ending of Moana, it's not quite what you expected. It's not as amazing as Zootopia, but Moana is still a genuinely good Disney movie. It has a superb single, a nice accompanying soundtrack, great characters, sublime animation with crystal-clear attention to detail, and wonderful themes and messages.",
    "Visually beautiful.",
    "A fun Disney animation without the heart to make it a classic. Moana owes a lot to past Disney hits like Mulan, The Lion King and Frozen, and while the island/ocean setting sets it apart, Moana doesn't actually end up delivering anything that fresh or new. Energetic and sometimes laugh out loud funny, Moana is an audience appeasing event that's tailor made to keep the little tykes content and the senses dazzled but the heart that transforms some Disney films into instant classics is curiously missing here.",
    "With Pixar dropping the ball lately it is good to see that Disney is picking it up with great movies like Tangled, Zootopia and now Moana. Moana really doesn't disappoint, it has good pacing, a great cast, and it is BEAUTIFUL. The music, as expected, was typically Disney which is a good thing. The cinematography was beautiful and it shows that the creators have done their research. Moana is a must see movie that won't disappoint. 8.5/10.",
    "One of Disney's hidden champions! The animation is one of Pixar's finest works to date; the music is beautifully written and very catchy, the characters and their development feels real and the story's message is inspiringly positive. Moana is not your typical princess fairy tale - it's an adventure set in one of the most beautiful settings on Earth. I highly recommend it - whether you have kids or not.",
    "As cynical as I am about Disney movies, I freely admit that I liked Moana quite a bit. The songs are good, the animation's beautiful, and the formulaic heroine with sidekicks really seem to work here. Moana has the spunk and strong personality to be a winning character. I could easily watch this again. 7/10.",
    "Lame rehash. Just a rehash of the girl-you-can-do-it Disney princess movies. Also wrestlers throwing in his obligatory growl now and again is annoying.",
    "It is indeed very rare that I toss a top rating at an animated movie, but Moana definitely deserves that. The storyline appeals to both a younger and older audience - it is a story of friendship, forgiveness and acceptance. The animation and art style was just spectacular. The levels of detail was just bedazzling. I was instantly blown away by the comedic element of the chicken. This is a fantastic animated movie in every sense of the word.",
    "Moana is a fabulous movie - it will certainly leave you crying and smiling. The animated graphics capture the magic and beauty of Polynesia, the land and the ocean. And the story is a lot of fun. Highly recommended for all ages.",
    "Disney within Disney.",
    "There's a theory that in this world there is only a handful of truly original stories, and the rest is just a combination and a rehash of those. The story of Moana definitely sounds kinda familiar. It's simply impossible not to fall in love with Moana, that's how beautifully this film is drawn. While this film is undoubtedly a joy to watch, it falls just a little short of becoming a real hit, probably because unlike its predecessors, it didn't bring into the genre anything fresh. Which doesn't mean that you won't have fun watching it.",
    "Maui, a demigod, steals the cornerstone of all creation: the heart of Te Fiti, resulting in the slow destruction of the ancient Polynesian islands. On paper, Moana is a quintessential Disney story. However, the movie chugs along with breathtaking animation, adorable interactions between the strong-willed girl and the hilariously self-obsessed demigod, and songs that make you shimmy involuntarily. Moana literally means 'the ocean' and as the heroine conquers her namesake, she also conquers your heart.",
    "This movie was pretty good but its story is way too formulaic. I could predict pretty much every major plot point as the film unfolded. On a technical level the film is truly remarkable - the animation is gorgeous and some aspects of it are even photorealistic. Don't expect anyone over age 12 to have their mind blown. 6.5/10.",
    "My daughters and I have fallen in love with this show. A movie set in the tropical islands, a heroine who saved the day, beautiful and fun music... what's not to love about it?!",
    "Expected more from Pixar. The new disney princess has astonishing visual effects and characters, but the story is somewhat boring.",
    "I LOVE this film. Unlike most Disney female leads we have had in the past, Moana is a leader. She is strong and brave. This film did not need a love interest despite it having a female lead, which is quite refreshing. The songs are catchy, the animation is realistically beautiful, and it shows the importance of family and community very well.",
    "The film has good songs, Moana is cute, and the chicken is funny. This film is not memorable for me, the characters are not deep built. Great graphics though.",
    "Why do people like this movie so much?",
    "One of the best kids' movies... my daughter loves watching this every time. This movie has a beautiful collection of songs that anyone can enjoy.",
    "I am a really big fan of Disney princess movies, and I have to say, this is one of the best ones yet. I love the characters, the animation, and the songs.",
    "I really appreciated the tribal basis of this animation. Moana was a very well represented cultural animation. I loved the fact that their facial features and body structure were not all lean and straight. I loved the plot as well - there was a point I actually got chills all over.",
    "Moana is elite and poops on Frozen from a great height. Fight me.",
    "Not a Disney Flick. Kinda boring. The sea is beautiful. The girl is cute. Except for that its nothing. The demi god is boring and his tattoo is double boring. The chicken is worst of all, not funny at all. The songs are not catchy. Please don't waste your time watching this thing.",
    "Uninspiring.",
    "For the kids: it has funny jokes, cool villains, and is visually stunning. For the more mature teens and adults, this is a beautiful story of self discovery. An incredible message of finding who you are and discovering what you are capable of. Finding inner strength when no one believes in you. A truly inspirational and deeply emotional story. 10/10 stars.",
    "Moana is a (thankfully) non-Americanized cultural piece surrounding islander mythos, reminiscent of old Disney. 'You're Welcome' is perhaps one of the best songs seen in these films. The idea of a hubris-filled demigod that conflicts with Moana is actually great, however Maui is an under-exploited plot device throughout the film. Despite attempts at worldbuilding and storytelling, it lacks depth. That all being said, it is visually stunning, entertaining, and captivating in its musical pieces and premise.",
    "Moana (or Vaiana) is a very interesting and touching movie. Auli'i Cravalho does a fantastic job with her insanely talented voice. Dwayne Johnson is great as Maui - he brings such a fun, lovely, and comforting voice to his character. With movies like Tangled, Wreck-it Ralph, Frozen, Big Hero 6, Zootopia and now Moana, Disney shows no sign of stopping in their road to greatness.",
    "This isn't just about Disney; recently, animated movies as a whole seem to have become overly simplistic. The backstory was intriguing, and the music was enjoyable, but the film lacked nuance and missed opportunities to create deeper emotional resonance. It overexplained everything. Not everything needs to be explicitly stated - leaving room for imagination would have made the experience far richer.",
    "Maui was one of the best modern Disney characters, undisputedly the best in the movie. The Rock's performance was excellent. The comedy was excellent. The story was very good. The graphics and character design were great. The songs were beautiful.",
    "It was okay, not great. The songs were kind of dumb and repetitive. The story seemed like it could have been a bit more interesting. Some funny moments and ends on a high note. Worth the watch but not something I would go back to.",
    "Disney is such a cliché. This must be the most over-hyped crap of a movie since Frozen. The first 2/3 of it are just frantic, predictable cultural jokes, and near the end it becomes an awkward mix between true romance/drama and slapstick heroism. Not recommended unless you are very easily amused.",
    "Another typical Disney movie of princess, magic, spells, curses, and CONFUSION. It just drags and drags and drags. It is also somewhat predictable. Two good things: the short story before the movie was outstanding, and Moana exhibits incredible computer animation. Other than that IT IS A HUGE DISAPPOINTMENT. I warned you!",
    "What I'll say is that this is a cute movie. What hooked me in was the mythology - seeing Maui as a combination of Loki and Prometheus fascinated me. The voice acting and the singing are great across the board. The animation is on point. This is a cute little film that I had fun watching. I'd recommend it, especially if you have children. 8 out of 10.",
    "Yes some reviews complain that it follows the standard order of Disney movies and it does, but wow is it amazing. From the animation to the music to the story. It's incredible. I can't watch it without tearing up and I've watched it 20 times so far.",
    "Beautiful looking and highly entertaining animated film from Disney. I really loved the vivid colors as they really jump off the screen. The film also works perfectly as an adventure. The Lava Monster was brilliantly designed. Both Moana and Maui add up to one of the more entertaining Disney duos in recent years. The vocal performances are all extremely good. Moana really is another winner for the company.",
    "With a strong female lead Moana is a story with great songs and even greater visuals. In the colorful environment Moana is joined by Maui, who is expertly cast as Dwayne Johnson. The cultures from the Polynesian area are shown in the story with respect. I would definitely recommend the movie.",
    "Visually Stunning.",
    "Lilo Stitch better. Good songs but the movie could have been better. Left a bit unsatisfied.",
    "Visually gorgeous, with its water effects and lighting being the highlight. Polynesian culture is shown beautifully here with so much love and attention for detail. The main cast is small but both characters are full of personality and the voice acting is perfect. Moana is a beautifully animated film with a strong story and fun characters with plenty of laughs to go around.",
    "This film is an epic in the truest sense and easily one of Disney's best Big Films. The animation is second to none. Unlike other Disney movies, Moana is perfectly capable of pursuing her goals completely by herself and doesn't have a love interest. The soundtrack is also exceptional. It was the second movie in which I've ever witnessed the theatre audience erupting into a round of applause when the credits rolled.",
    "Not one of the better Disney animated movies I've seen over the course of my years. Probably due to the weak and occasionally obnoxious soundtrack that made getting through some of the scenes a real struggle.",
    "I've heard people call this movie one of the best Disney animated movies and while it is certainly good, I personally think it's one of the weaker entries to the new Renaissance. The plot is pretty clichéd and weak. The animation was stunning. I like the song 'How Far I'll Go' but they use it THREE times. I recommend this movie but I definitely think it is one of the weaker entries to the new Renaissance. 6 Crab Solos out of 10!",
    "A friend and I went with the kids to watch Moana. I loved seeing Polynesian islands in animation. The scenery is beautiful. The storyline was typical Disney - I feel like I've seen this movie a dozen times before. Other than the predictability and the glorifying of pagan belief systems, it was an entertaining and uplifting movie.",
    "I really appreciated the tribal basis of this animation and the cultural representation. I loved the fact that their facial features and body structure were not all lean and small nosed. I loved the plot so much, there was a point I actually got chills all over. I can't remember how many times I've re-watched it.",
    "Moana is my favorite Disney movie and Disney Princess! I love the animation and the beauty of every movement. The story is empowering and inspiring! Moana is the bravest Disney Princess!",
    "I think the story is amazing and I love the feeling you get while watching it, really a happy movie. Two of the songs were amazing. Must watch!",
    "Good movie with amazing visuals. However, if you've seen Frozen, Tangled, Wreck it Ralph, Lilo and Stitch, or any other Disney film within the past 30 years, then no need to see this one; because it's basically the same plot.",
    "Moana is one of the best Disney films of the past decade, it is sure to soften many hearts with its gorgeous storytelling and fantastic songs, it's also absolutely hilarious.",
    "Moana has been our 'Corona'-comfort movie in our family. I love the story and the main character. I love that in this movie it's not about a love interest, but about self love. The message about knowing who you are - despite what other people do to you - is so important. The music is awesome and we can sing along to all of the songs.",
    "Another film I watched while with my fiance that she picked out. It was not bad and had its interesting moments and it was really nice to look at. The story is weak. A very pretty film to look at, you get a few catchy tunes, but you also get the same thing we have seen in quite a few Disney films lately.",
    "Another typical Disney movie. Please Stop Singing.",
]

add_show("Moana", moana_reviews)

✅ Moana predicted as: Balanced
✅ Added 66 reviews for Moana


In [103]:
spongebob_reviews = [
    "It's really sad that the creator of this show died but I would still love this show for the rest of my day :)",
    "If it was just the first ~5 seasons, I'd give it an 8 or 9, but then after that, it just gets worse and worse until now I'd say the show's like a 3, maybe 4, which sadly made me decide to bring the rating down.",
    "This show was my whole childhood. This show will never get old. It's timeless.",
    "Given the circumstances (this being a cartoon) there is some decent acting. Character voices sound beat down, like theyve been given the food but a saturday has been extended 90 years.",
    "Spongebob Squarepants used to be Nickelodeon's signature series. All the characters were good and the plot lines had meaning, and the show had talent. After the movie the show started going downhill. Everything that the first three seasons had set in stone was defied. All the characters have just became stupid. Spongebob has become high-pitched and annoying. Patrick is just oblivious to anything now. Mr. Krabs has become too cheap and reused. Squidward too sarcastic. Too many plot lines involving Plankton trying to steal the formula. It isn't funny anymore and the plot lines are extremely predictable.",
    "I kept hearing horror stories about new SpongeBob episodes where the characters became little more than shells of their former selves. I did watch the new SpongeBob movie and was glad Hillenberg came back. Newer episodes are a big improvement. The show has some of the best lines and scenarios in animation and it's good to see it getting back on track.",
    "This is the best family show ever. My daughter loves it as much as I do and even my dad used to watch it. It's for everyone and you can watch it for hours. It's pure fun and the comedy works for all ages.",
    "Initially fantastic, but now generally good. In its peak first three seasons this show is a jolly trip through surrealism and relatable humor. At its current state it's not bad, though rarely does it reach that original level.",
    "Spongebob's one of those shows that everyone knows because its appeal is very diverse, from children to adults.",
    "Whether you're 8 or 80, Spongebob Squarepants is an eternal reminder that the best things in life are found in laughter, love, and a little pineapple under the sea.",
    "Spongebob Squarepants is one of the most influential pieces of pop culture and my favorite show of all time. Seasons 1-3 are classics with incredible world-building. Later seasons have more annoying and poorly written episodes, but the early seasons are too iconic to ignore.",
    "SpongeBob was once the best Nickelodeon animation. Recently the show feels rushed and not as good as before. If they return to the classic style, it could be better again.",
    "I'm not a kid but I still love this show. I watch it to relax. It's a brainless show where anything goes and it makes no sense which is great.",
    "I like it, but sometimes the characters are so stupid I can't stand it. Maybe you will like it, but for me it's just okay.",
    "Spongebob Squarepants was a fun show with memorable episodes and great moments like graveyard shift and fry cook games. But after 2010 it went downhill with annoying episodes I can't stand to watch.",
    "I've only just started getting into cartoons again and now I watch every episode. It's surreal and funny. The setting in Bikini Bottom and the characters make it very entertaining.",
    "What makes the show funny is the situations SpongeBob and Patrick get into. They are very childish which leads to chaos and funny moments. The characters all have strong personalities and work well together.",
    "One of the funniest parts is the parody episodes like Mermaid Man and Barnacle Boy. It's insane enough for kids but funny enough for adults too.",
    "Was the best animated show on Nickelodeon, but newer episodes don't work as well. The older episodes were creative and funny, while newer ones rely on toilet humor and feel predictable.",
    "SpongeBob is a very popular show with quirky humor and colorful characters. It is hard to avoid and can easily cheer you up.",
    "The show focuses on SpongeBob and his adventures in Bikini Bottom with characters like Patrick, Sandy, Squidward, Mr Krabs and Plankton.",
    "SpongeBob has a mix of slapstick comedy and clever humor that appeals to both kids and adults.",
    "Some episodes rely too heavily on repetitive gags which can become tiring.",
    "Despite its flaws, SpongeBob remains a beloved classic with timeless humor and memorable characters.",
    "Old was gold, now it's getting worse. The earlier seasons were amazing but newer ones ruined characters like Patrick and lost the original meaning.",
    "I still watch this show as a teen and enjoy it, but mainly because of the earlier seasons.",
    "Unpopular opinion, but it's just okay. Sometimes the characters are too stupid which makes it hard to watch.",
    "The show is hilarious and weird, but sometimes focuses too much on exaggerated animation and gross humor.",
    "Mindless carefree fun. It's enjoyable watching SpongeBob and his friends go through random adventures.",
    "The perfect cartoon with memorable characters and consistent humor.",
    "Spongebob was great in the first seasons but now feels boring and less funny.",
    "Great show for all ages. It still makes me laugh even after years of watching.",
    "It has gone down in history as one of the most recognizable cartoons ever made.",
    "The humor is random and surreal which makes it unique compared to other shows.",
    "The characters have great chemistry and make the show enjoyable.",
    "It is very nostalgic for people who grew up watching it.",
    "The show is timeless and continues to be enjoyed by new generations.",
    "Some people think it should have ended earlier to preserve quality.",
    "The humor can be childish but still enjoyable.",
    "The world of Bikini Bottom is creative and interesting.",
    "It is easy to rewatch the early seasons multiple times.",
    "Later seasons are less rewatchable for many fans.",
    "The show has a distinct style that sets it apart.",
    "Overall it's a classic show with great moments and noticeable flaws."
]
add_show("SpongeBob SquarePants", spongebob_reviews)

✅ SpongeBob SquarePants predicted as: Balanced
✅ Added 44 reviews for SpongeBob SquarePants


In [104]:
paw_patrol_reviews = [
    "I have been watching this with my daughter for a while and after a few seasons this is what I’ve figured out. There is this rich orphan kid called Ryder who uses his money to build vehicles and fight problems in a city with the help of highly trained dogs. The people in the city often create problems that only Paw Patrol can fix. It’s bizarre but entertaining enough to keep watching.",

    "Loved this show until they made fighting bad guys the main focus. It was better when it was about doing good things for the right reasons. Also enjoy when they introduce new pups, but some characters don’t set a great example.",

    "My son loves this show, but some episodes raise strange questions. For example, when a character’s oven breaks, apparently no one else in the town has a stove. It makes you question how anything works in this world.",

    "Great choice for young kids getting into adventure stories. It has exciting rescues, likeable characters, and teaches problem-solving. It’s not too scary but still engaging. However, parents should be careful where they watch it online due to low-quality or inappropriate content elsewhere.",

    "I find the animation bland and the characters annoying. The show feels repetitive and formulaic, and overall just average compared to other kids shows.",

    "People need to remember this is a cartoon for kids. It teaches teamwork, responsibility, and helping others. It’s a clean and positive show for children.",

    "The show is enjoyable and teaches teamwork and helping others. The main characters are fun, but the adult characters are often immature and annoying, which can take away from the experience.",

    "This is a nonsensical kids show with repetitive plots and annoying characters. It feels like it exists mainly to sell merchandise, though it’s harmless enough for children.",

    "It’s clearly designed for kids and does its job well. It keeps children engaged with simple stories and characters. Adults probably won’t enjoy it much, but kids love it.",

    "I recommend this for young viewers. My child loved it, but I wish there was better female representation among the characters. Still, it’s fun and has good messages.",

    "The show feels low quality and lacks educational value. The stories and character decisions often don’t make sense, but kids seem to enjoy it anyway.",

    "It’s popular because it’s funny and gentle for preschoolers. It includes messages about teamwork and leadership, and has the potential to be a classic children’s show.",

    "The show feels like it exists mainly to sell products. It lacks meaningful lessons, character development, and creativity, making it boring despite decent animation.",

    "It’s a decent show with tolerable characters and some creative ideas, though not outstanding. It’s worth watching casually.",

    "It’s a kids show and works well for its audience. Children enjoy it, and that’s what matters most, even if adults don’t find it appealing.",

    "The show is simple, educational, and entertaining. It teaches teamwork, problem-solving, and kindness through straightforward stories designed for young children.",

    "The show highlights community involvement and communication. It presents characters similar to real-life helpers and teaches kids social responsibility in a fun way.",

    "It’s great for kids but can be tedious for adults. The characters are cute, but the stories are repetitive and lack depth for older viewers.",

    "The episodes are extremely repetitive and predictable. It feels like the creators have run out of ideas, making the show less enjoyable over time.",

    "It’s colorful and entertaining with clear messages about cooperation and responsibility. The characters and gadgets make it fun and engaging for kids.",

    "The show is very repetitive and lacks consequences for bad actions, which may send mixed messages to children.",

    "It’s entertaining and educational with lovable characters. It teaches teamwork and problem-solving in a fun and engaging way for kids.",

    "This show is boring, poorly written, and annoying. It feels like low-quality content that doesn’t deserve its popularity.",

    "The show has creative elements and lovable characters. It’s engaging and memorable for kids, even years later.",

    "It’s generally fun for kids, though some elements of the cast feel inconsistent or odd.",

    "It’s enjoyable for kids and even tolerable for parents. It teaches problem-solving and positive behavior.",

    "The show has poor gender representation and questionable messages about responsibility and authority.",

    "It’s a wholesome and cute show with kind characters and a positive tone, though it has some minor flaws and overexposure through merchandise.",

    "Watching this seemed to negatively affect my child’s behavior. I also dislike the gender representation and the focus on merchandise.",

    "It’s a positive and fun show that teaches kids important lessons in an engaging way.",

    "The show is predictable and formulaic, with repetitive structure and limited creativity.",

    "It’s one of the best children’s shows with strong lessons, adventure, and originality.",

    "The concept of a young boy leading rescue missions with dogs feels unrealistic and may give children odd expectations.",

    "My child loves the show because of the cute animals and humor. It even inspired an interest in science through one of the characters.",

    "The show is boring with too many episodes and weak storytelling.",

    "It’s a funny show with good teamwork lessons and appealing characters for kids.",

    "It’s a fun cartoon, though the premise becomes repetitive over time and stretched across too many episodes.",

    "It’s entertaining and helps me connect with modern children’s media. A good example of current kids’ cartoons."
]

add_show("PAW Patrol", paw_patrol_reviews)

✅ PAW Patrol predicted as: Balanced
✅ Added 38 reviews for PAW Patrol


In [105]:
attack_on_titan_reviews = [
    "I can't put into words the emotions I felt watching this show. It completely took over my life for months. The storytelling, foreshadowing, and attention to detail are unlike anything else. Every moment feels meaningful and contributes to a bigger picture.",
    "This story takes you to places mentally and emotionally you don’t expect. There are no clear heroes or villains, just a complex world where morality is gray. It’s tragic, intense, and deeply thought-provoking.",
    "The show is not meant to be joyful but instead presents a cruel and unfair world. Characters are forced to endure harsh realities, creating a powerful emotional experience.",
    "One of the most intense shows I’ve ever watched. Every episode leaves you on edge with constant twists, strong character connections, and incredible production quality.",
    "The storytelling is layered and constantly evolving. What starts as a simple premise becomes something far deeper and more complex than expected.",
    "The show builds an immersive world with strong emotional impact. The themes are powerful and make you reflect on real-world perspectives of right and wrong.",
    "It keeps getting better with each season, consistently raising the bar. The twists, character development, and world-building are exceptional.",
    "An almost perfect show with incredible storytelling, soundtrack, and character depth. It creates a strong emotional connection with the viewer.",
    "The story is complex and unpredictable, with some of the best plot twists in any series. It’s engaging, intense, and highly immersive.",
    "This show mixes action, emotion, and philosophy in a way that feels realistic and impactful. It challenges the idea of good versus evil.",
    "A masterpiece with intense emotional moments, powerful storytelling, and unforgettable characters. It leaves a lasting impression.",
    "The animation, music, and voice acting are all top-tier. Combined with the story, it creates a truly unforgettable experience.",
    "It starts simple but quickly becomes something much deeper. The brutality and horror elements are handled extremely well.",
    "The show is incredibly engaging, inspiring, and emotionally heavy. It can be both thrilling and heartbreaking at the same time.",
    "There are moments that are genuinely shocking and unpredictable. The story constantly evolves and never feels stagnant.",
    "A deeply emotional and thought-provoking series that explores complex themes and human nature in a unique way.",
    "The pacing and storytelling keep you hooked, with constant tension and high-stakes moments.",
    "It’s one of the most immersive shows ever made, pulling you into its world and making you care deeply about the characters.",
    "The series delivers powerful messages about war, morality, and humanity. It’s both entertaining and meaningful.",
    "An intense, sometimes overwhelming experience with strong emotional highs and lows.",
    "The story is filled with twists, betrayals, and shocking developments that keep you engaged throughout.",
    "It creates a sense of helplessness and fear that few shows can replicate.",
    "The characters are well-developed and feel real, making their struggles more impactful.",
    "The show maintains high quality throughout and rarely loses its momentum.",
    "A brutal yet beautiful story that combines action, horror, and deep philosophical themes.",
    "It’s incredibly rewatchable, with new details and foreshadowing becoming clear each time.",
    "The emotional weight of the story is one of its strongest aspects, often leaving a lasting impact.",
    "Some parts of the finale feel weaker compared to earlier seasons, but overall the story remains strong.",
    "The show can be overwhelming at times due to its intensity and complexity.",
    "Not everyone will enjoy its dark tone and heavy themes, but it stands out for its ambition and execution."
]
add_show("Attack on Titan", attack_on_titan_reviews)

✅ Attack on Titan predicted as: Balanced
✅ Added 30 reviews for Attack on Titan


In [106]:
hells_paradise_reviews = [
    "Hell's Paradise has been an absolute blast from the very 1st episode. It somehow manages to be dark yet gentle at the same time. It's sad yet wholesome at the same time. It's emotional yet full of thrills and action. It's disturbing, yet kind at the same time. Studio Mappa's work is impressive, the art during the action scenes is very appealing, the soundtrack is brilliant.",
    "It's a great anime that won't let you go from the beginning. The protagonist has to find a fruit on an island to survive, but on the way he gets into trouble. The main characters are introduced in detail. This is a great anime that you should not miss.",
    "I truly loved the first season. That almost Lovecraftian horror and the mystery surrounding the deities that govern the island of immortality are of an incredibly high caliber. The fights are fascinating, and seeing main characters die only adds to the excitement.",
    "From the beautiful well crafted opening with an amazing song to the story and the characters themselves - not to mention the out of this world aesthetic to the genre - Hell's Paradise is set to become one of the best anime of recent years.",
    "First half is okay. Second half is AMAZING. This will be the next demon slayer. The storyline seems pretty simple at first but it gets incredibly deep. The animation and the soundtrack are both outstanding.",
    "Hells Paradise is the third of the dark shonen trio that includes Jujutsu Kaisen and Chainsaw Man. The darker and more adult themes have been all the rage and Hell's Paradise delivers beautifully on that front.",
    "I go through a lot of animes and get bored quickly. This one stood out in story, in character, in animation. For those who enjoy blood and battles, this has excellent bouts of actions and gory violence while also being a compelling story.",
    "I was hooked to the anime from the first episode and couldn't wait after 7 episodes and started reading manga. The story is next level and I can't wait for it to be animated, this anime will break the internet when season 2 comes out.",
    "Berserk fans are gonna love this one! Well done, I was hooked from the start, great pacing, loveable heroes, little bit clunky animation but overall, pure enjoyment.",
    "Some of the worst criminals in feudal Japan are offered a pardon if they go on a suicide mission to fetch the elixir of immortality. Super-powered people fighting horrific monsters while trying to survive a lethal island. The show delivers on its premise well.",
    "Hell's Paradise is a masterpiece but not for everyone. This is one of the most balanced anime without any doubt. The story, the characters, the animation - everything is top notch.",
    "Based on the manga of the same name, Hell's Paradise follows Gabimaru the Hollow, an elite assassin sentenced to death who must travel to the deadly Shinsenkyo island to retrieve the elixir of life. The show is beautifully animated and emotionally engaging.",
    "Hell's Paradise sets the stage for a treacherous journey. The intense action scenes are very well done and the character development is outstanding. This is easily one of my favorite anime.",
    "Although I've only watched 9 episodes, I can easily say this is one of my favorite anime. The characters all have personalities that the voice actors do VERY well at. The intense action scenes are very well animated. The world-building is incredible.",
    "I really like the drawing in this anime. All the characters perfectly reveal themselves in the course of the plot. I'm really looking forward to the continuation of this fascinating story.",
    "Looking forward to the second season. Such a beautiful anime. The story telling has been brilliant and there is still so much to explore. Each character has a great backstory and I want to see where they all end up.",
    "Hell's Paradise Season 2 feels deeply human because it slows down and lets the characters breathe, fear, and reflect. Instead of focusing only on action, it prioritizes emotional truth and moral weight.",
    "Definitely solid and entertaining, despite its flaws. I feel like the animation wasn't as good as it could've been and some plot points were rushed. But overall a very enjoyable watch.",
    "This was a very good, well-written anime. I liked the fact that every character has their own motivations and backstory. The world building is exceptional and the action sequences are thrilling.",
    "It's a really fun idea sending murderers to an island and just stuff happens. It's fun, violent and engaging. The characters are all memorable and the island setting creates a genuinely threatening atmosphere.",
    "You can't go wrong with this anime. Don't think, just watch it! Looking at the cover page of the very first chapter feels like opening the gate to Hell, matching perfectly with its title.",
    "It's a great new anime. You find yourself wondering what happens next. The island they land on raises many questions. The story develops in a very interesting way with great character interactions.",
    "Hell's Paradise is a gripping and visually stunning anime that delves deep into the abyss of human nature, spirituality, and the relentless pursuit of survival. The artwork is breathtaking and the storyline is compelling.",
    "It's a MAPPA series, so there's a certain quality to it. Physical and psychological violence, philosophical dialogues, magnificent fights, an interesting universe to explore. Highly recommended.",
    "I love watching new anime and I've never had an anime that has me wanting to call a sick day at work just to watch it. Great characters, great action, great story. Absolutely brilliant.",
    "Hell's Paradise tells the story of the legendary shinobi Gabimaru who was sentenced to death. His only hope for pardon is to find the elixir of life on a remote island. The premise is executed brilliantly with stunning visuals and emotional depth.",
    "This is the most interesting anime I have seen to this date. Every character is a piece of philosophy and the tempo is good. The world building and lore are exceptional.",
    "A Dark, Thrilling Journey of Redemption and Survival. Hell's Paradise follows Gabimaru, a skilled assassin given a chance at redemption. The show delivers intense action alongside surprisingly deep emotional moments.",
    "Unlike some overhyped anime with a basic plot, this feels different. The animation is beautiful and the story is gripping from the very first episode. One of the best anime I have watched.",
    "It's a super cliché anime with below average animation and an ok story. There are a lot of fight skips and obvious reuse of animation. The story and character development don't make up for these shortcomings.",
    "Hell's Paradise is a dark action anime with intense fights and deep emotions. The story follows Gabimaru, a deadly ninja who wants to reunite with his wife. The emotional core of the show is surprisingly touching.",
    "MAPPA's brutal survival thriller delivers visceral action and striking animation. The death-row convict premise creates genuine tension and moral ambiguity, with Gabimaru's internal conflict anchoring the entire show.",
    "Massy and classy anime with great emotional depth. Instead of nonstop action, the direction emphasizes character development and emotional resonance. Every episode is crafted with care.",
    "Nice idea, but basically quite mediocre, although it had potential. The emotional moments don't come across well. We get the backstory but the emotional payoff feels rushed and underdeveloped.",
    "With so many animes these days, the bar for originality is becoming higher. Hell's Paradise puts that originality to the test. It starts really interesting but loses some momentum in the middle.",
    "Hells Paradise has Tao, a power system inspired by Nen from Hunter x Hunter, antagonist designs similar to Demon Slayer and the darker aspects of Chainsaw Man and JJK. A great blend of influences that creates something unique.",
]

add_show("Hell's Paradise", hells_paradise_reviews)

✅ Hell's Paradise predicted as: Balanced
✅ Added 36 reviews for Hell's Paradise


In [107]:
gumball_reviews = [
    "I've watched some episodes of The Amazing World of Gumball and it's absolutely brilliant. This is one of the best cartoons I've ever watched. It's also very funny and bizarre. My favourite character is Richard Watterson because he is the funniest character I've ever met. I'm nearly 27 years old and I'm still watching this cartoon.",
    "The Amazing World of Gumball is one of those shows I looked at and said how can anyone like that. But like other shows on Cartoon Network it surprised me to see it was actually a well written show. What I find really appealing about this show are the characters. Each character has a different animation style. The family is pretty close to real life too.",
    "This show is hilarious in many ways. It's a children's show that somehow reaches older ages as well. Characters are created in many ways such as traditional animation, claymation, stop-motion, CGI, pixilation and even puppets. The show has made several references to other things as well. The show is unbelievably cute at times as well as so funny that my parents have even laughed at it.",
    "I am a 42 year old man and this show makes me lol consistently. This show is clearly written for me. There are a lot of jokes that will fly directly above the heads of its target audience and at least make me crack a smile, usually laugh, and sometimes raise an eyebrow.",
    "The Amazing World of Gumball is you know AMAZING! I love the characters. Every time Gumball and Darwin scream I always have to laugh because they scream like little girls. Every time I'm bored I always have to watch this show.",
    "Not only is the animation an amazing blend of everything but this show is really funny. Gumball humor can go all over the place and still be really funny. Gumball and Darwin are a great duo and work off each other pretty well. The dad Richard is really a cut above the rest despite the dumb dad cliche being way too common.",
    "This show is one of the apples of our time. It has some sophisticated humor that makes us laugh and it's on Cartoon Network and it's current. It's Adventure Time and Phineas and Ferb combined. Something new, something fresh.",
    "The Amazing World of Gumball is unique, harmless, entertaining and like nothing you've ever seen before. Everything about it is unique with funny characters, harmless humor that is almost never crass yet still hilarious and inventive stories that are a mix of surrealism, drama and over-the-top hilarity.",
    "This show is great. It strives for improvement. There are so many references and jokes that are directed to older audience members. This show has feelings and if you have watched it from the beginning it really feels like you have grown up with Gumball. The jokes have grown up, the animation has become much better and the character development has improved a ton.",
    "I saw the commercial for this show and was hoping it wasn't going to turn out bad. But surprisingly it was quite good. It has laughs, weird yet entertaining stories and funny characters. The voice acting was great and hilarious. The parodies of movies and real life are funny and random.",
    "The Amazing World of Gumball is not very good. The show had some decent potential but it turned out to be another dull show with modern kids show clichés. The animation is the only thing really going for this show. Most of the jokes either involve toilet humor, cheap slapstick or randomly shouting things.",
    "After previously watching the pilot I was amazed by it. I thought the mix of 2D and 3D was amazing and I have never seen any other TV show do that. This show is amazing. Acting, story, characters, all of it just bonkers. Every episode differs in jokes and story lines so you never get bored.",
    "The Amazing World of Gumball is a great show overall however sometimes there will be episodes that down right suck. The best part about the show is its humor. I have not seen a show with that much slapstick humor since Ed Edd and Eddy. Regular Show, Adventure Time and this show are the only ones I can tolerate.",
    "While shows like Steven Universe are more popular on Cartoon Network I find this to be better. You get 2D animation, CGI, claymation and live action all in one gorgeous package. The characters may have the most distinct designs of any cartoon characters. The characters are amazing and act like real kids.",
    "This show is great and it's funny with lovable characters and fun plots. One of the very few cartoons that have a justified high rating here.",
    "Cartoon Network are not as good as they were back in the day but there are gems like Regular Show and Adventure Time. While not without its faults The Amazing World of Gumball is one of Cartoon Network's better shows. The animation is colourful and wonderfully surreal. The characters are admittedly cliché but they're funny, quirky and cute.",
    "Gumball is a fun series with an extremely intelligent style of comedy sometimes, other times the comedy can be the most basic. I can recommend many episodes but many others are better avoided due to their high level of stupidity. Its satire of today's society is simply delicious.",
    "This is my absolute favorite show because of its funny characters and stories. Most of it is great but there are some episodes that can be a bit hard to watch because of intended cringe or second-hand embarrassment. This show has a tiny bit of adult humor but it's definitely a kids show.",
    "Few if any animated shows have made me notice the job done by the voice actors than The Amazing World of Gumball. This is truly a funny show and not just at a level for little kids. I never tire of this show. If it's airing I have it on regardless of how many times I've seen the episode.",
    "The Amazing World of Gumball takes place in a mystical place known as Elmore where cartoons of all shapes sizes and even styles live in harmony. Beneath all of the appearances this show is surprisingly really well-made and knows what it's doing with its jokes and its plot. The world of Elmore has a ton of creativity in its cast of characters.",
    "Like the best of kids shows today this show is extremely enjoyable and entertaining for both kids and adults. Many in-jokes and sometimes whole episodes are clearly geared towards the adults while keeping the comedy easy for kids to enjoy. Everyone in my household loves this show.",
    "The show uses sarcasm, satire, tragedy and other literary tools for social commentary. If you like Bill Maher you might like this. It's progressive and forward-thinking and a lot of the jokes and references are actually for adults. One episode plainly states that it's not the content of a source of entertainment that makes something good or bad it's the way we teach our children to respond to it.",
    "The Amazing World of Gumball is for the lack of a better word perfect. It's playing with codes both visually with the 2 and 3D mix but also in terms of scenarios. Season 1 is a good starter but it starts to get really funny from season 2 and then improves. It's filled with adult non-sexual references.",
    "This cartoon is absolutely a masterpiece. The love you feel for this masterpiece is immense and the cartoon succeeds in the arduous task of making you passionate and fall in love with all the characters in the series. It is practically impossible not to recognise oneself in at least one of the characters.",
    "This is quite possibly my favorite show on Cartoon Network. From the excellent visuals to the clever humor to the well thought out plot this show makes me love cartoons again even as a teenager. Well played Cartoon Network keep going with the original content.",
    "This show is excellent, super-creative animation, jam packed with fun pop-culture references. This show is meta and super smart. As far as children's shows go this is one of the best.",
    "The Amazing World of Gumball is one of the best shows in animation that's out there and possibly of all time. The humor is always funny and it isn't random equals funny type of humor. The show writers make modern humor actually funny and not cringey.",
    "This is easily one of the most clever and creative Cartoon Network cartoons ever. The episode plots are usually pretty simple but they are executed in such creative and entertaining ways. The characters are witty, the jokes are hits, the show's pacing is perfect and the character designs are all over the place which is awesome.",
    "I give the show a seven through its entire run due to some of the worse episodes and season 1 nearly entirely. Otherwise this show is beautiful with all the animation styles meshing well with the live-action backgrounds. It also has quality in its scripts, characters, character arcs and voice acting.",
    "This show is amazing. It is so adult themed and witty. It deserves more attention from people. The PG-13 version of South Park I like to say. While its main appeal is for kids it has some hidden social topic references that adults will relate to and laugh out loud at.",
]

add_show("The Amazing World of Gumball", gumball_reviews)

✅ The Amazing World of Gumball predicted as: Balanced
✅ Added 30 reviews for The Amazing World of Gumball


In [108]:
adventure_time_fionna_cake_reviews = [
    "I remember watching the pilot for Adventure Time back when I was just seven years old in 2010. The new series is really good. I honestly feel like I'm 7 years old again witnessing a new show for the first time. But the show grew up with all of us. It's honestly something that I should cherish. I'm happy that I get to watch a sequel to this show. I'm currently 20 years old and I am happy that it's still relevant.",
    "I love this. Fionna and Cake is everything we loved from Adventure Time condensed and elevated, bloodier with more personal and heavy themes to keep up with the aging audience, and a familiar cast of characters in a new light. The humor is on point, the action is crisp, and I love the new directions they're exploring with the multiverse. A hit series better than its predecessor is in the making.",
    "Adventure Time: Fionna and Cake completely blew me away! Right from the very first scene, Fionna's dream sequence had amazing animation that set the tone for the rest of the show. What I really loved was how Fionna and Cake aren't just fictional characters written by the Ice King anymore - they're actually traveling across the multiverse. This series perfectly balances nostalgia, humor, emotion, and mature storytelling.",
    "I was a bit skeptical when they announced there's going to be a spinoff of my much beloved show Adventure Time. I honestly kept my expectations low but this show exceeded them. It really captures the essence of the original show while reaching new heights with the level of animation, writing, voice acting and pacing. I am so glad it didn't get cancelled.",
    "I absolutely love this series! Every character in this series is now more nuanced and mature. I love exploring what has happened during the time we've been absent from this series. It's rare to see a show grow with its audience and also very rare for a show to be so in touch with its fans' opinions and desires as Fionna and Cake is.",
    "This is getting really difficult to watch as a lot of intense emotions surface watching these episodes and I can't remember a show that ever felt like that, let alone for a reboot. The writers know how deep the imprint of Adventure Time still is in our soul so they genuinely sparkled that rebooted world with everything that made it successful.",
    "It's so cool that there's more Adventure Time! This is really good so far. Grown up Finn is so cool and funny! It's cool to me that the story arcs have gotten longer overall. Now this series builds on all of them and looks to be a continuing arc. So excited to see where it goes!",
    "AT is my favourite show of all time. It was truly heartfelt, creative, funny, crazy, inspiring, beautiful art. However this to me felt in comparison bland. Less profound, less earnest, more self-referential, a bit more cynical. Less funny. Relying too much on nostalgia. I'm sad to say that although not bad this didn't do it for me. It just felt uninspired.",
    "I'm loving it. The first 2 episodes that came out are a total banger and they differ a bit from the Adventure Time episode formula but still keep it amazing. The show is rated TV-14 and what makes it more adult are the themes in the show.",
    "Having been a huge fan of Adventure Time since I was 12 this one is pretty nostalgic to me. Adventure Time: Fionna and Cake tells the story of the two titular characters who get stuck in the multiverse. The creators are never afraid to test out their abilities to further expand the stories and how literally anything could happen.",
    "Fionna and Cake is my new favorite series. Growing up I watched Adventure Time from the age of 12 and it was an absolute marvel. When the series ended I craved more. This spin-off is amazing and near a masterpiece for sure. I got the hype for this animated series because it's so good.",
    "For a massive Adventure Time fan I am awestruck how picture perfect the series has continued the story of the world and characters we love. It has gone one step further and placed the main character Fionna into a more relatable world from the beginning. This progression of the Adventure Time world continues to grow with us further into adulthood.",
    "As an Adventure Time fan I genuinely couldn't be happier. This show was perfect beginning to end. Everything about Simon and Betty was incredible, moving and exceptionally handled. The action was amazing, the character and environment design was amazing, the stakes were high as hell. For me this is just at the same level as Adventure Time.",
    "I can't believe this is here. Adventure Time is one of the greatest TV shows to ever exist. While it doesn't hold all of the original charm of the original series it is bringing its own flavor to everything you loved about Adventure Time in an extremely clever way. Each episode gets better and better so far. The nostalgia is almost overwhelming.",
    "This is modern 2D animation for my generation. We grew up on Adventure Time so it's only natural that the series matures with us. The atmosphere is noticeably more depressing but there's still an abundance of thought-provoking material and profound revelations in Fionna and Cake.",
    "Nothing but a 10/10. As an Adventure Time fan I genuinely couldn't be happier. The treatment of the characters was by far the best thing about this show, the amount of care and love that the makers put into this show is absolutely astonishing. Everything about Simon and Betty was incredible, moving and exceptionally handled.",
    "I adore Adventure Time with Finn and Jake. I've rewatched it more times than any other series. No other series more perfectly exemplifies the dwarfing isolating feeling one gets when coming to grips with the vast expanse of the universe. Fionna and Cake is the funniest, most imaginative, most joyous exploration of the multiverse to rival Everything Everywhere All At Once.",
    "I have been a fan of Adventure Time and was devastated back when it ended. All 10 episodes of Fionna and Cake give the Adventure Time feel that 3 out of 4 episodes of Distant Lands didn't give me. No episodes had any parts that weren't good. Fionna and Cake gave me a feeling I have not felt from a show in a while.",
    "I made an account just to write this review. I can't even begin to express how much joy this show gave me. This is the adult show AT always was at heart. I started watching the original back in 2012 when I was only 14 and I'll be 26 soon. The references and callbacks you never thought were going to be mentioned ever again were cleverly written into Fionna and Cake.",
    "If you're an Adventure Time fan Fionna and Cake is an absolute must-watch. Nostalgia runs deep. It's like revisiting your favorite childhood memories with a fresh twist. What truly sets it apart is its extraordinary visual appeal. The colors are bold and vibrant and the character designs are nothing short of spectacular.",
    "I enjoyed watching this for the most part but it often felt like watered down Adventure Time fan fiction. Adventure Time is one of my favorite animated shows. Fionna and Cake however seems to be lacking some of that soul and imagination. It's not totally without it by any means but it isn't able to match Adventure Time.",
    "Really enjoyed this series. It was very relatable and every episode left me wanting more. I'm so glad we finally got a season of Fionna and Cake - it was more than I could ever hope for. They did wrap up the first season beautifully and it was such a fantastic series.",
    "Fionna and Cake is best described as perfect. It fleshed out the already huge world of Adventure Time even more without harming the continuity of the original. It even went so far as to answer questions left empty about the original Adventure Time. There is absolutely nothing wrong with this show. Honestly just flawless and impeccable writing and animation.",
    "This is a truly wonderful and incredible series. A great adventure with very well-written characters that you can identify with as well as the situations they face. I also feel nostalgic when I see the original characters again. All of the characters in this series have become more nuanced and mature.",
    "I love that they made the adult content exceptionally well. There are brutal scenes but they are treated with the weight they should be treated with. This show is not for kids and it makes sense since the people who watched Adventure Time as kids are now adults. They are in for a treat.",
    "The show is AMAZING. Lots of hanging threads for Simon are tied up from the original series. The interactions feel genuine and the relationships are so cute. The way that Fionna interacts with the rest of the characters and how the authors built the characters' backstories feels like a breath of fresh air.",
    "As a lifelong Adventure Time fan I was pretty excited for Fionna and Cake. This show managed to surpass my already high expectations. Especially Simon Petrikov the former Ice King. I thought Simon was the standout character in this show and his storyline was the most engaging to me. Overall I absolutely loved Fionna and Cake.",
    "I cannot describe how happy I am with the way this show turned out. Not only did the original cast come back but the writers managed to pull off such a captivating new story based on the original lore. They managed to condense a plethora of content into such a short runtime. This is a masterpiece and I love it.",
    "They did a great job of capturing the weirdness of the original series but it's a depressed and miserable version. We used to watch Adventure Time as an escape to laugh and feel joy. This show is just depressing. To have the show itself say that the old Adventure Time can't help anymore is ridiculous. This isn't the same as the show you loved. It's a weird different bummer version.",
    "Holy cow this is so good. Adventure Time changed my life. I've been watching and rewatching since it came out. I've wanted a sequel for years but Distant Lands was such a letdown, it just didn't have the AT spirit. This is full of it. I really hope they're able to get funding for more seasons so one of the best parts of my childhood can continue to extend into my adulthood.",
    "This show is simply wonderful. The writing, performances, animation and story are all top-notch. It feels like the perfect follow-up to Adventure Time and I appreciate that it leans into more mature themes than the original series. I love getting to catch up with some old Ooo favorites but also see into the lives of their gender-swapped variants.",
    "This end so well, absolutely beautiful. One of the best animated series I've ever seen. After all these years watching Adventure Time since childhood, now we all grow and the meaning of the series grows too. The humor, the dialogue, the characters, everything is more meaningful. I hope we get more like this in a few years.",
]

add_show("Adventure Time: Fionna & Cake", adventure_time_fionna_cake_reviews)

✅ Adventure Time: Fionna & Cake predicted as: Balanced
✅ Added 32 reviews for Adventure Time: Fionna & Cake


In [109]:
import pandas as pd

df = pd.read_csv('tv_reviews/imdb_tvshows.csv')

manual_moods = {
    'Andor': 'Balanced',
    'The Last of Us': 'Heavy',
    "The Handmaid's Tale": 'Heavy',
    'Yellowstone': 'Intense',
    'Reacher': 'Intense',
    'Peaky Blinders': 'Intense',
    'The White Lotus': 'Balanced',
    'Wednesday': 'Positive',
    'The Mandalorian': 'Positive',
    'The Witcher': 'Intense',
    'The Boys': 'Intense',
    'Game of Thrones': 'Heavy',
    'The Vampire Diaries': 'Intense',
    'Invincible': 'Intense',
    'One Piece': 'Positive',
    'Pokemon': 'Positive',
    "Hell's Paradise": 'Heavy',
    'Naruto': 'Positive',
    'Moana': 'Positive',
}

for show, mood in manual_moods.items():
    df.loc[df['show_title'] == show, 'mood_category'] = mood

df.to_csv('tv_reviews/imdb_tvshows.csv', index=False)
print(df.groupby(['show_title', 'mood_category']).size())

show_title                     mood_category
Adventure Time: Fionna & Cake  Balanced          30
                               Positive           2
Andor                          Balanced         413
Attack on Titan                Balanced          28
                               Positive           2
                                               ... 
Vikings                        Positive           1
Wednesday                      Positive         992
Yellowstone                    Intense          854
You                            Balanced          36
                               Positive           6
Length: 72, dtype: int64


In [110]:
#10
import pandas as pd

df = pd.read_csv('tv_reviews/imdb_tvshows.csv')
df = df.dropna(subset=['show_title'])

show_summaries = []
for show in df['show_title'].unique():
    show_df = df[df['show_title'] == show]
    show_summaries.append({
        'title': show,
        'mood_category': show_df['mood_category'].iloc[0] if 'mood_category' in show_df.columns else 'Unknown',
        'review_count': len(show_df),
        'avg_sentiment': show_df['sentiment_compound'].mean() if 'sentiment_compound' in show_df.columns else 0,
        'sample_reviews': show_df['review_text'].sample(min(3, len(show_df))).tolist()
    })

pd.DataFrame(show_summaries).to_csv('show_summaries.csv', index=False)
print("✅ show_summaries.csv updated!")

✅ show_summaries.csv updated!


In [111]:
%%writefile dashboard.py

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(
    page_title="TV Show Mood Impact Analyser",
    page_icon="TV",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── Custom CSS ────────────────────────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:wght@300;400;500;600&display=swap');

/* ── Base ── */
html, body, [class*="css"] {
    font-family: 'DM Sans', sans-serif;
}

.stApp {
    background-color: #0f1117;
    color: #e8e8e8;
}

/* ── Sidebar ── */
[data-testid="stSidebar"] {
    background-color: #161b27;
    border-right: 1px solid #2a2f3e;
}

[data-testid="stSidebar"] .stRadio label {
    color: #b0b8cc !important;
    font-size: 0.9rem;
    padding: 6px 0;
}

[data-testid="stSidebar"] h1 {
    color: #ffffff !important;
    font-family: 'DM Serif Display', serif !important;
    font-size: 1.3rem !important;
    letter-spacing: 0.02em;
    margin-bottom: 1.5rem;
}

/* ── Headings ── */
h1 {
    font-family: 'DM Serif Display', serif !important;
    font-size: 2.6rem !important;
    font-weight: 400 !important;
    color: #ffffff !important;
    letter-spacing: -0.02em;
    line-height: 1.15 !important;
}

h2 {
    font-family: 'DM Serif Display', serif !important;
    font-size: 1.6rem !important;
    font-weight: 400 !important;
    color: #d4d8e8 !important;
}

h3 {
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 600 !important;
    font-size: 1rem !important;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    color: #8892a4 !important;
}

/* ── Category Cards ── */
.category-card {
    border-radius: 12px;
    padding: 20px 24px;
    margin-bottom: 12px;
    border-left: 4px solid;
    background: #161b27;
}
.card-positive  { border-color: #4ecdc4; background: linear-gradient(135deg, #161b27 60%, #0d2e2c); }
.card-balanced  { border-color: #5b9bd5; background: linear-gradient(135deg, #161b27 60%, #0d1e2e); }
.card-intense   { border-color: #f4a261; background: linear-gradient(135deg, #161b27 60%, #2e1e0d); }
.card-heavy     { border-color: #e05c6f; background: linear-gradient(135deg, #161b27 60%, #2e0d15); }

.card-title {
    font-family: 'DM Serif Display', serif;
    font-size: 1.15rem;
    font-weight: 400;
    color: #ffffff;
    margin-bottom: 4px;
}
.card-desc {
    font-size: 0.85rem;
    color: #8892a4;
    margin: 0;
}

/* ── Mood Badge ── */
.mood-badge {
    display: inline-block;
    padding: 6px 18px;
    border-radius: 20px;
    font-size: 0.8rem;
    font-weight: 600;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    margin-bottom: 16px;
}
.badge-Positive { background: #0d2e2c; color: #4ecdc4; border: 1px solid #4ecdc4; }
.badge-Balanced { background: #0d1e2e; color: #5b9bd5; border: 1px solid #5b9bd5; }
.badge-Intense  { background: #2e1e0d; color: #f4a261; border: 1px solid #f4a261; }
.badge-Heavy    { background: #2e0d15; color: #e05c6f; border: 1px solid #e05c6f; }

/* ── Show Title ── */
.show-title {
    font-family: 'DM Serif Display', serif;
    font-size: 2rem;
    color: #ffffff;
    margin-bottom: 8px;
}

/* ── Stat Box ── */
.stat-box {
    background: #161b27;
    border: 1px solid #2a2f3e;
    border-radius: 10px;
    padding: 18px 20px;
    text-align: center;
}
.stat-number {
    font-family: 'DM Serif Display', serif;
    font-size: 2rem;
    color: #ffffff;
    line-height: 1;
}
.stat-label {
    font-size: 0.75rem;
    color: #8892a4;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    margin-top: 4px;
}

/* ── Info Panel ── */
.info-panel {
    background: #161b27;
    border: 1px solid #2a2f3e;
    border-radius: 12px;
    padding: 24px 28px;
    margin-top: 16px;
}
.info-panel p {
    color: #b0b8cc;
    font-size: 0.9rem;
    line-height: 1.7;
    margin: 0;
}

/* ── Divider ── */
.custom-divider {
    border: none;
    border-top: 1px solid #2a2f3e;
    margin: 24px 0;
}

/* ── Show Row ── */
.show-row {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 12px 16px;
    background: #161b27;
    border-radius: 8px;
    margin-bottom: 6px;
    border: 1px solid #2a2f3e;
}
.show-row-title {
    font-weight: 500;
    color: #e8e8e8;
    font-size: 0.9rem;
}
.show-row-meta {
    font-size: 0.8rem;
    color: #8892a4;
}

/* ── Metric overrides ── */
[data-testid="stMetric"] {
    background: #161b27;
    border: 1px solid #2a2f3e;
    border-radius: 10px;
    padding: 16px 20px;
}
[data-testid="stMetricLabel"] { color: #8892a4 !important; font-size: 0.75rem !important; text-transform: uppercase; letter-spacing: 0.06em; }
[data-testid="stMetricValue"] { color: #ffffff !important; font-family: 'DM Serif Display', serif !important; font-size: 1.8rem !important; }

/* ── Selectbox ── */
.stSelectbox > div > div {
    background: #161b27 !important;
    border: 1px solid #2a2f3e !important;
    color: #e8e8e8 !important;
    border-radius: 8px !important;
}

/* ── Button ── */
.stButton > button {
    background: #ffffff !important;
    color: #0f1117 !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
    font-size: 0.85rem !important;
    letter-spacing: 0.04em !important;
    padding: 10px 28px !important;
    transition: opacity 0.2s !important;
}
.stButton > button:hover {
    opacity: 0.85 !important;
}

/* ── Expander ── */
.streamlit-expanderHeader {
    background: #161b27 !important;
    border: 1px solid #2a2f3e !important;
    border-radius: 8px !important;
    color: #b0b8cc !important;
}

/* ── Alert boxes ── */
.stSuccess, .stInfo, .stWarning, .stError {
    border-radius: 10px !important;
    border-left-width: 4px !important;
}

/* ── Radio buttons ── */
.stRadio > div { gap: 4px !important; }

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-track { background: #0f1117; }
::-webkit-scrollbar-thumb { background: #2a2f3e; border-radius: 3px; }
</style>
""", unsafe_allow_html=True)

# ── Data ─────────────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    shows = pd.read_csv('show_summaries.csv')
    return shows

shows_df = load_data()

category_colors = {
    'Positive': '#4ecdc4',
    'Balanced': '#5b9bd5',
    'Intense':  '#f4a261',
    'Heavy':    '#e05c6f'
}

category_descriptions = {
    'Positive': 'Uplifting content that supports emotional wellbeing and stress relief.',
    'Balanced': 'Mixed emotional content — engaging but manageable for most viewers.',
    'Intense':  'Emotionally demanding viewing. Watch when you have the energy for it.',
    'Heavy':    'Dark, challenging content. Viewer discretion advised.',
}

# ── Sidebar ───────────────────────────────────────────────────────────────────
st.sidebar.title("Mood Impact Analyser")
st.sidebar.markdown("<hr style='border-color:#2a2f3e; margin: 8px 0 20px 0'>", unsafe_allow_html=True)
page = st.sidebar.radio("", ["Home", "Analyse Show", "All Shows", "Why It Matters", "About Model"])

st.sidebar.markdown("<hr style='border-color:#2a2f3e; margin: 20px 0 16px 0'>", unsafe_allow_html=True)
st.sidebar.markdown("""
<div style='font-size:0.75rem; color:#4a5568; line-height:1.6;'>
    <strong style='color:#6b7280'>Dataset</strong><br>
    8,385 reviews &bull; 50 shows<br><br>
    <strong style='color:#6b7280'>Model</strong><br>
    Random Forest &bull; 77% accuracy<br><br>
    <strong style='color:#6b7280'>Categories</strong><br>
    Positive &bull; Balanced &bull; Intense &bull; Heavy
</div>
""", unsafe_allow_html=True)

# ── HOME ─────────────────────────────────────────────────────────────────────
if page == "Home":
    st.markdown("<h1>TV Show<br><em>Mental Health Impact Analyser</em></h1>", unsafe_allow_html=True)
    st.markdown("<hr style='border-color:#2a2f3e; margin: 20px 0 28px 0'>", unsafe_allow_html=True)

    st.markdown("""
    <div class='info-panel'>
    <p>This tool uses machine learning to classify TV shows by their likely emotional and mental health impact on viewers.
    Unlike standard recommendation systems that focus on entertainment quality, this classifier analyses thousands of
    authentic viewer reviews to predict how a show might affect your <strong style='color:#e8e8e8'>mood and mental wellbeing</strong>.</p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    col1, col2, col3 = st.columns(3)
    with col1:
        st.markdown(f"""
        <div class='stat-box'>
            <div class='stat-number'>{len(shows_df)}</div>
            <div class='stat-label'>Shows Analysed</div>
        </div>""", unsafe_allow_html=True)
    with col2:
        st.markdown("""
        <div class='stat-box'>
            <div class='stat-number'>8,385</div>
            <div class='stat-label'>Reviews Processed</div>
        </div>""", unsafe_allow_html=True)
    with col3:
        st.markdown("""
        <div class='stat-box'>
            <div class='stat-number'>77%</div>
            <div class='stat-label'>Model Accuracy</div>
        </div>""", unsafe_allow_html=True)

    st.markdown("<br><h3>Mood Impact Categories</h3><br>", unsafe_allow_html=True)

    col1, col2 = st.columns(2)
    with col1:
        st.markdown("""
        <div class='category-card card-positive'>
            <div class='card-title'>Positive</div>
            <p class='card-desc'>Feel-good content, uplifting narratives, and stress relief. Safe for most emotional states.</p>
        </div>
        <div class='category-card card-intense'>
            <div class='card-title'>Intense</div>
            <p class='card-desc'>High tension and emotional investment. Gripping but potentially exhausting.</p>
        </div>
        """, unsafe_allow_html=True)
    with col2:
        st.markdown("""
        <div class='category-card card-balanced'>
            <div class='card-title'>Balanced</div>
            <p class='card-desc'>Mixed emotional tones. Emotionally engaging but manageable for most viewers.</p>
        </div>
        <div class='category-card card-heavy'>
            <div class='card-title'>Heavy</div>
            <p class='card-desc'>Dark themes and heavy emotional toll. May not be suitable when feeling vulnerable.</p>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("""
    <div style='background:#161b27; border:1px solid #2a2f3e; border-radius:10px; padding:16px 20px;'>
        <p style='color:#8892a4; font-size:0.82rem; margin:0;'>
        Use the <strong style='color:#b0b8cc'>sidebar</strong> to search for a specific show, browse all classifications, or learn more about how the model works.
        </p>
    </div>
    """, unsafe_allow_html=True)

# ── ANALYZE SHOW ─────────────────────────────────────────────────────────────
elif page == "Analyse Show":
    st.markdown("<h1>Analyse a Show</h1>", unsafe_allow_html=True)
    st.markdown("<hr style='border-color:#2a2f3e; margin: 16px 0 28px 0'>", unsafe_allow_html=True)

    show_list = sorted(shows_df['title'].tolist())
    selected_show = st.selectbox("Select a TV show:", show_list)

    if st.button("Analyse This Show", type="primary"):
        show_data = shows_df[shows_df['title'] == selected_show].iloc[0]
        category = show_data['mood_category']
        color = category_colors.get(category, '#ffffff')
        desc = category_descriptions.get(category, '')

        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown(f"<div class='show-title'>{selected_show}</div>", unsafe_allow_html=True)
        st.markdown(f"<div class='mood-badge badge-{category}'>{category}</div>", unsafe_allow_html=True)

        st.markdown(f"""
        <div style='background:#161b27; border-left:4px solid {color}; border-radius:0 10px 10px 0;
                    padding:16px 20px; margin-bottom:24px;'>
            <p style='color:#b0b8cc; font-size:0.9rem; margin:0;'>{desc}</p>
        </div>
        """, unsafe_allow_html=True)

        col1, col2, col3 = st.columns(3)
        with col1:
            st.metric("Reviews Analysed", f"{int(show_data['review_count']):,}")
        with col2:
            sentiment = show_data['avg_sentiment']
            st.metric("Avg Sentiment Score", f"{sentiment:.2f}")
        with col3:
            st.metric("Classification", category)

        st.markdown("<br>", unsafe_allow_html=True)

        # Sentiment bar
        sent_val = float(show_data['avg_sentiment'])
        sent_pct = int((sent_val + 1) / 2 * 100)
        st.markdown("<h3>Sentiment Profile</h3>", unsafe_allow_html=True)
        st.markdown(f"""
        <div style='margin: 12px 0 24px 0;'>
            <div style='display:flex; justify-content:space-between; margin-bottom:6px;'>
                <span style='font-size:0.8rem; color:#8892a4;'>Negative</span>
                <span style='font-size:0.8rem; color:#8892a4;'>Positive</span>
            </div>
            <div style='background:#2a2f3e; border-radius:4px; height:8px; overflow:hidden;'>
                <div style='width:{sent_pct}%; height:100%; background:{color}; border-radius:4px; transition:width 0.5s;'></div>
            </div>
            <div style='text-align:center; margin-top:6px;'>
                <span style='font-size:0.8rem; color:#8892a4;'>Score: {sent_val:.3f}</span>
            </div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("<h3>Viewer Reviews</h3>", unsafe_allow_html=True)
        import ast
        try:
            reviews = ast.literal_eval(show_data['sample_reviews'])
        except:
            reviews = [show_data['sample_reviews']]

        for i, review in enumerate(reviews[:3], 1):
            with st.expander(f"Review {i}"):
                st.write(review[:500] + "..." if len(str(review)) > 500 else review)

# ── ALL SHOWS ─────────────────────────────────────────────────────────────────
elif page == "All Shows":
    st.markdown("<h1>All Shows</h1>", unsafe_allow_html=True)
    st.markdown("<hr style='border-color:#2a2f3e; margin: 16px 0 28px 0'>", unsafe_allow_html=True)

    category_counts = shows_df['mood_category'].value_counts()

    fig = go.Figure(data=[
        go.Bar(
            x=category_counts.index.tolist(),
            y=category_counts.values.tolist(),
            marker_color=[category_colors.get(c, '#888') for c in category_counts.index],
            marker_line_width=0,
        )
    ])
    fig.update_layout(
        plot_bgcolor='#161b27',
        paper_bgcolor='#161b27',
        font=dict(family='DM Sans', color='#8892a4', size=12),
        xaxis=dict(gridcolor='#2a2f3e', showline=False, tickfont=dict(color='#b0b8cc')),
        yaxis=dict(gridcolor='#2a2f3e', showline=False, tickfont=dict(color='#b0b8cc')),
        margin=dict(l=20, r=20, t=20, b=20),
        height=280,
        showlegend=False,
    )
    st.plotly_chart(fig, use_container_width=True)

    st.markdown("<hr style='border-color:#2a2f3e; margin: 8px 0 24px 0'>", unsafe_allow_html=True)

    for category in ['Positive', 'Balanced', 'Intense', 'Heavy']:
        shows_in_cat = shows_df[shows_df['mood_category'] == category]
        if len(shows_in_cat) > 0:
            color = category_colors[category]
            st.markdown(f"""
            <div style='display:flex; align-items:center; gap:10px; margin-bottom:14px;'>
                <div style='width:4px; height:20px; background:{color}; border-radius:2px;'></div>
                <span style='font-family:DM Serif Display, serif; font-size:1.1rem; color:#ffffff;'>{category}</span>
                <span style='font-size:0.8rem; color:#8892a4; margin-left:4px;'>{len(shows_in_cat)} shows</span>
            </div>
            """, unsafe_allow_html=True)

            for _, show in shows_in_cat.iterrows():
                st.markdown(f"""
                <div class='show-row'>
                    <span class='show-row-title'>{show['title']}</span>
                    <span class='show-row-meta'>{int(show['review_count'])} reviews &nbsp;&bull;&nbsp; sentiment {show['avg_sentiment']:.2f}</span>
                </div>
                """, unsafe_allow_html=True)

            st.markdown("<br>", unsafe_allow_html=True)

# ── WHY IT MATTERS ───────────────────────────────────────────────────────────
elif page == "Why It Matters":
    st.markdown("<h1>Why It Matters</h1>", unsafe_allow_html=True)
    st.markdown("<hr style='border-color:#2a2f3e; margin: 16px 0 28px 0'>", unsafe_allow_html=True)

    st.markdown("""
    <div class='info-panel' style='margin-bottom: 20px;'>
        <p style='font-family: DM Serif Display, serif; font-size: 1.15rem; color: #ffffff; margin-bottom: 14px;'>
        The relationship between television and mental health
        </p>
        <p>
        Television has become one of the most significant leisure activities of the modern age, with millions
        of people turning to streaming platforms daily. Research consistently shows that what we watch has a
        measurable impact on our mood, stress levels, and overall mental wellbeing. According to Adam (2023),
        watching TV shows can reduce stress and anxiety levels and increase feelings of happiness and
        relaxation but critically, this effect depends heavily on the type of content consumed.
        Not all television affects viewers equally.
        </p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("""
    <div class='info-panel' style='margin-bottom: 20px;'>
        <p style='font-family: DM Serif Display, serif; font-size: 1.15rem; color: #ffffff; margin-bottom: 14px;'>
        The gap in existing systems
        </p>
        <p>
        Current content classification systems age ratings, content advisories, genre labels were designed
        to protect younger audiences from inappropriate material. They were never designed to communicate the
        emotional weight of a programme for adult viewers managing their mental health. A show rated 18+
        could be darkly comic, deeply traumatic, or grippingly tense existing labels make no distinction
        between these very different experiences.
        </p>
        <p style='margin-top: 12px;'>
        ShunSpirit (2024) notes that TV shows have a unique ability to tap into our emotions through narrative,
        character development, and audiovisual effects. Many viewers form strong parasocial connections with
        characters and find their emotions are heavily impacted by the events that unfold on screen. For
        someone already experiencing anxiety or low mood, stumbling into the wrong show at the wrong time
        can have a genuinely negative effect on their mental state.
        </p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("""
    <div class='info-panel' style='margin-bottom: 20px;'>
        <p style='font-family: DM Serif Display, serif; font-size: 1.15rem; color: #ffffff; margin-bottom: 14px;'>
        A growing concern
        </p>
        <p>
        A 2024 study by the USC Norman Lear Center found that accurate and nuanced portrayals of mental health
        in entertainment can reduce stigma and encourage help-seeking behaviour demonstrating that the
        entertainment industry is increasingly aware of its responsibility towards viewer wellbeing.
        Meanwhile research by Starosta et al. (2021) found that individuals experiencing anxiety are more
        likely to binge-watch as a coping mechanism, sometimes making their anxiety worse in the process.
        The content they choose during those vulnerable moments matters more than ever.
        </p>
        <p style='margin-top: 12px;'>
        This project was built in response to that gap a practical tool that helps viewers make more
        mindful, informed choices about what they watch, based not on entertainment quality but on
        emotional impact.
        </p>
    </div>
    """, unsafe_allow_html=True)

    # Quote highlight
    st.markdown("""
    <div style='border-left: 4px solid #4ecdc4; padding: 16px 24px; margin: 24px 0; background: #0d2e2c; border-radius: 0 10px 10px 0;'>
        <p style='color: #e8e8e8; font-family: DM Serif Display, serif; font-size: 1.05rem; font-style: italic; margin: 0 0 8px 0;'>
        "Many people experience a strong connection with TV show characters and find that their emotions
        are heavily impacted by the events that unfold on the screen."
        </p>
        <p style='color: #8892a4; font-size: 0.8rem; margin: 0;'>ShunSpirit, 2024</p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("""
    <div style='background:#161b27; border:1px solid #2a2f3e; border-radius:10px; padding:14px 20px; margin-top: 8px;'>
        <p style='color:#4a5568; font-size:0.8rem; margin:0;'>
        References: Adam (2023) The Display Blog — ShunSpirit (2024) shunspirit.com — USC Norman Lear Center (2024) — Starosta et al. (2021) Frontiers in Psychiatry
        </p>
    </div>
    """, unsafe_allow_html=True)

# ── ABOUT MODEL ──────────────────────────────────────────────────────────────
elif page == "About Model":
    st.markdown("<h1>About the Model</h1>", unsafe_allow_html=True)
    st.markdown("<hr style='border-color:#2a2f3e; margin: 16px 0 28px 0'>", unsafe_allow_html=True)

    col1, col2, col3 = st.columns(3)
    with col1:
        st.metric("Overall Accuracy", "77%")
    with col2:
        st.metric("Model Type", "Random Forest")
    with col3:
        st.metric("Features Per Review", "115")

    st.markdown("<br>", unsafe_allow_html=True)

    # Accuracy by category
    st.markdown("<h3>Precision by Category</h3>", unsafe_allow_html=True)
    cats = ['Positive', 'Balanced', 'Intense', 'Heavy']
    precisions = [97, 70, 65, 68]
    colors_list = [category_colors[c] for c in cats]

    fig2 = go.Figure(data=[
        go.Bar(
            x=cats,
            y=precisions,
            marker_color=colors_list,
            marker_line_width=0,
            text=[f"{p}%" for p in precisions],
            textposition='outside',
            textfont=dict(color='#b0b8cc', size=13)
        )
    ])
    fig2.update_layout(
        plot_bgcolor='#161b27',
        paper_bgcolor='#161b27',
        font=dict(family='DM Sans', color='#8892a4', size=12),
        xaxis=dict(gridcolor='#2a2f3e', showline=False, tickfont=dict(color='#b0b8cc', size=13)),
        yaxis=dict(gridcolor='#2a2f3e', showline=False, tickfont=dict(color='#b0b8cc'), range=[0, 110]),
        margin=dict(l=20, r=20, t=30, b=20),
        height=300,
        showlegend=False,
    )
    st.plotly_chart(fig2, use_container_width=True)

    st.markdown("<h3>Key Finding</h3>", unsafe_allow_html=True)
    st.markdown("""
    <div style='background:#161b27; border:1px solid #2a2f3e; border-radius:12px; padding:24px 28px; margin-bottom:20px;'>
        <div style='font-family:DM Serif Display,serif; font-size:2.4rem; color:#4ecdc4; line-height:1;'>+33 pts</div>
        <div style='color:#8892a4; font-size:0.8rem; text-transform:uppercase; letter-spacing:0.08em; margin-top:4px;'>Accuracy improvement from TF-IDF features</div>
        <p style='color:#b0b8cc; font-size:0.88rem; margin-top:12px; line-height:1.7;'>
        Removing TF-IDF and relying solely on VADER sentiment scores and emotion features reduced
        accuracy from 77% to approximately 44%. The specific vocabulary viewers use in their reviews
        carries far more predictive signal than sentiment polarity alone.
        </p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("<h3>Technical Details</h3>", unsafe_allow_html=True)

    col1, col2 = st.columns(2)
    with col1:
        st.markdown("""
        <div class='info-panel'>
            <p><strong style='color:#e8e8e8; display:block; margin-bottom:10px;'>Features Extracted</strong>
            VADER sentiment scores (compound, positive, negative, neutral)<br>
            text2emotion dimensions (happy, sad, angry, fear, surprise)<br>
            Mental health keyword counts<br>
            Review length and punctuation intensity<br>
            TF-IDF term frequency weights
            </p>
        </div>
        """, unsafe_allow_html=True)
    with col2:
        st.markdown("""
        <div class='info-panel'>
            <p><strong style='color:#e8e8e8; display:block; margin-bottom:10px;'>Training Setup</strong>
            8,385 reviews across 50 TV shows<br>
            80/20 train-test split<br>
            200 decision trees, max depth 20<br>
            StandardScaler feature normalisation<br>
            Manual override system for edge cases
            </p>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("""
    <div style='background:#161b27; border:1px solid #2a2f3e; border-radius:10px; padding:14px 20px;'>
        <p style='color:#4a5568; font-size:0.8rem; margin:0;'>
        This tool is for informational purposes only. Individual emotional responses to media vary considerably.
        </p>
    </div>
    """, unsafe_allow_html=True)

Overwriting dashboard.py


In [112]:
overrides = {
    'Wednesday': 'Intense',
    'The Mandalorian': 'Balanced',
    'You': 'Intense',
    'The Last Kingdom': 'Intense',
    'Attack on Titan': 'Heavy',
    'Breaking Bad': 'Heavy',
    'The Boys': 'Heavy',
    'Black Mirror': 'Heavy',
    'Chernobyl': 'Heavy',
    'Dora the Explorer': 'Positive',
    'Jujutsu Kaisen': 'Intense',
    'Vikings': 'Intense',
    'SpongeBob SquarePants': 'Positive',
    'PAW Patrol': 'Positive',
    'Bridgerton': 'Balanced',
    'Euphoria': 'Heavy',
    'Something Very Bad Is Going to Happen': 'Intense',
    'Power': 'Intense',
    'Money Heist': 'Intense',
}

df = pd.read_csv('tv_reviews/imdb_tvshows.csv')
for show, correct_mood in overrides.items():
    df.loc[df['show_title'] == show, 'mood_category'] = correct_mood
df.to_csv('tv_reviews/imdb_tvshows.csv', index=False)

df_summary = pd.read_csv('show_summaries.csv')
for show, correct_mood in overrides.items():
    df_summary.loc[df_summary['title'] == show, 'mood_category'] = correct_mood
df_summary.to_csv('show_summaries.csv', index=False)

print("Overrides applied!")
for show, mood in overrides.items():
    print(f"  {show} -> {mood}")

Overrides applied!
  Wednesday -> Intense
  The Mandalorian -> Balanced
  You -> Intense
  The Last Kingdom -> Intense
  Attack on Titan -> Heavy
  Breaking Bad -> Heavy
  The Boys -> Heavy
  Black Mirror -> Heavy
  Chernobyl -> Heavy
  Dora the Explorer -> Positive
  Jujutsu Kaisen -> Intense
  Vikings -> Intense
  SpongeBob SquarePants -> Positive
  PAW Patrol -> Positive
  Bridgerton -> Balanced
  Euphoria -> Heavy
  Something Very Bad Is Going to Happen -> Intense
  Power -> Intense
  Money Heist -> Intense


In [113]:
#15
!pip install streamlit

In [114]:
#16
!pip install pyngrok
from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token("3CMjQYvdh2qVi24EGRUuMfAMmUz_5NsUm4qGphUnUi57RHaYr")

import subprocess, time
subprocess.Popen(['streamlit', 'run', 'dashboard.py', '--server.port', '8501'])
time.sleep(5)

public_url = ngrok.connect(8501)
print(f"Dashboard URL: {public_url}")

Dashboard URL: NgrokTunnel: "https://registry-activism-singing.ngrok-free.dev" -> "http://localhost:8501"
